# CS781 Phase 1 — Resumable decoder-only Transformer training

This version preserves the project's existing Phase-4 data foundation and streaming JSONL pipeline:
- `legal_bpe_32k.json` tokenizer
- train/validation manifests
- byte-offset `source_index.jsonl`
- `WorkerSafeDocumentAwarePacker` (does not concatenate the whole corpus into RAM)
- randomly initialized GPT-2-style decoder model using the original `MODEL_CONFIG`

**Target:** train to optimizer step **3,000** (set `TOTAL_STEPS = 2_000` if you want the shorter target). Local checkpoints are written frequently; the latest checkpoint is uploaded to a private Kaggle Dataset periodically so a new Kaggle session can resume.

**Before running:** attach the existing Phase-4 artifact dataset as notebook input. Create a private Kaggle Dataset named `cs781-phase1-checkpoints` once, with a `dataset-metadata.json` matching the slug below, and add Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`. If the checkpoint dataset already exists, keep it attached as an input too.

Run the cells top-to-bottom. Do not run the old notebook's one-off training-step or hard-coded “resume from step 700” cells.


In [ ]:
# ===== Cell: read every small config/log file + peek at the corpus and the Phase 1 test file (read-only, ~1 min) =====
import os, json, glob
from collections import defaultdict
OWN = "/kaggle/input/datasets/oipranesh"
def show_json(p, limit=2500):
    try: obj = json.load(open(p, encoding="utf-8"))
    except Exception as e: print(f"   (could not parse: {e})"); return
    print(json.dumps(obj, indent=1)[:limit])

print("=== 1) small JSON files (configs, states, policies) ===")
for root in ("cs781-phase-1-checkpoints", "cs781-phase4-final-recovery", "phase4-step1000-verified-model", "cs781-phase4-training-prep"):
    for p in sorted(glob.glob(f"{OWN}/{root}/**/*.json", recursive=True)):
        n = os.path.getsize(p)
        if "SHA256" in p or n > 100000: continue
        print(f"\n##### {p[len(OWN)+1:]}  ({n} bytes)")
        if n < 8000: show_json(p)
        else: print("   (large; skipped)")

def summarize_log(lines):
    rows = []
    for l in lines:
        try: rows.append(json.loads(l))
        except Exception: pass
    if not rows or not isinstance(rows[0], dict): return
    print("   keys:", list(rows[0].keys()))
    for r in rows[:2] + rows[-3:]: print("   ", json.dumps(r)[:300])
    for k in [k for k in rows[0] if "loss" in k.lower()][:2]:
        vals = [(i, r[k]) for i, r in enumerate(rows) if isinstance(r.get(k), (int, float))]
        if vals: print(f"   {k} at 10 evenly spaced rows:", [(i, round(v, 3)) for i, v in [vals[int(j * (len(vals) - 1) / 9)] for j in range(10)]])
    for k in [k for k in rows[0] if k.lower() in ("lr", "learning_rate")]:
        v = [r[k] for r in rows if isinstance(r.get(k), (int, float))]
        if v: print(f"   {k}: first {v[0]:.2e} max {max(v):.2e} min {min(v):.2e} last {v[-1]:.2e} distinct values {len(set(v))}")

print("\n=== 2) JSONL logs / manifests (not the corpus) ===")
for root in ("cs781-phase-1-checkpoints", "cs781-phase4-final-recovery", "phase4-step1000-verified-model"):
    for p in sorted(glob.glob(f"{OWN}/{root}/**/*.jsonl", recursive=True)):
        n = os.path.getsize(p); print(f"\n##### {p[len(OWN)+1:]}  ({n/2**20:.1f} MiB)")
        with open(p, encoding="utf-8", errors="ignore") as f:
            if n < 50e6:
                lines = f.read().splitlines(); print(f"   {len(lines)} lines"); summarize_log(lines)
            else: print("   first line:", f.readline()[:500])

print("\n=== 3) pretraining corpus (cs781-phase4-training-prep): one line per folder ===")
base = f"{OWN}/cs781-phase4-training-prep"; groups = defaultdict(list)
for p in glob.glob(base + "/**/*.jsonl", recursive=True): groups[os.path.dirname(p)].append(p)
for d, fs in sorted(groups.items()):
    gb = sum(os.path.getsize(p) for p in fs) / 2**30; print(f"\n{d[len(base)+1:]}: {len(fs)} files, {gb:.2f} GiB")
    with open(sorted(fs)[0], encoding="utf-8", errors="ignore") as f: line = f.readline()
    try:
        rec = json.loads(line); print("   keys:", list(rec.keys()), "| text chars:", len(str(rec.get("text", ""))))
        print("   sample:", {k: str(v)[:100] for k, v in rec.items()})
    except Exception: print("   first line:", line[:300])

print("\n=== 4) phase1-test-json/test.json ===")
try:
    obj = json.load(open("/kaggle/input/datasets/shubhamjoshi1806/phase1-test-json/test.json", encoding="utf-8"))
    print(type(obj).__name__, len(obj))
    first = obj[0] if isinstance(obj, list) else {k: obj[k] for k in list(obj)[:3]}
    print(json.dumps(first, ensure_ascii=False)[:800])
except Exception as e: print("could not read:", e)

In [ ]:
# ===== Phase 1 checkpoint audit cell (read-only; ~1-2 min; needs only the checkpoint, optionally the tokenizer) =====
import glob, math, json, torch, numpy as np
from pathlib import Path
CK = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
ck = torch.load(CK, map_location="cpu", weights_only=False, mmap=True)
sd, opt, cfg = ck["model_state_dict"], ck["optimizer_state_dict"], ck["model_config"]

# 1) weight tying: is lm_head literally the embedding matrix?
wte, head = sd["transformer.wte.weight"], sd["lm_head.weight"]
print("lm_head == wte:", torch.equal(wte, head), "| same storage:", wte.data_ptr() == head.data_ptr())
print("optimizer param groups:", len(opt["param_groups"]), "| params in group 0:", len(opt["param_groups"][0]["params"]),
      "| has 'initial_lr' (=> an LR scheduler object was attached):", "initial_lr" in opt["param_groups"][0])

# 2) which positions were actually trained? untouched rows stay random (norm ~0.02*sqrt(768)=0.55, neighbours uncorrelated)
wpe = sd["transformer.wpe.weight"].float(); norm = wpe.norm(dim=1)
cos = torch.nn.functional.cosine_similarity(wpe[:-1], wpe[1:], dim=1)
print("\nposition block      mean-norm   adjacent-cosine   (trained positions drift from 0.55 and become smooth)")
for a in range(0, wpe.shape[0], 256): print(f"{a:5d}-{a+255:5d}      {norm[a:a+256].mean():.3f}       {cos[a:a+255].mean():+.3f}")

# 3) token coverage proxy: Adam's second moment of each embedding row (tiny = rarely-seen token)
v = opt["state"][0]["exp_avg_sq"].float().mean(dim=1)
q = torch.quantile(v, torch.tensor([0.01, 0.1, 0.5, 0.9, 0.99]))
print("\nper-token Adam second-moment quantiles (1%,10%,50%,90%,99%):", [f"{x:.2e}" for x in q.tolist()],
      "| tokens below 1e-12:", int((v < 1e-12).sum()), "of", v.numel())
print("max possible drift of a weight if lr were 1e-5 for all", int(opt["state"][0]["step"]), "steps:", round(1e-5 * int(opt["state"][0]["step"]), 4), "(init std ~0.02)")

# 4) load through Hugging Face GPT-2 and measure next-token loss + generate  (verifies the weights actually form a working LM)
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer
m = GPT2LMHeadModel(GPT2Config(vocab_size=cfg["vocab_size"], n_positions=cfg["context_length"], n_ctx=cfg["context_length"], n_embd=cfg["hidden_size"],
                               n_layer=cfg["num_layers"], n_head=cfg["num_heads"], resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
print("\nload_state_dict:", m.load_state_dict(sd, strict=True)); m.eval()
dev = "cuda" if torch.cuda.is_available() else "cpu"; m.to(dev)
tok = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/legal_bpe_32k.json", recursive=True))[0])
if "val_texts" in globals():  texts = list(val_texts.values())[:6]        # legal judgments from Phase 2 if they are loaded in this session
else:
    texts = ["The appellant filed an appeal against the order passed by the High Court, which had dismissed the writ petition. "
             "The respondent submitted that the impugned order does not suffer from any infirmity and that the appeal deserves to be dismissed. "
             "Having heard learned counsel for the parties, we are of the considered view that the findings of the Tribunal are based on evidence."] * 4
nll, n = 0.0, 0; ids_all = []
with torch.no_grad():
    for t in texts:
        ids = tok.encode(t, add_special_tokens=False).ids[:2048]; ids_all += ids
        for s in range(0, max(len(ids) - 1, 1), 512):
            x = torch.tensor([ids[s:s + 513]], device=dev)
            if x.shape[1] < 2: continue
            lg = m(x[:, :-1]).logits.float(); nll += torch.nn.functional.cross_entropy(lg[0], x[0, 1:], reduction="sum").item(); n += x.shape[1] - 1
cnt = np.bincount(ids_all, minlength=cfg["vocab_size"]).astype(float); p = cnt / cnt.sum()
print(f"\nnext-token loss on {n} tokens: {nll/n:.3f} nats (perplexity {math.exp(nll/n):.0f}) | uniform baseline {math.log(cfg['vocab_size']):.2f} | "
      f"unigram entropy of this same text {-(p[p>0]*np.log(p[p>0])).sum():.2f} (optimistic baseline: fitted on the text itself)")
prompt = tok.encode("The appellant", add_special_tokens=False).ids
with torch.no_grad():
    out = m.generate(torch.tensor([prompt], device=dev), max_new_tokens=60, do_sample=False, pad_token_id=0)
print("greedy continuation:", tok.decode(out[0].tolist()))
print("\nmetadata:", {k: ck[k] for k in ("step", "reason", "saved_unix_time") if k in ck}, "| optimizer step counter:", int(opt["state"][0]["step"]))
del ck, sd, opt, m

In [ ]:
# Install only if the environment is missing these packages.
%pip -q install transformers tokenizers


In [ ]:
# ===== CELL — FIND THE PHASE 1 MODEL TRAINED FOR 3000 STEPS =====

import os, glob, torch

roots = ["/kaggle/input"]
patterns = ["*.pt", "*.pth", "*.ckpt", "*.bin"]

files = []
for pattern in patterns:
    files.extend(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

files = sorted(set(files))
print("Checkpoint files found:", len(files))

for path in files:
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(ck, dict):
            continue

        cfg = ck.get("model_config", ck.get("config", {}))
        step = next(
            (ck[k] for k in ["step", "global_step", "iteration", "iter"]
             if k in ck and isinstance(ck[k], (int, float))),
            None
        )

        # Check common nested training-state formats.
        if step is None:
            for key in ["trainer_state", "training_state", "metadata"]:
                state = ck.get(key, {})
                if isinstance(state, dict):
                    step = next(
                        (state[k] for k in ["step", "global_step", "iteration"]
                         if k in state and isinstance(state[k], (int, float))),
                        None
                    )
                    if step is not None:
                        break

        sd = ck.get("model_state_dict", ck.get("state_dict", ck))
        tensor_count = sum(
            torch.is_tensor(v) for v in sd.values()
        ) if isinstance(sd, dict) else 0

        print("\nFILE:", path)
        print("  step:", step)
        print("  config:", cfg if isinstance(cfg, dict) else type(cfg).__name__)
        print("  tensor count:", tensor_count)
        print("  top-level keys:", list(ck.keys())[:15])

    except Exception as e:
        print("\nSKIPPED:", path, "|", str(e)[:150])

In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import gc
import json
import math
import time
import random
from pathlib import Path

import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

print("=" * 80)
print("CS781 PHASE 1 — TRAINING ENVIRONMENT")
print("=" * 80)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print(
        "VRAM    :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device  :", DEVICE)

assert torch.cuda.is_available(), "CUDA GPU is required for this run."

print("\n✓ Environment ready")

In [ ]:
# ============================================================
# CELL 2 — PHASE-1 ARTIFACT PATHS
# ============================================================

CANDIDATES = [
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"),
]

CHECKPOINT = None

for candidate in CANDIDATES:
    if (
        (candidate / "manifests/train_manifest.jsonl").exists()
        and
        (candidate / "tokenizer/legal_bpe_32k.json").exists()
    ):
        CHECKPOINT = candidate
        break

assert CHECKPOINT is not None, "Phase-1 training artifacts were not found."

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST = CHECKPOINT / "manifests/test_manifest.jsonl"
TOKENIZER_PATH = CHECKPOINT / "tokenizer/legal_bpe_32k.json"

print("=" * 80)
print("PHASE-1 ARTIFACTS")
print("=" * 80)

print("Checkpoint :", CHECKPOINT)
print("Train      :", TRAIN_MANIFEST)
print("Validation :", VAL_MANIFEST)
print("Test       :", TEST_MANIFEST)
print("Tokenizer  :", TOKENIZER_PATH)

assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()
assert TOKENIZER_PATH.exists()

print("\n✓ All required artifacts found")

In [ ]:
# ============================================================
# CELL 3 — TOKENIZER
# ============================================================

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("=" * 80)
print("TOKENIZER")
print("=" * 80)

print("Vocabulary :", VOCAB_SIZE)
print("PAD        :", PAD_ID)
print("UNK        :", UNK_ID)
print("BOS        :", BOS_ID)
print("EOS        :", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("\n✓ Tokenizer loaded")
print("✓ Vocabulary = 32,000")

In [ ]:
# ============================================================
# CELL 4 — STREAMING SOURCE INDEX
# ============================================================

INDEX_CANDIDATES = [
    Path("/kaggle/working/phase4_streaming_index/train_source_index.jsonl"),
    Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-final-recovery/indexes/source_index.jsonl"
    ),
    CHECKPOINT / "indexes/source_index.jsonl",
]

INDEX_PATH = None

for candidate in INDEX_CANDIDATES:
    if candidate.exists():
        INDEX_PATH = candidate
        break

if INDEX_PATH is not None:

    print("Existing source index found:")
    print(INDEX_PATH)

else:

    print("No existing source index found.")
    print("Building index from the Phase-1 deduplicated corpus...")

    DEDUP_ROOT = CHECKPOINT / "dedup"

    assert DEDUP_ROOT.exists(), DEDUP_ROOT

    INDEX_DIR = Path("/kaggle/working/phase4_streaming_index")
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

    INDEX_PATH = INDEX_DIR / "train_source_index.jsonl"

    # Discover source files referenced by the training manifest.
    source_names = set()

    with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                source_names.add(rec["source_file"])

    print("Manifest source files:", len(source_names))

    def resolve_source(source_file):
        candidates = [
            DEDUP_ROOT / source_file,
            DEDUP_ROOT / Path(source_file).with_suffix(""),
        ]

        for p in candidates:
            if p.exists() and p.is_file():
                return p

        matches = [
            p for p in DEDUP_ROOT.rglob(Path(source_file).name)
            if p.is_file()
        ]

        if len(matches) == 1:
            return matches[0]

        raise FileNotFoundError(
            f"Could not resolve source file: {source_file}"
        )

    source_paths = {
        name: resolve_source(name)
        for name in source_names
    }

    print("\nBuilding byte-offset index...")

    total_records = 0

    with open(INDEX_PATH, "w", encoding="utf-8") as out:

        for source_name, source_path in source_paths.items():

            print("Indexing:", source_name)

            with open(
                source_path,
                "rb",
                buffering=1024 * 1024,
            ) as src:

                while True:

                    offset = src.tell()
                    raw = src.readline()

                    if not raw:
                        break

                    try:
                        obj = json.loads(raw)
                    except Exception as e:
                        raise RuntimeError(
                            f"Invalid JSON at {source_path}:{offset}"
                        ) from e

                    text = obj.get("text")

                    if not isinstance(text, str):
                        continue

                    text_hash = __import__("hashlib").sha256(
                        text.encode("utf-8")
                    ).hexdigest()

                    out.write(
                        json.dumps(
                            {
                                "text_hash": text_hash,
                                "source_file": source_name,
                                "byte_offset": offset,
                                "byte_length": len(raw),
                            },
                            separators=(",", ":"),
                        )
                        + "\n"
                    )

                    total_records += 1

            print("  indexed")

    print("\nTotal indexed records:", f"{total_records:,}")

print("\nIndex:", INDEX_PATH)
assert INDEX_PATH.exists()

print("✓ Source index ready")

In [ ]:
# ============================================================
# CELL 5 — WORKER-SAFE DOCUMENT-AWARE STREAMING DATASET
# ============================================================

class WorkerSafeDocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=None,
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint or CHECKPOINT)
        self.split = split

        # ----------------------------------------------------
        # Load byte-offset index
        # ----------------------------------------------------

        self.index = {}

        with open(self.index_path, "r", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                rec = json.loads(line)

                self.index[rec["text_hash"]] = {
                    "source_file": rec["source_file"],
                    "byte_offset": rec["byte_offset"],
                    "byte_length": rec["byte_length"],
                }

        # ----------------------------------------------------
        # Load manifest
        # ----------------------------------------------------

        self.manifest = []

        with open(self.manifest_path, "r", encoding="utf-8") as f:

            for line in f:

                if line.strip():
                    self.manifest.append(json.loads(line))

        # ----------------------------------------------------
        # Special tokens
        # ----------------------------------------------------

        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

        print(
            f"{split}: "
            f"{len(self.manifest):,} manifest records | "
            f"{len(self.index):,} index records"
        )

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:
            raise KeyError(
                f"text_hash not found: {text_hash}"
            )

        idx = self.index[text_hash]

        source_name = idx["source_file"]

        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = (
            Path(self.checkpoint)
            / "dedup"
            / source_name
        )

        if not source_path.exists():

            # Robust fallback for unusual artifact layout.
            matches = list(
                Path(self.checkpoint)
                .rglob(Path(source_name).name)
            )

            if len(matches) != 1:
                raise FileNotFoundError(
                    f"Cannot resolve source:\n{source_name}"
                )

            source_path = matches[0]

        with open(source_path, "rb") as f:

            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])

        raw = raw.rstrip(b"\r\n")

        return json.loads(raw.decode("utf-8"))

    def _extract_text(self, record):

        if isinstance(record, str):
            return record

        for key in ("text", "content", "document", "body"):

            value = record.get(key)

            if isinstance(value, str):
                return value

        raise KeyError(
            f"No text field found. Keys: {list(record.keys())}"
        )

    def _tokenize_document(self, text):

        ids = self.tokenizer.encode(text).ids

        return (
            [self.bos_id]
            + list(map(int, ids))
            + [self.eos_id]
        )

    def _pack_document(self, token_ids):

        for start in range(
            0,
            len(token_ids),
            self.context_length,
        ):

            chunk = token_ids[
                start:start + self.context_length
            ]

            if len(chunk) < self.context_length:

                chunk = chunk + [
                    self.pad_id
                ] * (
                    self.context_length - len(chunk)
                )

            yield chunk

    def __iter__(self):

        worker_info = get_worker_info()

        if worker_info is None:

            worker_id = 0
            num_workers = 1

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

        # Strided worker partitioning.
        for manifest_idx in range(
            worker_id,
            len(self.manifest),
            num_workers,
        ):

            rec = self.manifest[manifest_idx]

            source_record = self._read_source_record(rec)

            text = self._extract_text(source_record)

            token_ids = self._tokenize_document(text)

            for packed in self._pack_document(token_ids):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": attention_mask,
                }


print("✓ Worker-safe streaming dataset class ready")

In [ ]:
# ============================================================
# CELL 6 — TRAIN / VALIDATION DATASETS + DATALOADERS
# ============================================================

CONTEXT_LENGTH = 4096

BATCH_SIZE = 1
NUM_WORKERS = 0

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="train",
)

validation_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=VAL_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="validation",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

print("=" * 80)
print("DATALOADERS READY")
print("=" * 80)

print("Train records :", f"{len(train_dataset.manifest):,}")
print("Val records   :", f"{len(validation_dataset.manifest):,}")
print("Batch size    :", BATCH_SIZE)
print("Workers       :", NUM_WORKERS)
print("Context       :", CONTEXT_LENGTH)

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

from dataclasses import dataclass

# Use the tokenizer that was already loaded, if available.
# Otherwise load it directly from the Phase-4 artifact.
if "tokenizer" not in globals():
    from tokenizers import Tokenizer

    TOKENIZER_PATH = (
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
    )

    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Resolve special-token IDs directly from the tokenizer.
VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Vocabulary:", VOCAB_SIZE)
print("PAD_ID:", PAD_ID)
print("BOS_ID:", BOS_ID)
print("EOS_ID:", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None


@dataclass
class LegalLMConfig:
    vocab_size: int = VOCAB_SIZE
    max_position_embeddings: int = 4096

    hidden_size: int = 768
    intermediate_size: int = 3072

    num_hidden_layers: int = 12
    num_attention_heads: int = 12

    hidden_dropout: float = 0.0
    attention_dropout: float = 0.0

    bos_token_id: int = BOS_ID
    eos_token_id: int = EOS_ID
    pad_token_id: int = PAD_ID


MODEL_CONFIG = LegalLMConfig()

print("\nModel configuration:")
print(MODEL_CONFIG)

In [ ]:
# ============================================================
# CREATE RANDOMLY INITIALIZED DECODER-ONLY TRANSFORMER
# ============================================================

from transformers import GPT2Config, GPT2LMHeadModel

hf_config = GPT2Config(
    vocab_size=MODEL_CONFIG.vocab_size,
    n_positions=MODEL_CONFIG.max_position_embeddings,
    n_ctx=MODEL_CONFIG.max_position_embeddings,

    n_embd=MODEL_CONFIG.hidden_size,
    n_layer=MODEL_CONFIG.num_hidden_layers,
    n_head=MODEL_CONFIG.num_attention_heads,
    n_inner=MODEL_CONFIG.intermediate_size,

    resid_pdrop=MODEL_CONFIG.hidden_dropout,
    embd_pdrop=MODEL_CONFIG.hidden_dropout,
    attn_pdrop=MODEL_CONFIG.attention_dropout,

    bos_token_id=MODEL_CONFIG.bos_token_id,
    eos_token_id=MODEL_CONFIG.eos_token_id,
    pad_token_id=MODEL_CONFIG.pad_token_id,

    use_cache=False,
)

# Fresh model from configuration.
# NO pretrained checkpoint is loaded.
model = GPT2LMHeadModel(hf_config)

print("✓ Model created from scratch")

In [ ]:
# ============================================================
# PARAMETER CHECK
# ============================================================

param_count = sum(p.numel() for p in model.parameters())

print(f"Parameter count: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000

print("✓ Parameter count is within Phase-1 requirement")

In [ ]:
# ============================================================
# CELL 9 — GPU + MEMORY OPTIMIZATION
# ============================================================

model = model.to(DEVICE)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Device :", next(model.parameters()).device)
print("Gradient checkpointing : enabled")
print("use_cache              :", model.config.use_cache)

assert next(model.parameters()).is_cuda

print("✓ Model ready for T4 training")

In [ ]:
# ============================================================
# TRAINING CONFIGURATION — resumable run
# ============================================================
import os, json, math, time, shutil, signal, subprocess
from pathlib import Path

# Set to 2_000 for a shorter run; 3_000 is the intended target.
TOTAL_STEPS = 3_000
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
LOCAL_CHECKPOINT_EVERY = 25
KAGGLE_UPLOAD_EVERY = 250
NUM_WORKERS = 2
BATCH_SIZE = 1
CONTEXT_LENGTH = 4096

OUTPUT_DIR = Path("/kaggle/working/phase1_decoder_transformer")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "training_log.jsonl"
STATE_PATH = OUTPUT_DIR / "training_state.json"
LATEST_PATH = CHECKPOINT_DIR / "checkpoint_latest.pt"

# Kaggle Dataset slug must already exist for `kaggle datasets version`.
KAGGLE_DATASET_SLUG = "oipranesh/cs781-phase-1-checkpoints"
UPLOAD_DIR = Path("/kaggle/working/phase1_checkpoint_dataset")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Pull Kaggle API credentials from Kaggle Secrets, if configured.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
    print("Kaggle API credentials loaded from Secrets.")
except Exception as e:
    print("Kaggle Secrets not available in this session:", repr(e))
    print("Training can still run locally; Dataset uploads will be skipped until credentials are configured.")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

print("Total target steps:", TOTAL_STEPS)
print("Local checkpoint every:", LOCAL_CHECKPOINT_EVERY)
print("Kaggle upload every:", KAGGLE_UPLOAD_EVERY)
print("Output:", OUTPUT_DIR)


In [ ]:
# ============================================================
# CHECKPOINT DISCOVERY + SAFE SAVE + KAGGLE DATASET VERSIONING
# ============================================================
import re

def find_resume_checkpoint():
    # Prefer rolling checkpoint files; use filename step numbers for snapshots
    # so we do not load multi-GB optimizer states merely to inspect their steps.
    candidates = []
    if LATEST_PATH.exists():
        candidates.append((10**18, LATEST_PATH))
    for p in Path("/kaggle/input").rglob("checkpoint_latest.pt"):
        candidates.append((10**18, p))
    for p in Path("/kaggle/input").rglob("checkpoint_step_*.pt"):
        match = re.search(r"checkpoint_step_(\\d+)\\.pt$", p.name)
        if match:
            candidates.append((int(match.group(1)), p))
    if not candidates:
        return None
    # For rolling checkpoint(s), inspect their saved step once each. Numbered
    # snapshots already carry a step in the filename.
    ranked = []
    for named_step, p in candidates:
        if named_step == 10**18:
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                step = int(ck.get("step", -1))
                if step >= 0 and "model_state_dict" in ck:
                    ranked.append((step, p))
            except Exception as e:
                print("Ignoring unreadable checkpoint:", p, repr(e))
        else:
            ranked.append((named_step, p))
    if not ranked:
        return None
    ranked.sort(key=lambda x: x[0])
    step, path = ranked[-1]
    print(f"Found resume checkpoint at step {step}: {path}")
    return path

def save_checkpoint(step, reason="periodic"):
    temp_path = CHECKPOINT_DIR / "checkpoint_tmp.pt"
    payload = {
        "step": int(step),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "model_config": {
            "vocab_size": MODEL_CONFIG.vocab_size,
            "context_length": MODEL_CONFIG.max_position_embeddings,
            "hidden_size": MODEL_CONFIG.hidden_size,
            "num_layers": MODEL_CONFIG.num_hidden_layers,
            "num_heads": MODEL_CONFIG.num_attention_heads,
        },
        "reason": reason,
        "saved_unix_time": time.time(),
    }
    torch.save(payload, temp_path)
    os.replace(temp_path, LATEST_PATH)
    # Keep a numbered snapshot alongside the rolling checkpoint.
    numbered = CHECKPOINT_DIR / f"checkpoint_step_{step:06d}.pt"
    shutil.copy2(LATEST_PATH, numbered)
    # Retain only the two most recent numbered snapshots to limit disk use.
    snapshots = sorted(CHECKPOINT_DIR.glob("checkpoint_step_*.pt"))
    for old in snapshots[:-2]:
        try: old.unlink()
        except OSError: pass
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump({"step": int(step), "reason": reason, "saved_unix_time": time.time()}, f, indent=2)
    print(f"Saved checkpoint: step={step}, size={LATEST_PATH.stat().st_size / 1e9:.2f} GB ({reason})")

def upload_checkpoint_to_kaggle(step):
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Skipping Dataset upload: Kaggle API credentials are not configured.")
        return False
    if not (UPLOAD_DIR / "dataset-metadata.json").exists():
        metadata = {
            "title": "CS781 Phase 1 Training Checkpoints",
            "id": KAGGLE_DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}],
            "private": True,
        }
        (UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
    shutil.copy2(LATEST_PATH, UPLOAD_DIR / "checkpoint_latest.pt")
    if LOG_PATH.exists():
        shutil.copy2(LOG_PATH, UPLOAD_DIR / "training_log.jsonl")
    (UPLOAD_DIR / "training_state.json").write_text(json.dumps({
        "step": int(step), "total_steps": int(TOTAL_STEPS), "updated_unix_time": time.time()
    }, indent=2))
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR),
             "-m", f"CS781 Phase 1 checkpoint step {step}", "--dir-mode", "tar"],
            check=True, text=True, capture_output=True, timeout=1800,
        )
        print(result.stdout[-3000:] if result.stdout else "Kaggle Dataset version uploaded.")
        return True
    except Exception as e:
        print("Kaggle upload failed (local checkpoint is safe):", repr(e))
        if getattr(e, "stdout", None): print(e.stdout[-2000:])
        if getattr(e, "stderr", None): print(e.stderr[-3000:])
        return False

# Resume only if a checkpoint is actually attached/found.
# Model architecture must match the checkpoint's recorded configuration.
RESUME_PATH = find_resume_checkpoint()
START_STEP = 0
if RESUME_PATH is not None:
    checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
    saved_cfg = checkpoint.get("model_config", {})
    current_cfg = {
        "vocab_size": MODEL_CONFIG.vocab_size,
        "context_length": MODEL_CONFIG.max_position_embeddings,
        "hidden_size": MODEL_CONFIG.hidden_size,
        "num_layers": MODEL_CONFIG.num_hidden_layers,
        "num_heads": MODEL_CONFIG.num_attention_heads,
    }
    if saved_cfg and saved_cfg != current_cfg:
        raise ValueError(f"Checkpoint architecture mismatch. Saved={saved_cfg}; current={current_cfg}")
    model.load_state_dict(checkpoint["model_state_dict"])
    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    START_STEP = int(checkpoint["step"])
    print("Resumed from optimizer step:", START_STEP)
else:
    print("No checkpoint found: starting from randomly initialized model at step 0.")

assert 0 <= START_STEP <= TOTAL_STEPS


In [ ]:
# ============================================================
# TRAIN TO TOTAL_STEPS
# ============================================================
model.train()
train_iterator = iter(train_loader)
running_loss = 0.0
logged_batches = 0
run_started = time.time()

def _save_on_termination(signum, frame):
    print(f"Received signal {signum}; attempting emergency checkpoint save.")
    try:
        save_checkpoint(current_step, reason=f"signal_{signum}")
    except Exception as e:
        print("Emergency save failed:", repr(e))
    raise SystemExit(128 + signum)

# Best effort only; Kaggle may terminate a session without delivering a signal.
try:
    signal.signal(signal.SIGTERM, _save_on_termination)
    signal.signal(signal.SIGINT, _save_on_termination)
except Exception:
    pass

current_step = START_STEP
for step in range(START_STEP + 1, TOTAL_STEPS + 1):
    current_step = step
    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM):
        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()
        accumulated_loss += float(loss.detach().float().item())

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()

    mean_loss = accumulated_loss / GRAD_ACCUM
    running_loss += mean_loss
    logged_batches += 1

    if step % LOG_EVERY == 0 or step == START_STEP + 1:
        avg_loss = running_loss / max(logged_batches, 1)
        record = {
            "step": step, "loss": avg_loss,
            "perplexity": math.exp(min(avg_loss, 20)),
            "grad_norm": float(grad_norm),
            "elapsed_seconds": time.time() - run_started,
        }
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
        print(f"step={step:5d}/{TOTAL_STEPS} | loss={avg_loss:.4f} | ppl={record['perplexity']:.2f} | grad={float(grad_norm):.3f} | elapsed={(time.time()-run_started)/60:.1f} min")
        running_loss = 0.0
        logged_batches = 0

    if step % LOCAL_CHECKPOINT_EVERY == 0 or step == TOTAL_STEPS:
        save_checkpoint(step, reason="final" if step == TOTAL_STEPS else "periodic")

    if step % KAGGLE_UPLOAD_EVERY == 0 or step == TOTAL_STEPS:
        # Save immediately before upload. If upload fails, the local checkpoint remains.
        if step % LOCAL_CHECKPOINT_EVERY != 0 and step != TOTAL_STEPS:
            save_checkpoint(step, reason="upload_boundary")
        upload_checkpoint_to_kaggle(step)

print("\nTraining loop finished.")
print("Final step:", current_step)
print("Latest checkpoint:", LATEST_PATH)


In [ ]:
from pathlib import Path

ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_files(names):
    found = []
    for root in ROOTS:
        if root.exists():
            for name in names:
                found.extend(root.rglob(name))
    return list(dict.fromkeys(p for p in found if p.is_file()))

checkpoints = find_files(["checkpoint_latest.pt"])
tests = find_files(["test.json", "Phase1- Test.json"])
tokenizer_configs = find_files(["tokenizer.json"])
tokenizer_vocab = find_files(["vocab.json"])

print("CHECKPOINTS:")
for p in checkpoints:
    print(p, f"({p.stat().st_size / 1e9:.2f} GB)")

print("\nTEST FILES:")
for p in tests:
    print(p)

print("\nTOKENIZER FILES:")
for p in tokenizer_configs + tokenizer_vocab:
    print(p)

In [ ]:

import json
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
)
TEST_PATH = Path(
    "/kaggle/input/datasets/shubhamjoshi1806/phase1-test-json/test.json"
)
TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Verify files exist
for path in [CHECKPOINT_PATH, TEST_PATH, TOKENIZER_PATH]:
    print(f"{path}: exists={path.is_file()}")
    if not path.is_file():
        raise FileNotFoundError(path)

# Inspect test data
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nTest type:", type(test_data).__name__)
print("Number of examples:", len(test_data))

first_key = next(iter(test_data))
print("First ID:", first_key)
print("First record:", str(test_data[first_key])[:1000])

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint keys:", list(checkpoint.keys()))
print("Saved training step:", checkpoint.get("step"))
print("Model config:", checkpoint.get("model_config"))

# Inspect tokenizer JSON structure
with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_json = json.load(f)

print("\nTokenizer JSON top-level keys:", list(tokenizer_json.keys()))
print("Tokenizer model type:",
      tokenizer_json.get("model", {}).get("type"))


In [ ]:
from tokenizers import Tokenizer, decoders

bpe = Tokenizer.from_file(str(TOKENIZER_PATH))
bpe.decoder = decoders.ByteLevel()

PAD_ID = bpe.token_to_id("[PAD]")
BOS_ID = bpe.token_to_id("[BOS]")
EOS_ID = bpe.token_to_id("[EOS]")

assert bpe.get_vocab_size() == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("Vocabulary:", bpe.get_vocab_size())
print("PAD/BOS/EOS:", PAD_ID, BOS_ID, EOS_ID)

test_text = "The Court shall consider the matter."
print("Round-trip:", bpe.decode(bpe.encode(test_text).ids))

In [ ]:
import torch

CONTEXT_LENGTH = 4096
MAX_GENERATION_TOKENS = 300

def generate_128_words(prompt, temperature=0.8, top_k=50, seed=42):
    encoded = bpe.encode(prompt)
    prompt_ids = encoded.ids

    if not prompt_ids:
        prompt_ids = [BOS_ID]

    # Reserve room for generated tokens.
    prompt_ids = prompt_ids[-(CONTEXT_LENGTH - MAX_GENERATION_TOKENS):]
    input_ids = torch.tensor(
        [prompt_ids], dtype=torch.long, device=device
    )
    generated = input_ids.clone()

    # Use a local generator so each example has reproducible sampling.
    rng = torch.Generator(device=device)
    rng.manual_seed(seed)

    with torch.inference_mode():
        for _ in range(MAX_GENERATION_TOKENS):
            context_ids = generated[:, -CONTEXT_LENGTH:]

            outputs = model(
                input_ids=context_ids,
                attention_mask=torch.ones_like(context_ids),
                use_cache=False,
            )
            logits = outputs.logits[:, -1, :] / temperature

            if top_k is not None:
                k = min(top_k, logits.shape[-1])
                values, indices = torch.topk(logits, k, dim=-1)
                filtered = torch.full_like(logits, float("-inf"))
                filtered.scatter_(1, indices, values)
                logits = filtered

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(
                probs, num_samples=1, generator=rng
            )
            next_id = next_token.item()

            if next_id == EOS_ID:
                break

            generated = torch.cat([generated, next_token], dim=1)

            new_ids = generated[0, input_ids.shape[1]:].tolist()
            text = bpe.decode(new_ids, skip_special_tokens=True).strip()

            if len(text.split()) >= 128:
                break

    new_ids = generated[0, input_ids.shape[1]:].tolist()
    text = bpe.decode(new_ids, skip_special_tokens=True).strip()
    return " ".join(text.split()[:128])

In [ ]:
first_id = next(iter(test_data))

sample_output = generate_128_words(
    test_data[first_id]["text"],
    seed=42,
)

print("ID:", first_id)
print("Generated words:", len(sample_output.split()))
print(sample_output)

assert "Ġ" not in sample_output
assert "Ċ" not in sample_output

In [ ]:
import json
import time
from pathlib import Path

output_path = Path("/kaggle/working/prediction.json")
predictions = {}
start_time = time.time()
total = len(test_data)

for i, (test_id, item) in enumerate(test_data.items(), start=1):
    predictions[str(test_id)] = {
        "text": generate_128_words(
            item["text"],
            seed=42 + i,
        )
    }

    if i % 10 == 0 or i == total:
        print(f"[{i}/{total}] ID={test_id}")

with output_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)
print("Examples:", len(predictions))
print(f"Elapsed: {(time.time() - start_time) / 60:.1f} minutes")

In [ ]:
with output_path.open("r", encoding="utf-8") as f:
    saved = json.load(f)

assert len(saved) == len(test_data)
assert list(saved.keys()) == [str(k) for k in test_data.keys()]

for test_id, item in saved.items():
    assert set(item) == {"text"}
    assert isinstance(item["text"], str)
    assert len(item["text"].split()) <= 128
    assert "Ġ" not in item["text"]
    assert "Ċ" not in item["text"]

print("Validation passed.")
print("Output file:", output_path)
print("Size:", f"{output_path.stat().st_size / 1024:.1f} KB")

## After a Kaggle session ends

1. Check the notebook output for the latest successful `Kaggle Dataset version uploaded` message and step number.
2. In the next session, attach the private `cs781-phase1-checkpoints` Dataset as an input.
3. Run this notebook from the top. It searches attached inputs for `checkpoint_latest.pt`, validates the architecture, and restores model, optimizer, scaler, and step.
4. If the session stops between uploads, you will resume from the most recent uploaded checkpoint, not necessarily the last local step.

Kaggle can terminate a session abruptly; no notebook can guarantee an upload at the exact moment of termination. The frequent local saves plus periodic Dataset versions reduce lost work.


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")


In [ ]:
# CELL 1 — Hugging Face authentication + Phase 2 file discovery

%pip -q install -U huggingface_hub

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is empty. Check Kaggle Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

HF_REPO = "Exploration-Lab/CS-781-Capstone"
api = HfApi(token=HF_TOKEN)

repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
)

phase2_files = [
    name for name in repo_files
    if "phase_2_lner" in name.lower()
]

print("Authenticated to Hugging Face.")
print("Phase 2 files discovered:", len(phase2_files))

for name in phase2_files:
    print(name)

assert phase2_files, (
    "No Phase_2_LNER files found. Check repository access and folder names."
)

In [ ]:
# CELL 2 — Download Phase 2 files using the authenticated HF token

import json
from pathlib import Path

PHASE2_DIR = Path("/kaggle/working/phase2_lner")
PHASE2_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "fold1-text.json",
    "fold1-gold.json",
    "fold2-text.json",
    "fold2-gold.json",
    "fold3-text.json",
    "labels.txt",
]

def find_repo_path(filename):
    matches = [
        name for name in phase2_files
        if Path(name).name.lower() == filename.lower()
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one repository file named {filename}; "
            f"found: {matches}"
        )
    return matches[0]

downloaded = {}

for filename in REQUIRED_FILES:
    repo_path = find_repo_path(filename)
    local_path = hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=repo_path,
        token=HF_TOKEN,
        local_dir=str(PHASE2_DIR),
    )
    downloaded[filename] = Path(local_path)
    print(f"{filename} -> {local_path}")

def load_json(filename):
    with downloaded[filename].open("r", encoding="utf-8") as f:
        return json.load(f)

train_text_1 = load_json("fold1-text.json")
train_gold_1 = load_json("fold1-gold.json")
train_text_2 = load_json("fold2-text.json")
train_gold_2 = load_json("fold2-gold.json")
test_text = load_json("fold3-text.json")

with downloaded["labels.txt"].open("r", encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]

print("\n=== Dataset summary ===")
print("Fold 1 texts:", len(train_text_1))
print("Fold 1 gold:", len(train_gold_1))
print("Fold 2 texts:", len(train_text_2))
print("Fold 2 gold:", len(train_gold_2))
print("Fold 3 test texts:", len(test_text))
print("Labels:", ENTITY_LABELS)

assert len(train_text_1) == len(train_gold_1), "Fold 1 count mismatch"
assert len(train_text_2) == len(train_gold_2), "Fold 2 count mismatch"
assert test_text, "Fold 3 test data is empty"
assert ENTITY_LABELS, "Label list is empty"

In [ ]:
# CELL 3 — Inspect actual annotation schema

def show_example(texts, gold, fold_name):
    doc_id = next(iter(texts))
    text = texts[doc_id]
    annotation = gold[doc_id]

    print(f"\n=== {fold_name} example ===")
    print("Document ID:", doc_id)
    print("Text type:", type(text).__name__)
    print("Text length:", len(text))
    print("Text preview:", repr(text[:500]))
    print("Annotation type:", type(annotation).__name__)
    print("Raw annotation:", repr(annotation)[:3000])

show_example(train_text_1, train_gold_1, "Fold 1")
show_example(train_text_2, train_gold_2, "Fold 2")

print("\n=== Test example ===")
test_id = next(iter(test_text))
print("Document ID:", test_id)
print("Text length:", len(test_text[test_id]))
print("Text preview:", repr(test_text[test_id][:500]))

In [ ]:
# CELL 4 — Load the Phase 1 tokenizer

from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found: {TOKENIZER_PATH}. "
        "Attach your Phase 1 tokenizer dataset to the notebook."
    )

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("Sample tokenization:", tokenizer.encode(
    "The Supreme Court considered the appeal."
).ids[:30])

assert tokenizer.get_vocab_size() == 32000

In [ ]:
# CELL 5 — Validate gold spans and build label IDs

from collections import Counter

assert set(train_text_1) == set(train_gold_1)
assert set(train_text_2) == set(train_gold_2)

all_labels = set(ENTITY_LABELS)
label_counts = Counter()
bad_spans = []
boundary_mismatches = []

def check_fold(texts, gold, fold_name):
    for doc_id, entities in gold.items():
        text = texts[doc_id]

        for ent in entities:
            start, end, label = int(ent[0]), int(ent[1]), str(ent[2])
            entity_text = ent[3] if len(ent) > 3 else None

            if label not in all_labels:
                bad_spans.append((fold_name, doc_id, ent, "unknown label"))
                continue

            if not (0 <= start < end <= len(text)):
                bad_spans.append((fold_name, doc_id, ent, "invalid offsets"))
                continue

            if entity_text is not None and text[start:end] != entity_text:
                boundary_mismatches.append(
                    (fold_name, doc_id, start, end, entity_text, text[start:end])
                )

            label_counts[label] += 1

check_fold(train_text_1, train_gold_1, "fold1")
check_fold(train_text_2, train_gold_2, "fold2")

print("Entity counts by label:")
for label, count in label_counts.most_common():
    print(f"{label:12s} {count:,}")

print("\nInvalid spans:", len(bad_spans))
print("Text/offset mismatches:", len(boundary_mismatches))

if bad_spans:
    print("First invalid span:", bad_spans[0])
    raise ValueError("Fix invalid annotations before training.")

if boundary_mismatches:
    print("First text/offset mismatch:", boundary_mismatches[0])
    print("Some annotations may have non-identical entity_text; inspect before proceeding.")

BIO_LABELS = ["O"]
for label in ENTITY_LABELS:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("\nNumber of BIO classes:", len(BIO_LABELS))
print(BIO_LABELS)

In [ ]:
# CELL 6 — Restore the Phase 1 Transformer checkpoint

import torch
from pathlib import Path
from transformers import GPT2Config, GPT2LMHeadModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Enable a Kaggle GPU before training."

checkpoint_candidates = list(
    Path("/kaggle/input").rglob("checkpoint_latest.pt")
)

# Prefer the known Transformer checkpoint dataset path if present.
preferred = [
    p for p in checkpoint_candidates
    if "cs781-phase-1-checkpoints" in str(p).lower()
]
if preferred:
    checkpoint_candidates = preferred

if not checkpoint_candidates:
    raise FileNotFoundError(
        "No checkpoint_latest.pt found under /kaggle/input. "
        "Attach your trained Phase 1 Transformer checkpoint dataset."
    )

loaded_checkpoint = None
CHECKPOINT_PATH = None

for path in checkpoint_candidates:
    try:
        candidate = torch.load(path, map_location="cpu", weights_only=False)
        state = candidate.get("model_state_dict", {})
        if any(k.startswith("transformer.wte.") for k in state):
            loaded_checkpoint = candidate
            CHECKPOINT_PATH = path
            break
    except Exception as exc:
        print("Could not inspect:", path, repr(exc))

if loaded_checkpoint is None:
    raise RuntimeError(
        "No GPT-2-style Transformer checkpoint was found. "
        "The attached checkpoint may belong to the RWKV architecture."
    )

print("Checkpoint:", CHECKPOINT_PATH)
print("Saved step:", loaded_checkpoint.get("step"))
print("Saved config:", loaded_checkpoint.get("model_config"))

saved_cfg = loaded_checkpoint.get("model_config", {})
vocab_size = int(saved_cfg.get("vocab_size", tokenizer.get_vocab_size()))
context_size = int(saved_cfg.get("context_length", 4096))
hidden_size = int(saved_cfg.get("hidden_size", 768))
num_layers = int(saved_cfg.get("num_layers", 12))
num_heads = int(saved_cfg.get("num_heads", 12))

assert vocab_size == tokenizer.get_vocab_size(), (
    f"Tokenizer/model vocabulary mismatch: {tokenizer.get_vocab_size()} vs {vocab_size}"
)

hf_config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=context_size,
    n_ctx=context_size,
    n_embd=hidden_size,
    n_layer=num_layers,
    n_head=num_heads,
    n_inner=hidden_size * 4,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    use_cache=False,
)

phase1_lm = GPT2LMHeadModel(hf_config)
phase1_lm.load_state_dict(loaded_checkpoint["model_state_dict"], strict=True)
phase1_lm.config.use_cache = False

print("Restored Phase 1 model.")
print("Parameters (M):", round(sum(p.numel() for p in phase1_lm.parameters()) / 1e6, 2))

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed initialized.")

In [ ]:
# CELL 7 — Tokenize documents and align gold spans

from collections import defaultdict

MAX_LEN = 512
WINDOW_STRIDE = 448  # 64-token overlap
IGNORE_INDEX = -100

def document_to_windows(text, entities, max_len=MAX_LEN, stride=WINDOW_STRIDE):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids
    offsets = enc.offsets

    # Per-token labels, initially outside any entity.
    tags = [label2id["O"]] * len(ids)

    # Assign B-/I- tags based on token character offsets.
    for start, end, label, *rest in entities:
        start, end, label = int(start), int(end), str(label)
        overlapping = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not overlapping:
            continue

        # Require tokenization to cover the entity boundaries exactly.
        first, last = overlapping[0], overlapping[-1]
        if offsets[first][0] != start or offsets[last][1] != end:
            # Keep the example but report this condition for later analysis.
            pass

        tags[first] = label2id[f"B-{label}"]
        for i in overlapping[1:]:
            tags[i] = label2id[f"I-{label}"]

    windows = []
    for start_idx in range(0, len(ids), stride):
        end_idx = min(start_idx + max_len, len(ids))
        if start_idx >= end_idx:
            break

        # Token IDs plus aligned BIO labels.
        win_ids = ids[start_idx:end_idx]
        win_tags = tags[start_idx:end_idx]
        win_offsets = offsets[start_idx:end_idx]

        # Avoid training on an entity fragment cut by a window boundary.
        # For the first token in a window, an I-tag without preceding context
        # is converted to B-tag for that entity.
        if win_tags and id2label[win_tags[0]].startswith("I-"):
            ent_type = id2label[win_tags[0]][2:]
            win_tags[0] = label2id[f"B-{ent_type}"]

        windows.append({
            "input_ids": win_ids,
            "labels": win_tags,
            "offsets": win_offsets,
        })

        if end_idx == len(ids):
            break

    return windows

# Document-level holdout prevents windows from the same document leaking
# into both training and validation.
all_documents = {}
for fold_text, fold_gold in [
    (train_text_1, train_gold_1),
    (train_text_2, train_gold_2),
]:
    for doc_id, text in fold_text.items():
        if doc_id in all_documents:
            raise ValueError(f"Duplicate document ID across folds: {doc_id}")
        all_documents[doc_id] = (text, fold_gold[doc_id])

doc_ids = list(all_documents)
random.Random(SEED).shuffle(doc_ids)
n_val = max(1, int(0.1 * len(doc_ids)))
val_ids = set(doc_ids[:n_val])
train_ids = set(doc_ids[n_val:])

def make_windows(ids):
    result = []
    for doc_id in ids:
        text, entities = all_documents[doc_id]
        result.extend(document_to_windows(text, entities))
    return result

train_windows = make_windows(train_ids)
val_windows = make_windows(val_ids)

print("Train documents:", len(train_ids))
print("Validation documents:", len(val_ids))
print("Train windows:", len(train_windows))
print("Validation windows:", len(val_windows))
print("Example window length:", len(train_windows[0]["input_ids"]))

assert train_windows and val_windows
assert max(len(x["input_ids"]) for x in train_windows) <= MAX_LEN

In [ ]:
# CELL 8 — Token classification head + data loaders

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

def collate_windows(items):
    batch_len = max(len(x["input_ids"]) for x in items)
    batch_size = len(items)

    input_ids = torch.full(
        (batch_size, batch_len), PAD_ID, dtype=torch.long
    )
    attention_mask = torch.zeros(
        (batch_size, batch_len), dtype=torch.long
    )
    labels = torch.full(
        (batch_size, batch_len), IGNORE_INDEX, dtype=torch.long
    )

    for i, item in enumerate(items):
        n = len(item["input_ids"])
        input_ids[i, :n] = torch.tensor(item["input_ids"])
        attention_mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

BATCH_SIZE = 1
NUM_WORKERS = 0

train_loader_ner = DataLoader(
    WindowDataset(train_windows),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_windows,
)

val_loader_ner = DataLoader(
    WindowDataset(val_windows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_windows,
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            pretrained_lm.config.n_embd, num_labels
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        hidden = self.dropout(outputs.last_hidden_state)
        return self.classifier(hidden)

ner_model = LegalNERModel(phase1_lm, len(BIO_LABELS)).to(DEVICE)

print("Device:", DEVICE)
print("Trainable parameters (M):", round(
    sum(p.numel() for p in ner_model.parameters() if p.requires_grad) / 1e6, 2
))
print("Train windows:", len(train_loader_ner))
print("Validation windows:", len(val_loader_ner))

# Smoke test before training.
sample_batch = next(iter(train_loader_ner))
with torch.no_grad():
    sample_logits = ner_model(
        sample_batch["input_ids"].to(DEVICE),
        sample_batch["attention_mask"].to(DEVICE),
    )
print("Smoke-test logits shape:", tuple(sample_logits.shape))
assert sample_logits.shape[-1] == len(BIO_LABELS)

In [ ]:
# CELL 9 — Fine-tune Phase 1 Transformer for legal NER

import math, time, json
from pathlib import Path
from sklearn.metrics import f1_score

NER_DIR = Path("/kaggle/working/phase2_ner")
NER_DIR.mkdir(parents=True, exist_ok=True)
BEST_NER_PATH = NER_DIR / "best_ner_model.pt"

EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0
USE_AMP = True

optimizer_ner = torch.optim.AdamW(
    ner_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scaler_ner = torch.amp.GradScaler(
    "cuda", enabled=(DEVICE.type == "cuda" and USE_AMP)
)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

def evaluate_token_f1():
    ner_model.eval()
    gold_all, pred_all = [], []
    total_loss, steps = 0.0, 0

    with torch.no_grad():
        for batch in val_loader_ner:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            gold = batch["labels"].to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda" and USE_AMP),
            ):
                logits = ner_model(ids, mask)
                loss = criterion(
                    logits.reshape(-1, len(BIO_LABELS)),
                    gold.reshape(-1),
                )

            pred = logits.argmax(dim=-1)
            valid = gold != IGNORE_INDEX

            gold_all.extend(gold[valid].cpu().tolist())
            pred_all.extend(pred[valid].cpu().tolist())
            total_loss += loss.item()
            steps += 1

    # Report macro F1 over entity BIO classes; exclude O from macro averaging.
    entity_ids = [
        idx for idx, label in id2label.items()
        if label != "O"
    ]
    score = f1_score(
        gold_all,
        pred_all,
        labels=entity_ids,
        average="macro",
        zero_division=0,
    ) if gold_all else 0.0

    return total_loss / max(steps, 1), score

for epoch in range(1, EPOCHS + 1):
    ner_model.train()
    optimizer_ner.zero_grad(set_to_none=True)
    running_loss = 0.0
    start_time = time.time()

    for step, batch in enumerate(train_loader_ner, start=1):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        gold = batch["labels"].to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda" and USE_AMP),
        ):
            logits = ner_model(ids, mask)
            loss = criterion(
                logits.reshape(-1, len(BIO_LABELS)),
                gold.reshape(-1),
            )
            scaled_loss = loss / GRAD_ACCUM

        scaler_ner.scale(scaled_loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader_ner):
            scaler_ner.unscale_(optimizer_ner)
            torch.nn.utils.clip_grad_norm_(
                ner_model.parameters(), MAX_GRAD_NORM
            )
            scaler_ner.step(optimizer_ner)
            scaler_ner.update()
            optimizer_ner.zero_grad(set_to_none=True)

        if step % 100 == 0:
            print(
                f"epoch={epoch}/{EPOCHS} step={step}/{len(train_loader_ner)} "
                f"loss={running_loss / 100:.4f}",
                flush=True,
            )
            running_loss = 0.0

    val_loss, val_macro_f1 = evaluate_token_f1()
    elapsed = (time.time() - start_time) / 60

    print(
        f"Epoch {epoch}: val_loss={val_loss:.4f} "
        f"val_token_macro_f1={val_macro_f1:.4f} "
        f"time={elapsed:.1f} min",
        flush=True,
    )

    if not BEST_NER_PATH.exists() or val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save({
            "model_state_dict": ner_model.state_dict(),
            "bio_labels": BIO_LABELS,
            "entity_labels": ENTITY_LABELS,
            "epoch": epoch,
            "val_token_macro_f1": val_macro_f1,
            "phase1_checkpoint": str(CHECKPOINT_PATH),
        }, BEST_NER_PATH)
        print("Saved best NER checkpoint:", BEST_NER_PATH)

print("Best validation token macro-F1:", best_val_f1)

In [ ]:
# CELL 10 — Fold 3 inference and submission JSON

import json
import torch
from pathlib import Path

# Load best model weights.
saved_ner = torch.load(
    BEST_NER_PATH,
    map_location=DEVICE,
    weights_only=False,
)
ner_model.load_state_dict(saved_ner["model_state_dict"])
ner_model.eval()

def decode_bio_spans(predicted_ids, offsets, text):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close_span():
        if current_label is not None and current_start is not None:
            if current_end > current_start:
                spans.append([current_start, current_end, current_label])

    for pred_id, (start, end) in zip(predicted_ids, offsets):
        if end <= start:
            continue

        tag = id2label[int(pred_id)]

        if tag == "O":
            close_span()
            current_label = current_start = current_end = None
            continue

        prefix, entity_label = tag.split("-", 1)

        if prefix == "B" or current_label != entity_label:
            close_span()
            current_label = entity_label
            current_start = start
            current_end = end
        else:
            # Continue the same entity.
            current_end = end

    close_span()

    # Preserve valid character offsets and exact text slices.
    valid = []
    for start, end, label in spans:
        if 0 <= start < end <= len(text):
            valid.append([int(start), int(end), label])
    return valid

def predict_document(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    all_ids = enc.ids
    all_offsets = enc.offsets
    predictions = []

    # Non-overlapping windows prevent duplicate predictions from overlap.
    for start_idx in range(0, len(all_ids), MAX_LEN):
        end_idx = min(start_idx + MAX_LEN, len(all_ids))
        ids = all_ids[start_idx:end_idx]
        offsets = all_offsets[start_idx:end_idx]

        if not ids:
            continue

        input_tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)
        attention_mask = torch.ones_like(input_tensor)

        with torch.no_grad():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda"),
            ):
                logits = ner_model(input_tensor, attention_mask)

        pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()
        predictions.extend(
            decode_bio_spans(pred_ids, offsets, text)
        )

    # Sort by position, then remove exact duplicates if any.
    predictions.sort(key=lambda x: (x[0], x[1], x[2]))
    deduplicated = []
    seen = set()

    for span in predictions:
        key = tuple(span)
        if key not in seen:
            seen.add(key)
            deduplicated.append(span)

    return deduplicated

prediction_path = Path("/kaggle/working/prediction.json")
phase2_predictions = {}

for i, (doc_id, text) in enumerate(test_text.items(), start=1):
    phase2_predictions[str(doc_id)] = predict_document(text)

    if i % 10 == 0 or i == len(test_text):
        print(f"Predicted {i}/{len(test_text)} documents", flush=True)

with prediction_path.open("w", encoding="utf-8") as f:
    json.dump(phase2_predictions, f, ensure_ascii=False)

print("\nSaved:", prediction_path)
print("Documents:", len(phase2_predictions))
print("Entities predicted:", sum(len(v) for v in phase2_predictions.values()))

# Output sanity checks.
assert list(phase2_predictions.keys()) == [str(k) for k in test_text.keys()]

for doc_id, spans in phase2_predictions.items():
    text = test_text[doc_id]
    for span in spans:
        assert len(span) == 3
        start, end, label = span
        assert 0 <= start < end <= len(text)
        assert label in ENTITY_LABELS

print("Output validation passed.")
print("Example:", next(iter(phase2_predictions.items())))

In [ ]:

import torch
from pathlib import Path

CHECKPOINT_PATH = Path("/kaggle/working/phase2_ner/best_ner_model.pt")
PREDICTION_PATH = Path("/kaggle/working/prediction.json")

assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"
assert "model" in globals(), "The trained NER model is not in memory. Rerun the model-loading cell."

print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Parameters (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))
print("Tokenizer available:", "tokenizer" in globals())
print("Best checkpoint:", CHECKPOINT_PATH.exists())
print("Test prediction file:", PREDICTION_PATH.exists())

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
print("Saved epoch:", ckpt.get("epoch"))
print("Saved validation token Macro-F1:", ckpt.get("val_token_macro_f1"))
print("Saved Phase 1 checkpoint:", ckpt.get("phase1_checkpoint"))
print("Saved BIO classes:", len(ckpt["bio_labels"]))


In [ ]:

import json
from pathlib import Path

DATA_DIR = Path("/kaggle/working/phase2_lner/Phase_2_LNER/Train")

with open(DATA_DIR / "fold2-text.json", encoding="utf-8") as f:
    val_texts = json.load(f)

with open(DATA_DIR / "fold2-gold.json", encoding="utf-8") as f:
    val_gold = json.load(f)

print("Validation documents:", len(val_texts))
print("Gold documents:", len(val_gold))
print("Matching document IDs:", set(val_texts) == set(val_gold))

assert set(val_texts) == set(val_gold), "Text and gold document IDs differ."

print("\nFirst document ID:", next(iter(val_texts)))
print("Text length:", len(val_texts[next(iter(val_texts))]))
print("Gold sample:", val_gold[next(iter(val_gold))][:3])


In [ ]:

import numpy as np
import torch

BIO_LABELS = ckpt["bio_labels"]
ENTITY_LABELS = ckpt["entity_labels"]
LABEL_TO_ID = {label: i for i, label in enumerate(BIO_LABELS)}

MAX_LEN = 512
STRIDE = 448
DEVICE = next(model.parameters()).device

def extract_logits(output):
    # Supports common custom-model return formats.
    if hasattr(output, "logits"):
        return output.logits
    if isinstance(output, dict):
        if "logits" in output:
            return output["logits"]
    if isinstance(output, (tuple, list)):
        for item in output:
            if torch.is_tensor(item) and item.ndim == 3:
                return item
    if torch.is_tensor(output) and output.ndim == 3:
        return output
    raise TypeError(
        f"Cannot extract token logits from output type {type(output)}"
    )

def decode_bio_to_spans(token_offsets, tag_ids, bio_labels):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close_span():
        nonlocal active_label, active_start, active_end
        if active_label is not None and active_end > active_start:
            spans.append((active_start, active_end, active_label))
        active_label = active_start = active_end = None

    for (start, end), tag_id in zip(token_offsets, tag_ids):
        if start == end:  # Special/empty-offset token
            continue

        tag = bio_labels[int(tag_id)]
        if tag == "O":
            close_span()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or active_label != label:
            close_span()
            active_label = label
            active_start = start
            active_end = end
        else:
            # Continue an existing I-label span.
            active_end = end

    close_span()
    return spans

@torch.inference_mode()
def predict_document_spans(text):
    encoded = tokenizer.encode(text, add_special_tokens=False)
    token_ids = encoded.ids
    offsets = encoded.offsets

    if not token_ids:
        return []

    # Aggregate logits from overlapping windows.
    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float64
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    was_training = model.training
    model.eval()

    try:
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_LEN, len(token_ids))
            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=DEVICE
            )
            mask = torch.ones_like(ids)

            output = model(input_ids=ids, attention_mask=mask)
            logits = extract_logits(output)[0].float().cpu().numpy()

            expected_classes = len(BIO_LABELS)
            if logits.shape[-1] != expected_classes:
                raise ValueError(
                    f"Model emits {logits.shape[-1]} classes, "
                    f"but checkpoint has {expected_classes} BIO labels."
                )

            logits_sum[start:end] += logits[:end-start]
            counts[start:end] += 1

            if end == len(token_ids):
                break
    finally:
        model.train(was_training)

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    predicted_ids = mean_logits.argmax(axis=-1)

    spans = decode_bio_to_spans(offsets, predicted_ids, BIO_LABELS)

    # Keep only spans inside the original document.
    return sorted({
        (int(s), int(e), label)
        for s, e, label in spans
        if 0 <= s < e <= len(text)
    })


In [ ]:

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build the NER wrapper around the existing Phase 1 GPT-2 model.
# `phase1_lm` should be the original pretrained GPT2LMHeadModel.
assert "phase1_lm" in globals(), (
    "phase1_lm is missing. Rerun the cell that restores the Phase 1 GPT-2 checkpoint."
)
assert "LegalNERModel" in globals(), (
    "LegalNERModel is missing. Rerun the cell that defines it."
)

model = LegalNERModel(
    pretrained_lm=phase1_lm,
    num_labels=len(ckpt["bio_labels"])
)

missing, unexpected = model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

print("Loaded NER model:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("NER classes:", len(ckpt["bio_labels"]))
print("Strict checkpoint load successful.")


In [ ]:

# Use a short test sequence; no training or weight updates.
sample_ids = tokenizer.encode(
    next(iter(val_texts.values()))[:1000],
    add_special_tokens=False
).ids[:64]

input_ids = torch.tensor([sample_ids], dtype=torch.long, device=DEVICE)
attention_mask = torch.ones_like(input_ids)

with torch.inference_mode():
    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Input shape:", tuple(input_ids.shape))
print("NER logits shape:", tuple(logits.shape))

assert logits.shape[-1] == len(ckpt["bio_labels"]), (
    f"Expected {len(ckpt['bio_labels'])} NER classes, got {logits.shape[-1]}"
)

print("PASS: model returns 25 NER classes.")


In [ ]:

# Inspect the tokenizer's exact character offsets on real gold entities.
from pathlib import Path

doc_id = next(iter(val_texts))
text = val_texts[doc_id]
gold_entities = val_gold[doc_id]

encoded = tokenizer.encode(text, add_special_tokens=False)

print("Document:", doc_id)
print("Text length:", len(text))
print("Token count:", len(encoded.ids))
print("\nFirst 12 gold entities and overlapping tokens:\n")

for start, end, label, entity_text in gold_entities[:12]:
    print(f"\nGOLD {label} [{start}:{end}] {text[start:end]!r}")

    overlaps = []
    for token_id, (ts, te) in zip(encoded.ids, encoded.offsets):
        if ts < end and te > start:
            overlaps.append((ts, te, text[ts:te], token_id))

    for ts, te, token_text, token_id in overlaps:
        print(f"  token [{ts}:{te}] {token_text!r} (id={token_id})")

print("\nTokenizer:", type(tokenizer).__name__)
print("First offsets:", encoded.offsets[:10])


In [ ]:

def decode_bio_to_spans(token_offsets, tag_ids, bio_labels, text):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close_span():
        nonlocal active_label, active_start, active_end

        if active_label is not None:
            # Trim spaces introduced by byte-level BPE offsets.
            start = active_start
            end = active_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((start, end, active_label))

        active_label = active_start = active_end = None

    for (start, end), tag_id in zip(token_offsets, tag_ids):
        if start == end:
            continue

        tag = bio_labels[int(tag_id)]

        if tag == "O":
            close_span()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or active_label != label:
            close_span()
            active_label = label
            active_start = start
            active_end = end
        else:
            active_end = end

    close_span()
    return spans


In [ ]:

import numpy as np
import torch

BIO_LABELS = ckpt["bio_labels"]
ENTITY_LABELS = ckpt["entity_labels"]
DEVICE = next(model.parameters()).device

MAX_LEN = 512
STRIDE = 448

def decode_bio_to_spans(token_offsets, tag_ids, bio_labels, text):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close_span():
        nonlocal active_label, active_start, active_end

        if active_label is not None:
            start, end = active_start, active_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((int(start), int(end), active_label))

        active_label = active_start = active_end = None

    for (start, end), tag_id in zip(token_offsets, tag_ids):
        if start == end:
            continue

        tag = bio_labels[int(tag_id)]

        if tag == "O":
            close_span()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or active_label != label:
            close_span()
            active_label = label
            active_start = start
            active_end = end
        else:
            active_end = end

    close_span()
    return spans


def extract_logits(output):
    if hasattr(output, "logits"):
        return output.logits
    if isinstance(output, dict) and "logits" in output:
        return output["logits"]
    if isinstance(output, (tuple, list)):
        for item in output:
            if torch.is_tensor(item) and item.ndim == 3:
                return item
    if torch.is_tensor(output) and output.ndim == 3:
        return output
    raise TypeError(f"Cannot extract logits from {type(output)}")


@torch.inference_mode()
def predict_document_spans(text):
    encoded = tokenizer.encode(text, add_special_tokens=False)
    token_ids = encoded.ids
    offsets = encoded.offsets

    if not token_ids:
        return []

    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float64
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    was_training = model.training
    model.eval()

    try:
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_LEN, len(token_ids))

            ids = torch.tensor(
                [token_ids[start:end]],
                dtype=torch.long,
                device=DEVICE,
            )
            mask = torch.ones_like(ids)

            output = model(input_ids=ids, attention_mask=mask)
            logits = extract_logits(output)[0].float().cpu().numpy()

            if logits.shape[-1] != len(BIO_LABELS):
                raise ValueError(
                    f"Expected {len(BIO_LABELS)} classes, "
                    f"received {logits.shape[-1]}"
                )

            logits_sum[start:end] += logits[:end-start]
            counts[start:end] += 1

            if end == len(token_ids):
                break
    finally:
        model.train(was_training)

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    predicted_ids = mean_logits.argmax(axis=-1)

    spans = decode_bio_to_spans(
        offsets, predicted_ids, BIO_LABELS, text
    )

    return sorted(set(
        (s, e, label)
        for s, e, label in spans
        if 0 <= s < e <= len(text)
    ))

print("Corrected inference function ready.")


In [ ]:

val_predictions = {}

for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
    val_predictions[doc_id] = predict_document_spans(text)

    if i % 5 == 0 or i == len(val_texts):
        print(f"Processed {i}/{len(val_texts)} documents")

print("\nDocuments:", len(val_predictions))
print("Example predictions:", next(iter(val_predictions.values()))[:10])


In [ ]:

from collections import defaultdict

gold_by_label = defaultdict(set)
pred_by_label = defaultdict(set)

for doc_id, entities in val_gold.items():
    for start, end, label, *_ in entities:
        gold_by_label[label].add(
            (str(doc_id), int(start), int(end), label)
        )

for doc_id, entities in val_predictions.items():
    for start, end, label in entities:
        pred_by_label[label].add(
            (str(doc_id), int(start), int(end), label)
        )

scores = {}

print(
    f"{'Label':<12}{'Gold':>7}{'Pred':>7}{'TP':>7}"
    f"{'Prec':>9}{'Recall':>9}{'F1':>9}"
)

for label in ENTITY_LABELS:
    gold_set = gold_by_label[label]
    pred_set = pred_by_label[label]

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    scores[label] = {
        "gold": len(gold_set),
        "predicted": len(pred_set),
        "TP": tp,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }

    print(
        f"{label:<12}{len(gold_set):>7}{len(pred_set):>7}{tp:>7}"
        f"{precision:>9.3f}{recall:>9.3f}{f1:>9.3f}"
    )

exact_macro_f1 = sum(
    scores[label]["F1"] for label in ENTITY_LABELS
) / len(ENTITY_LABELS)

print(f"\nExact-span entity Macro-F1: {exact_macro_f1:.4f}")


In [ ]:

for doc_id in list(val_texts.keys())[:3]:
    text = val_texts[doc_id]
    gold = [
        (int(s), int(e), label)
        for s, e, label, *_ in val_gold[doc_id]
    ]
    pred = val_predictions[doc_id]

    print(f"\n{'=' * 70}")
    print("Document:", doc_id)
    print(f"Gold: {len(gold)} | Predicted: {len(pred)}")

    print("\nGOLD:")
    for s, e, label in sorted(gold)[:12]:
        print(f"{label:<12} [{s}:{e}] {text[s:e]!r}")

    print("\nPREDICTED:")
    for s, e, label in sorted(pred)[:12]:
        print(f"{label:<12} [{s}:{e}] {text[s:e]!r}")


In [ ]:

import torch
from collections import Counter

ckpt = torch.load(
    "/kaggle/working/phase2_ner/best_ner_model.pt",
    map_location="cpu",
    weights_only=False
)

print("Saved epoch:", ckpt["epoch"])
print("Saved token Macro-F1:", ckpt["val_token_macro_f1"])

# Inspect classifier bias and weight magnitudes.
state = ckpt["model_state_dict"]
W = state["classifier.weight"].float()
b = state["classifier.bias"].float()
labels = ckpt["bio_labels"]

print("\nClassifier bias by class:")
for i, label in enumerate(labels):
    print(f"{label:<15} bias={b[i].item(): .4f} weight_norm={W[i].norm().item():.3f}")

# Count predictions on one validation document.
text = next(iter(val_texts.values()))
enc = tokenizer.encode(text, add_special_tokens=False)
ids = torch.tensor([enc.ids[:512]], dtype=torch.long, device=DEVICE)
mask = torch.ones_like(ids)

model.eval()
with torch.inference_mode():
    logits = model(input_ids=ids, attention_mask=mask)
    pred_ids = logits.argmax(-1)[0].cpu().tolist()

counts = Counter(labels[i] for i in pred_ids)
print("\nPredicted BIO token counts for first 512 tokens:")
for label, count in counts.most_common():
    print(f"{label:<15} {count}")


In [ ]:

from collections import Counter

stats = Counter()
examples = []

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)

    for start, end, label, entity_text in val_gold[doc_id]:
        overlapping = [
            (ts, te) for ts, te in enc.offsets
            if ts < end and te > start
        ]

        if not overlapping:
            stats["no_overlap"] += 1
            continue

        first_start, first_end = overlapping[0]
        last_start, last_end = overlapping[-1]

        stats["entities"] += 1
        if first_start < start or last_end > end:
            stats["boundary_inside_token"] += 1
            if len(examples) < 12:
                examples.append({
                    "doc_id": doc_id,
                    "gold": (start, end, label, entity_text),
                    "first_token": (first_start, first_end, text[first_start:first_end]),
                    "last_token": (last_start, last_end, text[last_start:last_end]),
                })
        else:
            stats["token_boundaries_cover_entity"] += 1

print("Alignment summary:", dict(stats))
print("\nBoundary examples:")
for item in examples:
    print(item)


In [ ]:

from collections import Counter

stats = Counter()
unrepresentable = []

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)

    for start, end, label, entity_text in val_gold[doc_id]:
        # Token offsets fully inside the gold entity.
        inside = [
            (ts, te) for ts, te in enc.offsets
            if start <= ts and te <= end and ts < te
        ]

        # Exact coverage means a contiguous token sequence starts and ends
        # exactly at the gold character boundaries.
        exact = bool(inside) and inside[0][0] == start and inside[-1][1] == end

        if exact:
            stats["exactly_representable"] += 1
        else:
            stats["not_exactly_representable"] += 1
            if len(unrepresentable) < 15:
                unrepresentable.append(
                    (doc_id, start, end, label, entity_text, inside[:3])
                )

total = stats["exactly_representable"] + stats["not_exactly_representable"]

print("Gold entities:", total)
print("Exactly representable:", stats["exactly_representable"])
print("Not exactly representable:", stats["not_exactly_representable"])
print("Representable fraction:",
      round(stats["exactly_representable"] / max(total, 1), 4))

print("\nExamples not exactly representable:")
for item in unrepresentable:
    print(item)


In [ ]:

from collections import defaultdict
import numpy as np
import torch

def gold_char_labels_to_token_labels(text, gold_entities, offsets):
    # Diagnostic only: assign each token the label with the largest
    # character overlap. This does not alter the training checkpoint.
    labels = ["O"] * len(offsets)

    for start, end, label, *_ in gold_entities:
        for i, (ts, te) in enumerate(offsets):
            overlap = max(0, min(te, end) - max(ts, start))
            if overlap > 0:
                labels[i] = label

    return labels

token_tp = defaultdict(int)
token_fp = defaultdict(int)
token_fn = defaultdict(int)

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids
    offsets = enc.offsets

    gold_labels = gold_char_labels_to_token_labels(
        text, val_gold[doc_id], offsets
    )

    predicted_labels = []
    model.eval()

    with torch.inference_mode():
        for start in range(0, len(ids), 448):
            end = min(start + 512, len(ids))
            input_ids = torch.tensor(
                [ids[start:end]], device=DEVICE, dtype=torch.long
            )
            mask = torch.ones_like(input_ids)
            logits = extract_logits(
                model(input_ids=input_ids, attention_mask=mask)
            )[0]
            predicted_labels.extend(
                logits.argmax(-1).cpu().tolist()
            )
            if end == len(ids):
                break

    predicted_labels = [BIO_LABELS[i] for i in predicted_labels]

    for label in ENTITY_LABELS:
        gold_set = {
            i for i, value in enumerate(gold_labels) if value == label
        }
        pred_set = {
            i for i, value in enumerate(predicted_labels)
            if value.endswith("-" + label)
        }

        token_tp[label] += len(gold_set & pred_set)
        token_fp[label] += len(pred_set - gold_set)
        token_fn[label] += len(gold_set - pred_set)

print("Token-level type-only diagnostic (not the official metric)")
print(f"{'Label':<12}{'Precision':>12}{'Recall':>12}{'F1':>10}")

type_f1s = []
for label in ENTITY_LABELS:
    tp, fp, fn = token_tp[label], token_fp[label], token_fn[label]
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    type_f1s.append(f1)
    print(f"{label:<12}{p:>12.3f}{r:>12.3f}{f1:>10.3f}")

print("\nToken-level type-only Macro-F1:", round(float(np.mean(type_f1s)), 4))


In [ ]:

# CELL 1 — Build a clean fold-1 / fold-2 experiment

import random
import numpy as np
import torch
from collections import Counter
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Enable a Kaggle GPU."

MAX_LEN = 512
WINDOW_STRIDE = 448
IGNORE_INDEX = -100

# Use fold 1 for training and fold 2 for validation.
# No document appears in both sets.
train_texts = train_text_1
train_gold = train_gold_1
val_texts = train_text_2
val_gold = train_gold_2

BIO_LABELS = ["O"]
for label in ENTITY_LABELS:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for label, i in label2id.items()}

def document_to_windows(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    tags = [label2id["O"]] * len(ids)

    # Assign BIO labels to every token overlapping a gold entity.
    for entity in entities:
        start, end, label = int(entity[0]), int(entity[1]), str(entity[2])
        overlapping = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        if not overlapping:
            continue

        tags[overlapping[0]] = label2id[f"B-{label}"]
        for i in overlapping[1:]:
            tags[i] = label2id[f"I-{label}"]

    windows = []
    for start in range(0, len(ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break

        windows.append({
            "input_ids": ids[start:end],
            "labels": tags[start:end],
        })

        if end == len(ids):
            break

    return windows

train_windows = [
    window
    for doc_id, text in train_texts.items()
    for window in document_to_windows(text, train_gold[doc_id])
]
val_windows = [
    window
    for doc_id, text in val_texts.items()
    for window in document_to_windows(text, val_gold[doc_id])
]

print("Device:", DEVICE)
print("Train documents:", len(train_texts))
print("Validation documents:", len(val_texts))
print("Train windows:", len(train_windows))
print("Validation windows:", len(val_windows))
print("BIO classes:", len(BIO_LABELS))

assert set(train_texts).isdisjoint(set(val_texts))
assert train_windows and val_windows


In [ ]:

# CELL 2 — GPT-2 encoder + NER head + class weights

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            pretrained_lm.config.n_embd, num_labels
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        return self.classifier(self.dropout(outputs.last_hidden_state))

class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

# Padding is ignored by attention_mask and IGNORE_INDEX in the loss.
PAD_ID = 0

def collate_windows(items):
    length = max(len(item["input_ids"]) for item in items)
    batch = len(items)

    ids = torch.full((batch, length), PAD_ID, dtype=torch.long)
    mask = torch.zeros((batch, length), dtype=torch.long)
    labels = torch.full(
        (batch, length), IGNORE_INDEX, dtype=torch.long
    )

    for i, item in enumerate(items):
        n = len(item["input_ids"])
        ids[i, :n] = torch.tensor(item["input_ids"])
        mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"])

    return ids, mask, labels

BATCH_SIZE = 1

train_loader = DataLoader(
    WindowDataset(train_windows),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_windows,
)
val_loader = DataLoader(
    WindowDataset(val_windows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_windows,
)

# Build weights from actual training-token frequencies.
counts = Counter(
    tag
    for window in train_windows
    for tag in window["labels"]
)
frequencies = torch.tensor(
    [max(counts.get(i, 0), 1) for i in range(len(BIO_LABELS))],
    dtype=torch.float32,
)

# Moderately reduce the dominance of O without extreme rare-class weights.
weights = torch.sqrt(frequencies[0] / frequencies)
weights[0] = 1.0
weights = weights.clamp(min=1.0, max=5.0)
weights = weights / weights.mean()

criterion = nn.CrossEntropyLoss(
    weight=weights.to(DEVICE),
    ignore_index=IGNORE_INDEX,
)

ner_model = LegalNERModel(
    pretrained_lm=phase1_lm,
    num_labels=len(BIO_LABELS),
).to(DEVICE)

print("Trainable parameters (M):",
      round(sum(p.numel() for p in ner_model.parameters()) / 1e6, 2))
print("\nClass weights:")
for label, weight in zip(BIO_LABELS, weights.tolist()):
    print(f"{label:<15} {weight:.3f}")

ids, mask, labels = next(iter(train_loader))
with torch.no_grad():
    logits = ner_model(ids.to(DEVICE), mask.to(DEVICE))
print("\nSmoke-test logits:", tuple(logits.shape))
assert logits.shape[-1] == len(BIO_LABELS)


In [ ]:

# CELL 3 — Weighted fine-tuning and exact-span validation

import time
import numpy as np
from pathlib import Path
from collections import defaultdict

NER_DIR = Path("/kaggle/working/phase2_ner_corrected")
NER_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = NER_DIR / "best_ner_model.pt"

EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
USE_AMP = True

optimizer = torch.optim.AdamW(
    ner_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler(
    "cuda", enabled=(DEVICE.type == "cuda" and USE_AMP)
)

def decode_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            start, end = current_start, current_end
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1
            if start < end:
                spans.append((start, end, current_label))
        current_label = current_start = current_end = None

    for (start, end), pred_id in zip(offsets, pred_ids):
        if end <= start:
            continue
        tag = BIO_LABELS[int(pred_id)]

        if tag == "O":
            close()
            continue

        prefix, label = tag.split("-", 1)
        if prefix == "B" or current_label != label:
            close()
            current_label, current_start, current_end = label, start, end
        else:
            current_end = end

    close()
    return sorted(set(spans))

@torch.inference_mode()
def predict_doc(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    if not ids:
        return []

    logits_sum = np.zeros((len(ids), len(BIO_LABELS)), dtype=np.float64)
    counts = np.zeros(len(ids), dtype=np.int32)

    was_training = ner_model.training
    ner_model.eval()
    try:
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            x = torch.tensor([ids[start:end]], device=DEVICE)
            mask = torch.ones_like(x)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda" and USE_AMP),
            ):
                logits = ner_model(x, mask)

            logits_sum[start:end] += logits[0].float().cpu().numpy()
            counts[start:end] += 1

            if end == len(ids):
                break
    finally:
        ner_model.train(was_training)

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    return decode_spans(text, offsets, mean_logits.argmax(-1))

def exact_span_macro_f1(texts, gold):
    gold_by_label = defaultdict(set)
    pred_by_label = defaultdict(set)

    for doc_id, text in texts.items():
        for start, end, label, *_ in gold[doc_id]:
            gold_by_label[label].add(
                (str(doc_id), int(start), int(end), label)
            )
        for start, end, label in predict_doc(text):
            pred_by_label[label].add(
                (str(doc_id), start, end, label)
            )

    per_label = {}
    for label in ENTITY_LABELS:
        g, p = gold_by_label[label], pred_by_label[label]
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = f1

    return float(np.mean([per_label[x] for x in ENTITY_LABELS])), per_label

best_score = -1.0

for epoch in range(1, EPOCHS + 1):
    ner_model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    start_time = time.time()

    for step, (ids, mask, gold) in enumerate(train_loader, start=1):
        ids, mask, gold = ids.to(DEVICE), mask.to(DEVICE), gold.to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda" and USE_AMP),
        ):
            logits = ner_model(ids, mask)
            loss = criterion(
                logits.reshape(-1, len(BIO_LABELS)),
                gold.reshape(-1),
            )

        scaler.scale(loss / GRAD_ACCUM).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(ner_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if step % 100 == 0 or step == len(train_loader):
            print(
                f"epoch={epoch}/{EPOCHS} step={step}/{len(train_loader)} "
                f"loss={running_loss / min(100, step):.4f}",
                flush=True,
            )
            running_loss = 0.0

    val_macro, per_label = exact_span_macro_f1(val_texts, val_gold)
    elapsed = (time.time() - start_time) / 60

    print(f"\nEpoch {epoch}: exact-span Macro-F1={val_macro:.4f}; "
          f"time={elapsed:.1f} min")
    print("Per-label F1:", {
        label: round(per_label[label], 3) for label in ENTITY_LABELS
    })

    if val_macro > best_score:
        best_score = val_macro
        torch.save({
            "model_state_dict": ner_model.state_dict(),
            "bio_labels": BIO_LABELS,
            "entity_labels": ENTITY_LABELS,
            "epoch": epoch,
            "val_exact_span_macro_f1": val_macro,
            "phase1_checkpoint": str(CHECKPOINT_PATH),
            "train_fold": 1,
            "validation_fold": 2,
        }, BEST_PATH)
        print("Saved best checkpoint:", BEST_PATH)

print("\nBest validation exact-span Macro-F1:", best_score)


In [ ]:

# CELL 4 — Test inference using the corrected checkpoint

import json
from pathlib import Path

saved = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
ner_model.load_state_dict(saved["model_state_dict"], strict=True)
ner_model.eval()

# `test_text` should be the dictionary loaded from fold3-text.json.
assert "test_text" in globals(), "Load fold3-text.json into test_text first."

output_path = Path("/kaggle/working/prediction_corrected.json")
predictions = {}

for i, (doc_id, text) in enumerate(test_text.items(), start=1):
    predictions[str(doc_id)] = [
        [int(start), int(end), label]
        for start, end, label in predict_doc(text)
    ]
    if i % 5 == 0 or i == len(test_text):
        print(f"Processed {i}/{len(test_text)}")

with output_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)

# Validate offsets and labels against the source texts.
for doc_id, spans in predictions.items():
    text = test_text[doc_id]
    for start, end, label in spans:
        assert 0 <= start < end <= len(text)
        assert label in ENTITY_LABELS

print("\nSaved:", output_path)
print("Documents:", len(predictions))
print("Total entities:", sum(len(v) for v in predictions.values()))
print("Best validation Macro-F1:", saved["val_exact_span_macro_f1"])


In [ ]:

import torch
from pathlib import Path
from collections import Counter

BEST_PATH = Path(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
)

saved = torch.load(
    BEST_PATH, map_location=DEVICE, weights_only=False
)
ner_model.load_state_dict(saved["model_state_dict"], strict=True)
ner_model.eval()

print("Checkpoint epoch:", saved["epoch"])
print("Validation exact-span Macro-F1:",
      saved["val_exact_span_macro_f1"])

counts = Counter()
for spans in val_predictions.values():
    counts.update(label for _, _, label in spans)

print("\nValidation predicted entities by label:")
for label in ENTITY_LABELS:
    print(f"{label:<12} {counts[label]:>6}")

print("\nMean predicted entities/document:",
      round(sum(map(len, val_predictions.values())) / len(val_predictions), 1))


In [ ]:

from collections import Counter
import numpy as np
import torch

# Use the corrected epoch-1 checkpoint already loaded into ner_model.
ner_model.eval()

fresh_val_predictions = {}
for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
    fresh_val_predictions[doc_id] = predict_doc(text)
    if i % 5 == 0 or i == len(val_texts):
        print(f"Processed {i}/{len(val_texts)}")

counts = Counter()
for spans in fresh_val_predictions.values():
    counts.update(label for _, _, label in spans)

print("\nFresh validation predictions:")
print("Documents:", len(fresh_val_predictions))
print("Total entities:", sum(counts.values()))
print("Mean entities/document:",
      round(sum(counts.values()) / max(len(fresh_val_predictions), 1), 1))

for label in ENTITY_LABELS:
    print(f"{label:<12} {counts[label]:>6}")


In [ ]:

from collections import defaultdict
import numpy as np

gold_by_label = defaultdict(set)
pred_by_label = defaultdict(set)

for doc_id, entities in val_gold.items():
    for start, end, label, *_ in entities:
        gold_by_label[label].add(
            (str(doc_id), int(start), int(end), label)
        )

for doc_id, entities in fresh_val_predictions.items():
    for start, end, label in entities:
        pred_by_label[label].add(
            (str(doc_id), int(start), int(end), label)
        )

per_label_f1 = {}
print(f"{'Label':<12}{'Gold':>7}{'Pred':>7}{'TP':>7}{'F1':>9}")

for label in ENTITY_LABELS:
    g, p = gold_by_label[label], pred_by_label[label]
    tp = len(g & p)
    fp = len(p - g)
    fn = len(g - p)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    per_label_f1[label] = f1

    print(f"{label:<12}{len(g):>7}{len(p):>7}{tp:>7}{f1:>9.3f}")

fresh_macro_f1 = float(np.mean([per_label_f1[x] for x in ENTITY_LABELS]))
print(f"\nFresh exact-span Macro-F1: {fresh_macro_f1:.4f}")


In [ ]:

import inspect
from collections import Counter, defaultdict

print("BIO_LABELS:")
print(BIO_LABELS)

print("\nDecoder implementation:")
try:
    print(inspect.getsource(decode_spans))
except Exception as e:
    print("Could not inspect decoder:", e)

pred_lengths = defaultdict(list)
pred_counts = Counter()
pred_chars = Counter()

for doc_id, spans in fresh_val_predictions.items():
    for start, end, label in spans:
        pred_counts[label] += 1
        pred_lengths[label].append(end - start)
        pred_chars[label] += end - start

print("\nPredicted span-length statistics:")
print(f"{'Label':<12}{'Count':>8}{'Mean len':>11}{'Median len':>13}{'1-char %':>11}")

for label in ENTITY_LABELS:
    lengths = pred_lengths[label]
    if lengths:
        print(
            f"{label:<12}{len(lengths):>8}"
            f"{sum(lengths)/len(lengths):>11.1f}"
            f"{sorted(lengths)[len(lengths)//2]:>13}"
            f"{100*sum(x == 1 for x in lengths)/len(lengths):>10.1f}%"
        )
    else:
        print(f"{label:<12}{0:>8}")

print("\nFirst 30 predictions from one validation document:")
sample_id = next(iter(fresh_val_predictions))
sample_text = val_texts[sample_id]

for start, end, label in fresh_val_predictions[sample_id][:30]:
    print(
        f"{label:<12} [{start:>6}:{end:<6}] "
        f"{sample_text[start:end]!r}"
    )


In [ ]:

from collections import Counter
import torch

ner_model.eval()

sample_id = next(iter(val_texts))
sample_text = val_texts[sample_id]
enc = tokenizer.encode(sample_text, add_special_tokens=False)
ids, offsets = enc.ids, enc.offsets

raw_bio_counts = Counter()

for left in range(0, len(ids), WINDOW_STRIDE):
    right = min(left + MAX_LEN, len(ids))
    if left >= right:
        break

    input_ids = torch.tensor(
        [ids[left:right]], dtype=torch.long, device=DEVICE
    )
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        logits = ner_model(input_ids, attention_mask)

    pred_ids = logits[0].argmax(dim=-1).detach().cpu().tolist()

    for pred_id, (start, end) in zip(pred_ids, offsets[left:right]):
        if end > start:
            raw_bio_counts[BIO_LABELS[int(pred_id)]] += 1

    if right == len(ids):
        break

print("Raw predicted BIO-tag distribution:")
for tag in BIO_LABELS:
    print(f"{tag:<16} {raw_bio_counts[tag]:>10}")


In [ ]:

def audit_entity_overlaps(gold_by_doc):
    total_entities = 0
    docs_with_overlap = 0
    total_conflicts = 0
    examples = []

    for doc_id, entities in gold_by_doc.items():
        spans = sorted(
            (int(e[0]), int(e[1]), str(e[2]), str(e[3]) if len(e) > 3 else "")
            for e in entities
        )
        total_entities += len(spans)
        doc_has_overlap = False

        for i, a in enumerate(spans):
            for b in spans[i + 1:]:
                if b[0] >= a[1]:
                    break
                if max(a[0], b[0]) < min(a[1], b[1]):
                    total_conflicts += 1
                    doc_has_overlap = True
                    if len(examples) < 10:
                        examples.append((doc_id, a, b))

        if doc_has_overlap:
            docs_with_overlap += 1

    print("Documents:", len(gold_by_doc))
    print("Entities:", total_entities)
    print("Documents with overlapping entities:", docs_with_overlap)
    print("Overlapping entity pairs:", total_conflicts)
    print("\nExamples:")
    for doc_id, a, b in examples:
        print(doc_id, a, "<->", b)

audit_entity_overlaps(train_gold)


In [ ]:

from collections import Counter
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 512
WINDOW_STRIDE = 448
IGNORE_INDEX = -100

def make_unique_loss_windows(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    tags = [label2id["O"]] * len(ids)

    # Assign gold BIO tags to overlapping tokens.
    for entity in entities:
        start, end, label = int(entity[0]), int(entity[1]), str(entity[2])
        overlapping = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]
        if not overlapping:
            continue

        tags[overlapping[0]] = label2id[f"B-{label}"]
        for i in overlapping[1:]:
            tags[i] = label2id[f"I-{label}"]

    windows = []
    for start in range(0, len(ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break

        window_labels = tags[start:end].copy()

        # Earlier windows own tokens in the overlap.
        if start > 0:
            overlap = MAX_LEN - WINDOW_STRIDE
            for j in range(min(overlap, len(window_labels))):
                window_labels[j] = IGNORE_INDEX

        # Avoid a window that contributes no training targets.
        if any(x != IGNORE_INDEX for x in window_labels):
            windows.append({
                "input_ids": ids[start:end],
                "labels": window_labels,
            })

        if end == len(ids):
            break

    return windows


train_windows_v2 = []
for doc_id, text in train_texts.items():
    train_windows_v2.extend(
        make_unique_loss_windows(text, train_gold[doc_id])
    )

val_windows_v2 = []
for doc_id, text in val_texts.items():
    val_windows_v2.extend(
        make_unique_loss_windows(text, val_gold[doc_id])
    )

print("Train windows:", len(train_windows_v2))
print("Validation windows:", len(val_windows_v2))

counts = Counter()
for window in train_windows_v2:
    counts.update(
        int(x) for x in window["labels"] if x != IGNORE_INDEX
    )

print("\nTraining target counts:")
for i, tag in enumerate(BIO_LABELS):
    print(f"{tag:<16} {counts[i]:>9}")


In [ ]:

import math
import torch
from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]


def collate_ner(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        ids = item["input_ids"]
        labs = item["labels"]
        pad = max_len - len(ids)

        input_ids.append(ids + [PAD_ID] * pad)
        attention_masks.append([1] * len(ids) + [0] * pad)
        labels.append(labs + [IGNORE_INDEX] * pad)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


BATCH_SIZE = 2

train_loader_v2 = DataLoader(
    WindowDataset(train_windows_v2),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_ner,
)

# Count only tokens that actually contribute to loss.
freq = torch.tensor(
    [counts[i] for i in range(len(BIO_LABELS))],
    dtype=torch.float32,
).clamp(min=1)

class_weights = torch.sqrt(freq.max() / freq).clamp(max=8.0)
class_weights = class_weights / class_weights.mean()

class_weights = class_weights.to(DEVICE)

print("Class weights:")
for tag, weight in zip(BIO_LABELS, class_weights.tolist()):
    print(f"{tag:<16} {weight:.3f}")

print("Training batches:", len(train_loader_v2))


In [ ]:

import copy
import json
import numpy as np
import torch
from pathlib import Path
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

# Reinitialize the NER model from the original Phase 1 LM.
# phase1_lm must still be the original pretrained language model.
ner_model_v2 = LegalNERModel(
    pretrained_lm=phase1_lm,
    num_labels=len(BIO_LABELS),
).to(DEVICE)

optimizer_v2 = AdamW(
    ner_model_v2.parameters(),
    lr=1e-5,
    weight_decay=0.01,
)

criterion_v2 = CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX,
)

EPOCHS_V2 = 4
GRAD_ACCUM_V2 = 8
BEST_PATH_V2 = Path(
    "/kaggle/working/phase2_ner_corrected/best_ner_model_v2.pt"
)
BEST_PATH_V2.parent.mkdir(parents=True, exist_ok=True)

best_f1_v2 = -1.0
best_state_v2 = None

for epoch in range(1, EPOCHS_V2 + 1):
    ner_model_v2.train()
    optimizer_v2.zero_grad(set_to_none=True)
    running_loss = 0.0

    for step, batch in enumerate(train_loader_v2, start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = ner_model_v2(input_ids, attention_mask)
        loss = criterion_v2(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1),
        )
        (loss / GRAD_ACCUM_V2).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM_V2 == 0 or step == len(train_loader_v2):
            torch.nn.utils.clip_grad_norm_(
                ner_model_v2.parameters(), max_norm=1.0
            )
            optimizer_v2.step()
            optimizer_v2.zero_grad(set_to_none=True)

    # Evaluate using the existing exact-span validation function.
    # Temporarily point predict_doc at this model if your function uses
    # the global ner_model variable.
    print(
        f"Epoch {epoch}/{EPOCHS_V2} | "
        f"mean training loss: {running_loss / max(len(train_loader_v2), 1):.4f}"
    )

    # IMPORTANT: evaluate this model using the validation loop below.
    # We will not select a checkpoint using training loss alone.


In [ ]:

import numpy as np
import torch
from collections import defaultdict
from pathlib import Path

ner_model_v2.eval()

# predict_doc() uses the global variable named ner_model.
old_model_reference = ner_model
ner_model = ner_model_v2

fresh_val_predictions_v2 = {}
try:
    for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
        fresh_val_predictions_v2[doc_id] = predict_doc(text)
        if i % 5 == 0 or i == len(val_texts):
            print(f"Validated {i}/{len(val_texts)}")
finally:
    ner_model = old_model_reference

gold_by_label = defaultdict(set)
pred_by_label = defaultdict(set)

for doc_id, entities in val_gold.items():
    for entity in entities:
        start, end, label = int(entity[0]), int(entity[1]), str(entity[2])
        gold_by_label[label].add((str(doc_id), start, end, label))

for doc_id, entities in fresh_val_predictions_v2.items():
    for start, end, label in entities:
        pred_by_label[label].add((str(doc_id), int(start), int(end), label))

scores_v2 = {}
print(f"\n{'Label':<12}{'Gold':>7}{'Pred':>7}{'TP':>7}{'F1':>9}")

for label in ENTITY_LABELS:
    g, p = gold_by_label[label], pred_by_label[label]
    tp = len(g & p)
    precision = tp / len(p) if p else 0.0
    recall = tp / len(g) if g else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    scores_v2[label] = f1
    print(f"{label:<12}{len(g):>7}{len(p):>7}{tp:>7}{f1:>9.3f}")

macro_f1_v2 = float(np.mean([scores_v2[x] for x in ENTITY_LABELS]))
total_pred_v2 = sum(len(x) for x in fresh_val_predictions_v2.values())

print(f"\nNew exact-span Macro-F1: {macro_f1_v2:.4f}")
print(f"Previous Macro-F1:       {0.0720136:.4f}")
print(f"Predicted entities:      {total_pred_v2}")
print(f"Mean entities/document:  {total_pred_v2 / len(val_texts):.1f}")

# Save this evaluated model separately; keep the original checkpoint untouched.
EVALUATED_PATH_V2 = Path(
    "/kaggle/working/phase2_ner_corrected/evaluated_ner_model_v2.pt"
)
torch.save({
    "model_state_dict": ner_model_v2.state_dict(),
    "bio_labels": BIO_LABELS,
    "entity_labels": ENTITY_LABELS,
    "val_exact_span_macro_f1": macro_f1_v2,
    "epoch": 4,
    "phase1_checkpoint": str(
        "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
    ),
}, EVALUATED_PATH_V2)

print("Saved evaluated checkpoint:", EVALUATED_PATH_V2)


In [ ]:

from collections import defaultdict
import numpy as np

def audit_alignment(texts, gold_by_doc, split_name):
    stats = defaultdict(lambda: {
        "total": 0, "exact": 0,
        "start_mismatch": 0, "end_mismatch": 0,
        "both_mismatch": 0, "uncovered": 0
    })
    examples = []

    for doc_id, text in texts.items():
        entities = gold_by_doc[doc_id]
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        for entity in entities:
            gs, ge, label = int(entity[0]), int(entity[1]), str(entity[2])
            s = stats[label]
            s["total"] += 1

            overlapping = [
                (i, a, b) for i, (a, b) in enumerate(offsets)
                if b > a and a < ge and b > gs
            ]

            if not overlapping:
                s["uncovered"] += 1
                if len(examples) < 30:
                    examples.append({
                        "split": split_name, "doc_id": doc_id,
                        "label": label, "gold": (gs, ge),
                        "token_bounds": None,
                        "gold_text": text[gs:ge],
                        "reason": "No overlapping token",
                    })
                continue

            ts = overlapping[0][1]
            te = overlapping[-1][2]
            start_bad = ts != gs
            end_bad = te != ge

            if not start_bad and not end_bad:
                s["exact"] += 1
            elif start_bad and end_bad:
                s["both_mismatch"] += 1
            elif start_bad:
                s["start_mismatch"] += 1
            else:
                s["end_mismatch"] += 1

            if (start_bad or end_bad) and len(examples) < 30:
                examples.append({
                    "split": split_name, "doc_id": doc_id,
                    "label": label, "gold": (gs, ge),
                    "token_bounds": (ts, te),
                    "start_error": ts - gs,
                    "end_error": te - ge,
                    "gold_text": text[gs:ge],
                    "token_text": text[ts:te],
                })

    print(f"\n=== {split_name}: token-boundary alignment ===")
    print(
        f"{'Label':<12}{'Total':>7}{'Exact':>7}"
        f"{'Exact %':>9}{'Start err':>11}{'End err':>10}"
        f"{'Both':>7}{'No token':>10}"
    )

    for label in ENTITY_LABELS:
        s = stats[label]
        n = s["total"]
        pct = 100 * s["exact"] / n if n else 0
        print(
            f"{label:<12}{n:>7}{s['exact']:>7}{pct:>8.1f}%"
            f"{s['start_mismatch']:>11}{s['end_mismatch']:>10}"
            f"{s['both_mismatch']:>7}{s['uncovered']:>10}"
        )

    total = sum(s["total"] for s in stats.values())
    exact = sum(s["exact"] for s in stats.values())
    print(f"\nOverall exact-boundary rate: {100*exact/max(total, 1):.2f}%")
    print(f"Entities audited: {total}")

    return stats, examples


train_alignment_stats, train_alignment_examples = audit_alignment(
    train_texts, train_gold, "FOLD 1 TRAIN"
)

val_alignment_stats, val_alignment_examples = audit_alignment(
    val_texts, val_gold, "FOLD 2 VALIDATION"
)


In [ ]:

def show_alignment_examples(examples, limit=25):
    for i, ex in enumerate(examples[:limit], start=1):
        print(f"\n{i}. {ex['split']} | doc={ex['doc_id']} | {ex['label']}")
        print("Gold bounds:  ", ex["gold"])
        print("Token bounds: ", ex["token_bounds"])

        if ex["token_bounds"] is not None:
            print("Start error:  ", ex["start_error"], "chars")
            print("End error:    ", ex["end_error"], "chars")

        print("Gold text:    ", repr(ex["gold_text"]))
        print("Token text:   ", repr(ex.get("token_text")))


print("TRAIN MISMATCHES")
show_alignment_examples(train_alignment_examples)

print("\nVALIDATION MISMATCHES")
show_alignment_examples(val_alignment_examples)


In [ ]:

from collections import defaultdict
import numpy as np

def boundary_error_summary(texts, gold_by_doc):
    errors = defaultdict(lambda: {"start": [], "end": []})

    for doc_id, text in texts.items():
        offsets = tokenizer.encode(
            text, add_special_tokens=False
        ).offsets

        for entity in gold_by_doc[doc_id]:
            gs, ge, label = int(entity[0]), int(entity[1]), str(entity[2])
            overlapping = [
                (a, b) for a, b in offsets
                if b > a and a < ge and b > gs
            ]
            if not overlapping:
                continue

            ts, te = overlapping[0][0], overlapping[-1][1]
            errors[label]["start"].append(ts - gs)
            errors[label]["end"].append(te - ge)

    print(
        f"{'Label':<12}{'Start median':>14}{'Start MAE':>12}"
        f"{'End median':>13}{'End MAE':>10}"
    )

    for label in ENTITY_LABELS:
        s = errors[label]["start"]
        e = errors[label]["end"]
        if not s:
            continue
        print(
            f"{label:<12}"
            f"{np.median(s):>14.1f}{np.mean(np.abs(s)):>12.2f}"
            f"{np.median(e):>13.1f}{np.mean(np.abs(e)):>10.2f}"
        )

print("FOLD 1")
boundary_error_summary(train_texts, train_gold)

print("\nFOLD 2")
boundary_error_summary(val_texts, val_gold)


In [ ]:

def trim_predicted_spans(text, spans):
    corrected = []
    for start, end, label in spans:
        while start < end and text[start].isspace():
            start += 1
        while end > start and text[end - 1].isspace():
            end -= 1
        if start < end:
            corrected.append((start, end, label))
    return sorted(set(corrected))


def evaluate_span_predictions(texts, gold_by_doc, predictions, title):
    from collections import defaultdict
    import numpy as np

    gold_sets = defaultdict(set)
    pred_sets = defaultdict(set)

    for doc_id, entities in gold_by_doc.items():
        for e in entities:
            gold_sets[str(e[2])].add(
                (str(doc_id), int(e[0]), int(e[1]), str(e[2]))
            )

    for doc_id, spans in predictions.items():
        for start, end, label in spans:
            pred_sets[str(label)].add(
                (str(doc_id), int(start), int(end), str(label))
            )

    f1s = {}
    print(f"\n=== {title} ===")
    print(f"{'Label':<12}{'Gold':>7}{'Pred':>7}{'TP':>7}{'F1':>9}")

    for label in ENTITY_LABELS:
        g, p = gold_sets[label], pred_sets[label]
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        f1s[label] = f1
        print(f"{label:<12}{len(g):>7}{len(p):>7}{tp:>7}{f1:>9.3f}")

    macro = float(np.mean([f1s[x] for x in ENTITY_LABELS]))
    total = sum(len(x) for x in predictions.values())
    print(f"Macro-F1: {macro:.4f}")
    print(f"Total predictions: {total}")
    return macro


# Re-evaluate the saved predictions without changing them.
original_macro = evaluate_span_predictions(
    val_texts, val_gold, fresh_val_predictions,
    "Original checkpoint, unmodified predictions"
)

trimmed_original = {
    doc_id: trim_predicted_spans(val_texts[doc_id], spans)
    for doc_id, spans in fresh_val_predictions.items()
}
trimmed_macro = evaluate_span_predictions(
    val_texts, val_gold, trimmed_original,
    "Original checkpoint, leading-whitespace trim"
)

print("\nChange in Macro-F1:", round(trimmed_macro - original_macro, 4))


In [ ]:

def inspect_near_misses(texts, gold_by_doc, predictions, limit=30):
    shown = 0
    for doc_id, entities in gold_by_doc.items():
        preds = predictions.get(doc_id, [])
        for entity in entities:
            gs, ge, gl = int(entity[0]), int(entity[1]), str(entity[2])

            exact = any(
                (int(ps), int(pe), pl) == (gs, ge, gl)
                for ps, pe, pl in preds
            )
            if exact:
                continue

            candidates = [
                (abs(int(ps) - gs) + abs(int(pe) - ge), ps, pe, pl)
                for ps, pe, pl in preds
                if pl == gl and abs(int(ps) - gs) <= 3 and abs(int(pe) - ge) <= 3
            ]
            if candidates:
                _, ps, pe, pl = min(candidates)
                print(
                    f"doc={doc_id} label={gl} "
                    f"gold=({gs},{ge}) pred=({ps},{pe}) "
                    f"gold_text={texts[doc_id][gs:ge]!r} "
                    f"pred_text={texts[doc_id][ps:pe]!r}"
                )
                shown += 1
                if shown >= limit:
                    return

inspect_near_misses(
    val_texts, val_gold, trimmed_original, limit=30
)


In [ ]:

import string

TRAILING_PUNCT = set(".,;:!?)]}")

def trim_trailing_punctuation(text, spans):
    out = []
    for start, end, label in spans:
        # Remove trailing punctuation only when doing so leaves non-punctuation text.
        while end > start and text[end - 1] in TRAILING_PUNCT:
            end -= 1
        if start < end:
            out.append((start, end, label))
    return sorted(set(out))


trimmed_punct_predictions = {
    doc_id: trim_trailing_punctuation(val_texts[doc_id], spans)
    for doc_id, spans in fresh_val_predictions.items()
}

punct_macro = evaluate_span_predictions(
    val_texts, val_gold, trimmed_punct_predictions,
    "Original model + trailing-punctuation trim"
)

print("\nBaseline Macro-F1:", round(original_macro, 4))
print("Punctuation-trim Macro-F1:", round(punct_macro, 4))
print("Change:", round(punct_macro - original_macro, 4))


In [ ]:

from collections import Counter

error_types = Counter()

for doc_id, entities in val_gold.items():
    preds = trimmed_punct_predictions[doc_id]

    for entity in entities:
        gs, ge, gl = int(entity[0]), int(entity[1]), str(entity[2])

        exact = any(
            (int(ps), int(pe), pl) == (gs, ge, gl)
            for ps, pe, pl in preds
        )
        if exact:
            continue

        same_boundary_wrong_label = any(
            int(ps) == gs and int(pe) == ge and pl != gl
            for ps, pe, pl in preds
        )
        same_label_near_boundary = any(
            pl == gl
            and abs(int(ps) - gs) <= 3
            and abs(int(pe) - ge) <= 3
            for ps, pe, pl in preds
        )

        if same_boundary_wrong_label:
            error_types["Exact boundary, wrong label"] += 1
        elif same_label_near_boundary:
            error_types["Correct label, near boundary"] += 1
        else:
            error_types["No close matching prediction"] += 1

print("Unmatched gold entity error categories:")
for name, count in error_types.most_common():
    print(f"{name:<36} {count}")


In [ ]:

def trim_by_label(text, spans, enabled_labels, punctuation):
    result = []

    for start, end, label in spans:
        start, end = int(start), int(end)

        if label in enabled_labels:
            while end > start and text[end - 1] in punctuation:
                end -= 1

        if start < end:
            result.append((start, end, label))

    return sorted(set(result))


def score_per_label(predictions):
    from collections import defaultdict

    gold = defaultdict(set)
    pred = defaultdict(set)

    for doc_id, entities in val_gold.items():
        for e in entities:
            label = str(e[2])
            gold[label].add(
                (str(doc_id), int(e[0]), int(e[1]), label)
            )

    for doc_id, spans in predictions.items():
        for start, end, label in spans:
            label = str(label)
            pred[label].add(
                (str(doc_id), int(start), int(end), label)
            )

    scores = {}
    for label in ENTITY_LABELS:
        g, p = gold[label], pred[label]
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0

        scores[label] = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

    return scores


baseline_scores = score_per_label(fresh_val_predictions)

print(f"{'Label':<12}{'Baseline':>11}{'Trimmed':>11}{'Change':>11}")

for label in ENTITY_LABELS:
    one_label_trim = {
        doc_id: trim_by_label(
            val_texts[doc_id],
            spans,
            {label},
            set(".,;:!?)]}")
        )
        for doc_id, spans in fresh_val_predictions.items()
    }

    trimmed_scores = score_per_label(one_label_trim)

    print(
        f"{label:<12}{baseline_scores[label]:>11.3f}"
        f"{trimmed_scores[label]:>11.3f}"
        f"{trimmed_scores[label] - baseline_scores[label]:>+11.3f}"
    )


In [ ]:

def score_per_label(predictions):
    from collections import defaultdict

    gold = defaultdict(set)
    pred = defaultdict(set)

    for doc_id, entities in val_gold.items():
        for e in entities:
            label = str(e[2])
            gold[label].add(
                (str(doc_id), int(e[0]), int(e[1]), label)
            )

    for doc_id, spans in predictions.items():
        for start, end, label in spans:
            label = str(label)
            pred[label].add(
                (str(doc_id), int(start), int(end), label)
            )

    result = {}
    for label in ENTITY_LABELS:
        g = gold[label]
        p = pred[label]
        tp = len(g & p)

        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0

        result[label] = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

    return result


baseline_scores = score_per_label(fresh_val_predictions)

print(f"{'Label':<12}{'Baseline':>11}{'Trimmed':>11}{'Change':>11}")

for label in ENTITY_LABELS:
    one_label_trim = {
        doc_id: trim_by_label(
            val_texts[doc_id],
            spans,
            {label},
            set(".,;:!?)]}")
        )
        for doc_id, spans in fresh_val_predictions.items()
    }

    scores = score_per_label(one_label_trim)

    print(
        f"{label:<12}{baseline_scores[label]:>11.3f}"
        f"{scores[label]:>11.3f}"
        f"{scores[label] - baseline_scores[label]:>+11.3f}"
    )


In [ ]:

COMBINED_TRIM_LABELS = {"DATE", "PREC", "CASENO", "STAT"}
TRAILING_PUNCT = set(".,;:!?)]}")

combined_trim_predictions = {
    doc_id: trim_by_label(
        val_texts[doc_id],
        spans,
        COMBINED_TRIM_LABELS,
        TRAILING_PUNCT,
    )
    for doc_id, spans in fresh_val_predictions.items()
}

combined_macro = evaluate_span_predictions(
    val_texts,
    val_gold,
    combined_trim_predictions,
    "Combined label-specific punctuation trim",
)

print("\nOriginal Macro-F1:       ", round(original_macro, 4))
print("Global trim Macro-F1:    ", round(punct_macro, 4))
print("Combined trim Macro-F1:  ", round(combined_macro, 4))


In [ ]:

import json
from pathlib import Path

# Ensure inference uses the original model, not the experimental v2 model.
ner_model.eval()

test_predictions_combined = {}
for i, (doc_id, text) in enumerate(test_text.items(), start=1):
    raw_spans = predict_doc(text)
    test_predictions_combined[str(doc_id)] = [
        [start, end, label]
        for start, end, label in trim_by_label(
            text,
            raw_spans,
            COMBINED_TRIM_LABELS,
            TRAILING_PUNCT,
        )
    ]

    if i % 5 == 0 or i == len(test_text):
        print(f"Processed {i}/{len(test_text)} test documents")

out_path = Path("/kaggle/working/prediction_punct_trim.json")
with out_path.open("w", encoding="utf-8") as f:
    json.dump(test_predictions_combined, f, ensure_ascii=False)

print("Saved:", out_path)
print("Documents:", len(test_predictions_combined))
print(
    "Total entities:",
    sum(len(spans) for spans in test_predictions_combined.values())
)


In [ ]:

DATE_ONLY = {"DATE"}

date_only_predictions = {
    doc_id: trim_by_label(
        val_texts[doc_id],
        spans,
        DATE_ONLY,
        set(".,;:!?)]}")
    )
    for doc_id, spans in fresh_val_predictions.items()
}

date_only_macro = evaluate_span_predictions(
    val_texts,
    val_gold,
    date_only_predictions,
    "DATE-only trailing-punctuation trim",
)

print("\nOriginal:", round(original_macro, 4))
print("DATE-only:", round(date_only_macro, 4))
print("Combined:", round(combined_macro, 4))


In [ ]:

import json
from pathlib import Path

# Restore the original checkpoint if ner_model currently points to v2.
original_ckpt_path = Path(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
)
saved = torch.load(
    original_ckpt_path,
    map_location=DEVICE,
    weights_only=False,
)
ner_model.load_state_dict(saved["model_state_dict"], strict=True)
ner_model.eval()

COMBINED_TRIM_LABELS = {"DATE", "PREC", "CASENO", "STAT"}
TRAILING_PUNCT = set(".,;:!?)]}")

test_predictions_combined = {}

for i, (doc_id, text) in enumerate(test_text.items(), start=1):
    raw_spans = predict_doc(text)
    corrected = trim_by_label(
        text,
        raw_spans,
        COMBINED_TRIM_LABELS,
        TRAILING_PUNCT,
    )

    test_predictions_combined[str(doc_id)] = [
        [int(start), int(end), str(label)]
        for start, end, label in corrected
    ]

    if i % 5 == 0 or i == len(test_text):
        print(f"Processed {i}/{len(test_text)} test documents")

out_path = Path("/kaggle/working/prediction_punct_trim.json")
with out_path.open("w", encoding="utf-8") as f:
    json.dump(test_predictions_combined, f, ensure_ascii=False)

print("\nSaved:", out_path)
print("Documents:", len(test_predictions_combined))
print(
    "Total entities:",
    sum(len(spans) for spans in test_predictions_combined.values())
)

# Validate schema and character bounds.
assert len(test_predictions_combined) == len(test_text)

for doc_id, spans in test_predictions_combined.items():
    text = test_text[doc_id]
    for start, end, label in spans:
        assert 0 <= start < end <= len(text), (
            doc_id, start, end, len(text)
        )
        assert label in ENTITY_LABELS, (doc_id, label)

print("Schema and character-bound checks passed.")


In [ ]:

import torch
from pathlib import Path

PHASE1_PATH = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase-1-checkpoints/checkpoint_latest.pt"
)

phase1_ckpt = torch.load(
    PHASE1_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint type:", type(phase1_ckpt))

if isinstance(phase1_ckpt, dict):
    print("Top-level keys:", list(phase1_ckpt.keys()))

    for key, value in phase1_ckpt.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            print("First keys:", list(value.keys())[:8])
        elif torch.is_tensor(value):
            print(f"{key}: tensor {tuple(value.shape)}")
        else:
            print(f"{key}: {type(value).__name__}", str(value)[:120])


In [ ]:

from collections import Counter
import torch

IGNORE_INDEX = -100

label_counts = Counter()
for window in train_windows_v2:
    label_counts.update(
        int(label)
        for label in window["labels"]
        if int(label) != IGNORE_INDEX
    )

freq = torch.tensor(
    [label_counts[i] for i in range(len(BIO_LABELS))],
    dtype=torch.float32,
).clamp(min=1)

# Inverse-square-root frequency, capped relative to the O class.
weights = torch.sqrt(freq[0] / freq)
weights = weights.clamp(min=0.5, max=8.0)

print("Class weights:")
for label, weight in zip(BIO_LABELS, weights.tolist()):
    print(f"{label:<16} {weight:.3f}")

print("\nO weight:", round(weights[0].item(), 3))
print("Minimum weight:", round(weights.min().item(), 3))
print("Maximum weight:", round(weights.max().item(), 3))


In [ ]:
import torch

print("Saved Phase 1 configuration:")
print(phase1_ckpt["model_config"])

state = phase1_ckpt["model_state_dict"]

print("\nEmbedding shapes:")
print("Token embeddings:", state["transformer.wte.weight"].shape)
print("Position embeddings:", state["transformer.wpe.weight"].shape)
print("First attention layer:", state["transformer.h.0.attn.c_attn.weight"].shape)

assert state["transformer.wte.weight"].shape[0] == 32000
assert state["transformer.wte.weight"].shape[1] == 768
assert state["transformer.wpe.weight"].shape[0] == 4096

print("\nCheckpoint architecture checks passed.")

In [ ]:
from collections import OrderedDict
import torch
from transformers import GPT2Model

state = phase1_ckpt["model_state_dict"]

mapped_state = OrderedDict()

for key, value in state.items():
    if key == "lm_head.weight":
        continue  # Not part of GPT2Model's transformer backbone

    if key.startswith("transformer."):
        mapped_key = key[len("transformer."):]
    else:
        mapped_key = key

    mapped_state[mapped_key] = value

phase1_fresh = GPT2Model(gpt2_config)

# Strict loading should now pass.
phase1_fresh.load_state_dict(mapped_state, strict=True)

phase1_fresh = phase1_fresh.to(device)
phase1_fresh.eval()

print("Successfully loaded Phase 1 transformer weights.")
print("Device:", next(phase1_fresh.parameters()).device)

In [ ]:
test_ids = torch.tensor([[1, 2, 3, 4]], dtype=torch.long, device=device)
test_mask = torch.ones_like(test_ids)

with torch.no_grad():
    out = phase1_fresh(
        input_ids=test_ids,
        attention_mask=test_mask
    )

print("Output shape:", out.last_hidden_state.shape)
assert out.last_hidden_state.shape == (1, 4, 768)
print("Backbone smoke test passed.")

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Config

cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

phase1_fresh = GPT2LMHeadModel(gpt2_config)

# The original checkpoint includes both transformer and LM-head weights.
phase1_fresh.load_state_dict(
    phase1_ckpt["model_state_dict"],
    strict=True
)

phase1_fresh = phase1_fresh.to(device)

print("Phase 1 checkpoint loaded strictly.")
print("Backbone type:", type(phase1_fresh).__name__)
print("Has .transformer:", hasattr(phase1_fresh, "transformer"))

In [ ]:
ner_model = LegalNERModel(
    pretrained_lm=phase1_fresh,
    num_labels=len(BIO_LABELS)
).to(device)

sample = train_windows_v2[0]

sample_ids = torch.as_tensor(
    sample["input_ids"], dtype=torch.long, device=device
).reshape(1, -1)

sample_mask = (
    torch.as_tensor(
        sample["attention_mask"], dtype=torch.long, device=device
    ).reshape(1, -1)
    if "attention_mask" in sample
    else torch.ones_like(sample_ids)
)

ner_model.eval()

with torch.no_grad():
    output = ner_model(
        input_ids=sample_ids,
        attention_mask=sample_mask
    )

if isinstance(output, dict):
    logits = output["logits"]
elif hasattr(output, "logits"):
    logits = output.logits
elif isinstance(output, (tuple, list)):
    logits = output[0]
else:
    logits = output

print("Input shape:", tuple(sample_ids.shape))
print("Logits shape:", tuple(logits.shape))

assert logits.shape == (
    1, sample_ids.shape[1], len(BIO_LABELS)
)
print("NER forward-pass check passed.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from pathlib import Path

IGNORE_INDEX = -100
MAX_EPOCHS = 4
BATCH_SIZE = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

SAVE_DIR = Path("/kaggle/working/phase2_ner_experiment")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

class_weights = weights.to(device=device, dtype=torch.float32)

assert len(class_weights) == len(BIO_LABELS)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX
)

optimizer = torch.optim.AdamW(
    ner_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Train windows:", len(train_windows_v2))
print("Labels:", len(BIO_LABELS))
print("Device:", device)
print("Checkpoint directory:", SAVE_DIR)

In [ ]:
def collate_ner_windows(batch):
    ids_list = [
        torch.as_tensor(x["input_ids"], dtype=torch.long)
        for x in batch
    ]
    labels_list = [
        torch.as_tensor(x["labels"], dtype=torch.long)
        for x in batch
    ]

    input_ids = pad_sequence(
        ids_list, batch_first=True, padding_value=0
    )
    labels = pad_sequence(
        labels_list, batch_first=True, padding_value=IGNORE_INDEX
    )

    attention_mask = torch.zeros_like(input_ids)
    for i, ids in enumerate(ids_list):
        attention_mask[i, :len(ids)] = 1

    return input_ids, attention_mask, labels


train_loader = DataLoader(
    train_windows_v2,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_ner_windows
)

def train_one_epoch(model):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(output, dict):
            logits = output["logits"]
        elif hasattr(output, "logits"):
            logits = output.logits
        elif isinstance(output, (tuple, list)):
            logits = output[0]
        else:
            logits = output

        loss = criterion(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), GRAD_CLIP
        )
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)


print("Training loader ready:", len(train_loader), "batches")

In [ ]:
# Cell 12: baseline validation
ner_model.eval()

baseline_predictions = {
    doc_id: predict_doc(text)
    for doc_id, text in val_texts.items()
}

evaluate_span_predictions(
    val_texts,
    val_gold,
    baseline_predictions,
    "Fresh Phase 1 baseline"
)

In [ ]:
# Cell 13: train, validate, save the best model
import copy
import torch

best_macro_f1 = -1.0
best_epoch = None
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(ner_model)

    ner_model.eval()
    val_predictions = {
        doc_id: predict_doc(text)
        for doc_id, text in val_texts.items()
    }

    print(f"\nEpoch {epoch}/{MAX_EPOCHS}")
    print(f"Training loss: {train_loss:.4f}")

    evaluate_span_predictions(
        val_texts,
        val_gold,
        val_predictions,
        f"Epoch {epoch} validation"
    )

    # Get exact-span macro-F1 from the existing evaluator's output
    # by calculating it directly from the prediction scorer.
    scores = score_per_label(val_predictions)

    # Expected score_per_label format is checked below if needed.
    print("Per-label score result type:", type(scores).__name__)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
    })

    # Save an epoch checkpoint regardless of score.
    torch.save({
        "epoch": epoch,
        "model_state_dict": ner_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "bio_labels": BIO_LABELS,
    }, SAVE_DIR / f"epoch_{epoch}.pt")

print("\nTraining finished.")
print("Epoch checkpoints saved in:", SAVE_DIR)

In [ ]:
import torch

best_epoch_path = SAVE_DIR / "epoch_3.pt"
checkpoint = torch.load(
    best_epoch_path,
    map_location=device,
    weights_only=False
)

ner_model.load_state_dict(checkpoint["model_state_dict"])
ner_model.eval()

epoch3_predictions = {
    doc_id: predict_doc(text)
    for doc_id, text in val_texts.items()
}

print("Epoch 3, untrimmed:")
evaluate_span_predictions(
    val_texts,
    val_gold,
    epoch3_predictions,
    "Epoch 3 — untrimmed"
)

TRIM_LABELS = {"DATE", "PREC", "CASENO", "STAT"}
TRAILING_PUNCT = set(".,;:!?)]}")

epoch3_trimmed = {
    doc_id: trim_by_label(
        val_texts[doc_id],
        spans,
        TRIM_LABELS,
        TRAILING_PUNCT
    )
    for doc_id, spans in epoch3_predictions.items()
}

print("\nEpoch 3, punctuation-trimmed:")
evaluate_span_predictions(
    val_texts,
    val_gold,
    epoch3_trimmed,
    "Epoch 3 — punctuation-trimmed"
)

In [ ]:
import json
from pathlib import Path

comparison = {
    "previous_corrected_model": {
        "untrimmed_macro_f1": 0.0720,
        "trimmed_macro_f1": 0.0789,
        "trimmed_predictions": 15349,
    },
    "new_epoch_3": {
        "untrimmed_macro_f1": 0.0646,
        "trimmed_macro_f1": 0.07237250397743235,
        "trimmed_predictions": 20454,
    },
}

comparison_path = SAVE_DIR / "validation_comparison.json"

with open(comparison_path, "w") as f:
    json.dump(comparison, f, indent=2)

print("Comparison saved to:", comparison_path)
print("Current champion: earlier corrected model")
print("Candidate checkpoint:", SAVE_DIR / "epoch_3.pt")

In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2LMHeadModel
from pathlib import Path
from collections import Counter
import copy
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PHASE1_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
)

phase1_ckpt = torch.load(
    PHASE1_PATH,
    map_location="cpu",
    weights_only=False
)

cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

fresh_lm = GPT2LMHeadModel(gpt2_config)
fresh_lm.load_state_dict(
    phase1_ckpt["model_state_dict"],
    strict=True
)

ner_model = LegalNERModel(
    pretrained_lm=fresh_lm,
    num_labels=len(BIO_LABELS)
).to(device)

print("Fresh Phase 1 weights loaded.")
print("Device:", device)
print("NER labels:", len(BIO_LABELS))

In [ ]:
IGNORE_INDEX = -100

label_counts = Counter()

for window in train_windows_v2:
    label_counts.update(
        int(label)
        for label in window["labels"]
        if int(label) != IGNORE_INDEX
    )

freq = torch.tensor(
    [label_counts[i] for i in range(len(BIO_LABELS))],
    dtype=torch.float32
).clamp(min=1)

# Keep O at 1 and limit the entity-tag weights.
class_weights = torch.sqrt(freq[0] / freq)
class_weights = class_weights.clamp(min=1.0, max=3.0)
class_weights[0] = 1.0

print("Moderate class weights:")
for label, weight in zip(BIO_LABELS, class_weights.tolist()):
    print(f"{label:15s} {weight:.3f}")

In [ ]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

MAX_EPOCHS = 5
BATCH_SIZE = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    ignore_index=IGNORE_INDEX
)

optimizer = torch.optim.AdamW(
    ner_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

def collate_ner_windows(batch):
    ids_list = [
        torch.as_tensor(x["input_ids"], dtype=torch.long)
        for x in batch
    ]
    labels_list = [
        torch.as_tensor(x["labels"], dtype=torch.long)
        for x in batch
    ]

    input_ids = pad_sequence(
        ids_list, batch_first=True, padding_value=0
    )
    labels = pad_sequence(
        labels_list, batch_first=True,
        padding_value=IGNORE_INDEX
    )

    attention_mask = torch.zeros_like(input_ids)
    for i, ids in enumerate(ids_list):
        attention_mask[i, :len(ids)] = 1

    return input_ids, attention_mask, labels

train_loader = DataLoader(
    train_windows_v2,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_ner_windows
)

SAVE_DIR_B = Path("/kaggle/working/phase2_ner_experiment_B")
SAVE_DIR_B.mkdir(parents=True, exist_ok=True)

print("Batches per epoch:", len(train_loader))
print("Epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)

In [ ]:
def train_one_epoch_B():
    ner_model.train()
    total_loss = 0.0
    batches = 0

    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        output = ner_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(output, dict):
            logits = output["logits"]
        elif hasattr(output, "logits"):
            logits = output.logits
        elif isinstance(output, (tuple, list)):
            logits = output[0]
        else:
            logits = output

        loss = criterion(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            ner_model.parameters(), GRAD_CLIP
        )
        optimizer.step()

        total_loss += loss.item()
        batches += 1

    return total_loss / max(batches, 1)

print("Training function ready.")

In [ ]:
ENTITY_LABELS = [
    label for label in BIO_LABELS
    if label != "O" and label.startswith("B-")
]
ENTITY_LABELS = [label[2:] for label in ENTITY_LABELS]

TRIM_LABELS = {"DATE", "PREC", "CASENO", "STAT"}
TRAILING_PUNCT = set(".,;:!?)]}")

def normalize_spans(spans):
    normalized = set()
    for span in spans:
        start, end, label = int(span[0]), int(span[1]), str(span[2])
        normalized.add((start, end, label))
    return normalized

def exact_span_scores(gold_by_doc, pred_by_doc):
    counts = {
        label: {"gold": 0, "pred": 0, "tp": 0}
        for label in ENTITY_LABELS
    }

    for doc_id, gold_spans in gold_by_doc.items():
        gold_set = normalize_spans(gold_spans)
        pred_set = normalize_spans(pred_by_doc.get(doc_id, []))

        for label in ENTITY_LABELS:
            g = {s for s in gold_set if s[2] == label}
            p = {s for s in pred_set if s[2] == label}

            counts[label]["gold"] += len(g)
            counts[label]["pred"] += len(p)
            counts[label]["tp"] += len(g & p)

    f1s = []
    for label in ENTITY_LABELS:
        c = counts[label]
        denom = c["gold"] + c["pred"]
        f1 = (2 * c["tp"] / denom) if denom else 0.0
        c["f1"] = f1
        f1s.append(f1)

    return sum(f1s) / len(f1s), counts

def trim_predictions(texts, predictions):
    trimmed = {}
    for doc_id, spans in predictions.items():
        trimmed[doc_id] = trim_by_label(
            texts[doc_id],
            spans,
            TRIM_LABELS,
            TRAILING_PUNCT
        )
    return trimmed

print("Exact-span scorer ready.")

In [ ]:
best_f1_B = -1.0
best_epoch_B = None
history_B = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch_B()

    ner_model.eval()
    val_predictions = {
        doc_id: predict_doc(text)
        for doc_id, text in val_texts.items()
    }

    val_predictions_trimmed = trim_predictions(
        val_texts, val_predictions
    )

    macro_f1, per_label = exact_span_scores(
        val_gold, val_predictions_trimmed
    )

    print(f"\nEpoch {epoch}/{MAX_EPOCHS}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Trimmed exact-span macro-F1: {macro_f1:.4f}")
    print(f"Predicted spans: {sum(len(x) for x in val_predictions_trimmed.values())}")

    for label in ENTITY_LABELS:
        c = per_label[label]
        print(
            f"{label:12s} "
            f"Gold={c['gold']:4d} "
            f"Pred={c['pred']:5d} "
            f"TP={c['tp']:4d} "
            f"F1={c['f1']:.3f}"
        )

    history_B.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "trimmed_macro_f1": macro_f1,
    })

    torch.save({
        "epoch": epoch,
        "model_state_dict": ner_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "trimmed_macro_f1": macro_f1,
        "bio_labels": BIO_LABELS,
    }, SAVE_DIR_B / f"epoch_{epoch}.pt")

    if macro_f1 > best_f1_B:
        best_f1_B = macro_f1
        best_epoch_B = epoch

        torch.save({
            "epoch": epoch,
            "model_state_dict": ner_model.state_dict(),
            "trimmed_macro_f1": macro_f1,
            "bio_labels": BIO_LABELS,
        }, SAVE_DIR_B / "best_model.pt")

        print("Saved new best checkpoint.")

with open(SAVE_DIR_B / "history.json", "w") as f:
    json.dump(history_B, f, indent=2)

print("\nExperiment B complete.")
print("Best epoch:", best_epoch_B)
print("Best trimmed validation macro-F1:", round(best_f1_B, 4))
print("Saved to:", SAVE_DIR_B)

In [ ]:
import inspect

for name in [
    "make_unique_loss_windows",
    "train_windows_v2",
    "decode_spans",
    "predict_doc",
    "tokenizer",
]:
    if name not in globals():
        print(f"{name}: NOT FOUND")
        continue

    obj = globals()[name]
    print(f"\n--- {name} ---")

    if callable(obj) and name != "tokenizer":
        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Source unavailable:", obj)
    elif name == "train_windows_v2":
        print("Window count:", len(obj))
        print("First window keys:", obj[0].keys())
    else:
        print(type(obj))

In [ ]:
from collections import Counter
import numpy as np

def audit_alignment(texts, gold_by_doc, tokenizer):
    stats = Counter()
    examples = []

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        for entity in gold_by_doc[doc_id]:
            start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

            overlapping = [
                (i, a, b)
                for i, (a, b) in enumerate(offsets)
                if b > a and a < end and b > start
            ]

            stats["entities"] += 1

            if not overlapping:
                stats["no_overlapping_tokens"] += 1
                if len(examples) < 10:
                    examples.append((doc_id, text[start:end], start, end, []))
                continue

            first_i, first_a, first_b = overlapping[0]
            last_i, last_a, last_b = overlapping[-1]

            exact_start = first_a == start
            exact_end = last_b == end

            stats["exact_start"] += exact_start
            stats["exact_end"] += exact_end
            stats["exact_both"] += exact_start and exact_end

            # Tokens extending beyond either entity boundary
            left_overhang = max(0, start - first_a)
            right_overhang = max(0, last_b - end)

            stats["left_overhang_chars"] += left_overhang
            stats["right_overhang_chars"] += right_overhang
            stats["partial_boundary_entities"] += (
                left_overhang > 0 or right_overhang > 0
            )

            if (left_overhang > 0 or right_overhang > 0) and len(examples) < 10:
                examples.append((
                    doc_id, text[start:end], start, end,
                    [(a, b, text[a:b]) for _, a, b in overlapping]
                ))

    n = stats["entities"]
    print("Entities audited:", n)
    print("No overlapping token:", stats["no_overlapping_tokens"])
    print("Exact start:", stats["exact_start"], f"({stats['exact_start']/max(n,1):.1%})")
    print("Exact end:", stats["exact_end"], f"({stats['exact_end']/max(n,1):.1%})")
    print("Exact both boundaries:", stats["exact_both"], f"({stats['exact_both']/max(n,1):.1%})")
    print("Entities with partial token boundaries:", stats["partial_boundary_entities"])
    print("Left overhang characters:", stats["left_overhang_chars"])
    print("Right overhang characters:", stats["right_overhang_chars"])

    print("\nBoundary examples:")
    for ex in examples:
        print(ex)

audit_alignment(train_texts, train_gold, tokenizer)

In [ ]:
def audit_window_boundaries(texts, gold_by_doc):
    crossing = 0
    near_boundary = 0
    total = 0
    examples = []

    for doc_id, text in texts.items():
        ids = tokenizer.encode(text, add_special_tokens=False).ids
        n_tokens = len(ids)

        if n_tokens == 0:
            continue

        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        for entity in gold_by_doc[doc_id]:
            start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

            token_ids = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < end and b > start
            ]

            if not token_ids:
                continue

            total += 1
            first_token, last_token = min(token_ids), max(token_ids)

            # Boundaries where window ownership changes
            boundaries = list(range(WINDOW_STRIDE, n_tokens, WINDOW_STRIDE))

            if any(first_token < b <= last_token for b in boundaries):
                crossing += 1
                if len(examples) < 10:
                    examples.append((
                        doc_id, text[start:end], first_token,
                        last_token, boundaries
                    ))

            if any(abs(first_token - b) <= 4 or abs(last_token - b) <= 4
                   for b in boundaries):
                near_boundary += 1

    print("Entities audited:", total)
    print("Crossing a stride boundary:", crossing)
    print("Near a stride boundary (within 4 tokens):", near_boundary)
    print("\nExamples:", examples)

audit_window_boundaries(train_texts, train_gold)

In [ ]:
# Inspect one entity with a leading-token overhang
doc_id = "115651329"
text = train_texts[doc_id]
gold_entities = train_gold[doc_id]

enc = tokenizer.encode(text, add_special_tokens=False)

for entity in gold_entities[:10]:
    start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

    overlapping = [
        (i, a, b, text[a:b])
        for i, (a, b) in enumerate(enc.offsets)
        if b > a and a < end and b > start
    ]

    print("\nGold:", (start, end, label, text[start:end]))
    print("Overlapping tokens:")
    for item in overlapping:
        print(item)

In [ ]:
def make_contained_loss_windows(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    tags = [label2id["O"]] * len(ids)

    for entity in entities:
        start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

        contained = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a >= start and b <= end
        ]

        if not contained:
            continue

        tags[contained[0]] = label2id[f"B-{label}"]

        for i in contained[1:]:
            tags[i] = label2id[f"I-{label}"]

    windows = []

    for start in range(0, len(ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break

        window_labels = tags[start:end].copy()

        if start > 0:
            overlap = MAX_LEN - WINDOW_STRIDE
            for j in range(min(overlap, len(window_labels))):
                window_labels[j] = IGNORE_INDEX

        if any(x != IGNORE_INDEX for x in window_labels):
            windows.append({
                "input_ids": ids[start:end],
                "labels": window_labels,
            })

        if end == len(ids):
            break

    return windows


print("Boundary-aware window builder defined.")

In [ ]:
contained_train_windows = []

for doc_id, text in train_texts.items():
    contained_train_windows.extend(
        make_contained_loss_windows(text, train_gold[doc_id])
    )

print("Original windows:", len(train_windows_v2))
print("Boundary-aware windows:", len(contained_train_windows))

def count_entity_targets(windows):
    counts = Counter()

    for window in windows:
        for tag_id in window["labels"]:
            if tag_id != IGNORE_INDEX:
                tag = BIO_LABELS[int(tag_id)]
                if tag != "O":
                    counts[tag] += 1

    return counts

original_counts = count_entity_targets(train_windows_v2)
contained_counts = count_entity_targets(contained_train_windows)

print("\nTag counts: original vs boundary-aware")
for tag in BIO_LABELS:
    if tag != "O":
        print(
            f"{tag:15s} "
            f"{original_counts[tag]:6d} -> {contained_counts[tag]:6d}"
        )

In [ ]:
from collections import Counter

def boundary_correction_audit(texts, gold_by_doc):
    stats = Counter()

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)

        for entity in gold_by_doc[doc_id]:
            start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

            overlapping = [
                (a, b) for a, b in enc.offsets
                if b > a and a < end and b > start
            ]
            if not overlapping:
                continue

            first_start = overlapping[0][0]
            last_end = overlapping[-1][1]

            stats["total"] += 1
            stats["start_mismatch"] += first_start != start
            stats["end_mismatch"] += last_end != end

            # Current decoder trims whitespace from the predicted span.
            corrected_start = first_start
            corrected_end = last_end

            while corrected_start < corrected_end and text[corrected_start].isspace():
                corrected_start += 1
            while corrected_end > corrected_start and text[corrected_end - 1].isspace():
                corrected_end -= 1

            stats["whitespace_corrected_exact"] += (
                corrected_start == start and corrected_end == end
            )

    print("Total entities:", stats["total"])
    print("Start mismatches:", stats["start_mismatch"])
    print("End mismatches:", stats["end_mismatch"])
    print(
        "Gold boundaries recovered by whitespace trimming:",
        stats["whitespace_corrected_exact"]
    )

boundary_correction_audit(train_texts, train_gold)

In [ ]:
import numpy as np
import torch

@torch.inference_mode()
def predict_doc_with_confidence(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return []

    logits_sum = np.zeros(
        (len(ids), len(BIO_LABELS)), dtype=np.float64
    )
    counts = np.zeros(len(ids), dtype=np.int32)

    was_training = ner_model.training
    ner_model.eval()

    try:
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            x = torch.tensor([ids[start:end]], device=device)
            mask = torch.ones_like(x)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda" and USE_AMP),
            ):
                output = ner_model(
                    input_ids=x,
                    attention_mask=mask
                )

            if isinstance(output, dict):
                logits = output["logits"]
            elif hasattr(output, "logits"):
                logits = output.logits
            elif isinstance(output, (tuple, list)):
                logits = output[0]
            else:
                logits = output

            logits_sum[start:end] += (
                logits[0].float().cpu().numpy()
            )
            counts[start:end] += 1

            if end == len(ids):
                break
    finally:
        ner_model.train(was_training)

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)

    # Stable softmax.
    shifted = mean_logits - mean_logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

    pred_ids = mean_logits.argmax(axis=-1)
    spans = decode_spans(text, offsets, pred_ids)

    # Score each span by the average probability of its predicted BIO tags.
    results = []
    for start, end, label in spans:
        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not token_indices:
            continue

        confidence = float(np.mean([
            probs[i, pred_ids[i]] for i in token_indices
        ]))

        results.append((start, end, label, confidence))

    return results


print("Confidence-aware predictor ready.")

In [ ]:
from pathlib import Path

CHAMPION_PATH = Path(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
)

print("Champion checkpoint exists:", CHAMPION_PATH.exists())

In [ ]:
ckpt = torch.load(
    CHAMPION_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint keys:", ckpt.keys())

In [ ]:
import torch

ckpt = torch.load(
    CHAMPION_PATH,
    map_location="cpu",
    weights_only=False
)

assert ckpt["bio_labels"] == BIO_LABELS, (
    "BIO label ordering mismatch; refusing to load."
)

ner_model.load_state_dict(ckpt["model_state_dict"], strict=True)
ner_model = ner_model.to(device)
ner_model.eval()

print("Champion checkpoint loaded.")
print("Epoch:", ckpt.get("epoch"))
print("Recorded validation macro-F1:", ckpt.get("val_exact_span_macro_f1"))
print("Training fold:", ckpt.get("train_fold"))
print("Validation fold:", ckpt.get("validation_fold"))

In [ ]:
confidence_predictions = {
    doc_id: predict_doc_with_confidence(text)
    for doc_id, text in val_texts.items()
}

print("Validation documents:", len(confidence_predictions))
print(
    "Raw predicted spans:",
    sum(len(spans) for spans in confidence_predictions.values())
)
print("Example:", next(iter(confidence_predictions.items())))

In [ ]:
def filter_confidence_predictions(predictions, threshold):
    filtered = {}

    for doc_id, spans in predictions.items():
        filtered[doc_id] = [
            (start, end, label)
            for start, end, label, confidence in spans
            if confidence >= threshold
        ]

    return filtered


thresholds = [0.0, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]

threshold_results = []

for threshold in thresholds:
    filtered = filter_confidence_predictions(
        confidence_predictions, threshold
    )

    trimmed = trim_predictions(val_texts, filtered)

    macro_f1, per_label = exact_span_scores(
        val_gold, trimmed
    )

    total_pred = sum(len(spans) for spans in trimmed.values())

    threshold_results.append({
        "threshold": threshold,
        "macro_f1": macro_f1,
        "predictions": total_pred,
    })

for row in threshold_results:
    print(
        f"Threshold={row['threshold']:.2f} | "
        f"Macro-F1={row['macro_f1']:.4f} | "
        f"Predictions={row['predictions']}"
    )

best_threshold_row = max(
    threshold_results,
    key=lambda row: row["macro_f1"]
)

print("\nBest threshold:", best_threshold_row["threshold"])
print("Best macro-F1:", best_threshold_row["macro_f1"])
print("Existing benchmark: 0.0789")

In [ ]:
import itertools

threshold_grid = [0.35, 0.45, 0.50, 0.55, 0.60, 0.65]

# Start with the single-threshold winner as the baseline.
best_global_threshold = 0.55

# Coordinate search: optimize one label at a time while keeping
# all other labels at the current threshold.
label_thresholds = {
    label: best_global_threshold
    for label in ENTITY_LABELS
}

def filter_by_label_thresholds(predictions, thresholds):
    filtered = {}

    for doc_id, spans in predictions.items():
        filtered[doc_id] = [
            (start, end, label)
            for start, end, label, confidence in spans
            if confidence >= thresholds.get(label, best_global_threshold)
        ]

    return filtered

def score_thresholds(thresholds):
    filtered = filter_by_label_thresholds(
        confidence_predictions, thresholds
    )
    trimmed = trim_predictions(val_texts, filtered)
    macro_f1, per_label = exact_span_scores(val_gold, trimmed)
    return macro_f1, per_label, sum(len(v) for v in trimmed.values())

current_score, _, _ = score_thresholds(label_thresholds)
print("Starting score:", current_score)
print("Starting thresholds:", label_thresholds)

for label in ENTITY_LABELS:
    best_label_score = current_score
    best_label_threshold = label_thresholds[label]

    for threshold in threshold_grid:
        candidate = dict(label_thresholds)
        candidate[label] = threshold

        score, _, _ = score_thresholds(candidate)

        if score > best_label_score:
            best_label_score = score
            best_label_threshold = threshold

    label_thresholds[label] = best_label_threshold
    current_score = best_label_score

    print(
        f"{label:12s} threshold={best_label_threshold:.2f} "
        f"macro-F1={current_score:.4f}"
    )

final_score, final_per_label, final_count = score_thresholds(label_thresholds)

print("\nFinal coordinate-search score:", round(final_score, 4))
print("Predicted spans:", final_count)
print("Thresholds:", label_thresholds)

In [ ]:
import torch
from pathlib import Path

CHAMPION_PATH = Path(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
)

ckpt = torch.load(
    CHAMPION_PATH,
    map_location="cpu",
    weights_only=False
)

assert ckpt["bio_labels"] == BIO_LABELS
ner_model.load_state_dict(ckpt["model_state_dict"], strict=True)
ner_model = ner_model.to(device).eval()

LABEL_THRESHOLDS = {
    "APP": 0.55,
    "RESP": 0.55,
    "A.COUNSEL": 0.55,
    "R.COUNSEL": 0.55,
    "JUDGE": 0.35,
    "WIT": 0.55,
    "AUTH": 0.35,
    "COURT": 0.65,
    "STAT": 0.45,
    "PREC": 0.55,
    "DATE": 0.65,
    "CASENO": 0.55,
}

print("Champion checkpoint restored.")
print("Per-label thresholds frozen.")

In [ ]:
import json
from pathlib import Path

# Change this assignment only if your test text dictionary has another name.
test_docs = test_text

test_confidence_predictions = {
    doc_id: predict_doc_with_confidence(text)
    for doc_id, text in test_docs.items()
}

test_filtered = {
    doc_id: [
        (start, end, label)
        for start, end, label, confidence in spans
        if confidence >= LABEL_THRESHOLDS[label]
    ]
    for doc_id, spans in test_confidence_predictions.items()
}

test_trimmed = trim_predictions(test_docs, test_filtered)

# Required format: doc_id -> [[start, end, label], ...]
prediction = {
    str(doc_id): [
        [int(start), int(end), str(label)]
        for start, end, label in sorted(spans)
    ]
    for doc_id, spans in test_trimmed.items()
}

OUTPUT_PATH = Path("/kaggle/working/prediction.json")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False)

print("Saved:", OUTPUT_PATH)
print("Documents:", len(prediction))
print("Total predicted entities:", sum(len(v) for v in prediction.values()))
print("Example:", next(iter(prediction.items())))

In [ ]:
import json

with open("/kaggle/working/prediction.json", encoding="utf-8") as f:
    check = json.load(f)

assert len(check) == len(test_text)
assert set(check.keys()) == {str(k) for k in test_text.keys()}

valid_labels = set(ENTITY_LABELS)

for doc_id, spans in check.items():
    text = test_text[doc_id]

    for span in spans:
        assert len(span) == 3, (doc_id, span)

        start, end, label = span

        assert isinstance(start, int) and isinstance(end, int)
        assert 0 <= start < end <= len(text), (doc_id, span)
        assert label in valid_labels, (doc_id, label)

print("Output validation passed.")
print("Documents:", len(check))
print("Total predicted entities:", sum(len(v) for v in check.values()))
print("File: /kaggle/working/prediction.json")

In [ ]:
# EXPERIMENT 1 — Diagnose validation errors by entity label

import pandas as pd

# Use the thresholds selected by your previous coordinate search.
diagnostic_thresholds = dict(label_thresholds)

filtered_predictions = filter_by_label_thresholds(
    confidence_predictions,
    diagnostic_thresholds
)

trimmed_predictions = trim_predictions(
    val_texts,
    filtered_predictions
)

diagnostic_macro_f1, diagnostic_counts = exact_span_scores(
    val_gold,
    trimmed_predictions
)

rows = []

for label in ENTITY_LABELS:
    c = diagnostic_counts[label]

    gold_n = c["gold"]
    pred_n = c["pred"]
    tp = c["tp"]

    rows.append({
        "label": label,
        "gold": gold_n,
        "predicted": pred_n,
        "correct": tp,
        "precision": round(tp / pred_n, 4) if pred_n else 0.0,
        "recall": round(tp / gold_n, 4) if gold_n else 0.0,
        "F1": round(c["f1"], 4),
        "missed": gold_n - tp,
        "false_positives": pred_n - tp,
    })

report = pd.DataFrame(rows).sort_values("F1")

print(f"Exact-span validation macro-F1: {diagnostic_macro_f1:.4f}")
print("Total gold entities:", sum(r["gold"] for r in rows))
print("Total predicted entities:", sum(r["predicted"] for r in rows))
display(report.reset_index(drop=True))

In [ ]:
# EXPERIMENT 2A — Audit token/entity boundary alignment
# Uses existing tokenizer, BIO_LABELS, label2id, and fold data.

from collections import Counter

def audit_alignment(texts, gold_by_doc, fold_name):
    stats = Counter()
    examples = []

    for doc_id, text in texts.items():
        entities = gold_by_doc[doc_id]
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        for ent in entities:
            start, end, label = int(ent[0]), int(ent[1]), str(ent[2])

            overlapping = [
                (i, s, e, text[s:e])
                for i, (s, e) in enumerate(offsets)
                if e > s and s < end and e > start
            ]

            stats["entities"] += 1

            if not overlapping:
                stats["no_overlapping_token"] += 1
                if len(examples) < 8:
                    examples.append((doc_id, start, end, label, text[start:end], []))
                continue

            first_i, first_s, first_e, _ = overlapping[0]
            last_i, last_s, last_e, _ = overlapping[-1]

            exact_start = first_s == start
            exact_end = last_e == end

            stats["exact_start"] += int(exact_start)
            stats["exact_end"] += int(exact_end)
            stats["exact_both"] += int(exact_start and exact_end)
            stats["left_overhang"] += int(first_s < start)
            stats["right_overhang"] += int(last_e > end)

            if not (exact_start and exact_end) and len(examples) < 8:
                examples.append((
                    doc_id, start, end, label, text[start:end],
                    [(i, s, e, piece) for i, s, e, piece in overlapping]
                ))

    n = stats["entities"]

    print(f"\n=== {fold_name} ===")
    print("Gold entities:", n)
    print("No overlapping token:", stats["no_overlapping_token"])
    print("Exact start:", f"{stats['exact_start']}/{n} ({stats['exact_start']/max(n,1):.1%})")
    print("Exact end:", f"{stats['exact_end']}/{n} ({stats['exact_end']/max(n,1):.1%})")
    print("Exact both boundaries:", f"{stats['exact_both']}/{n} ({stats['exact_both']/max(n,1):.1%})")
    print("Left overhang:", stats["left_overhang"])
    print("Right overhang:", stats["right_overhang"])

    print("\nBoundary mismatch examples:")
    for ex in examples:
        print(ex)

audit_alignment(train_text_1, train_gold_1, "Fold 1")
audit_alignment(train_text_2, train_gold_2, "Fold 2")

In [ ]:
# EXPERIMENT 2B — Inspect BIO labels generated by the current window builder

from collections import Counter

def inspect_training_labels(texts, gold_by_doc, max_docs=5):
    token_counts = Counter()
    entity_token_counts = Counter()
    problems = []

    for doc_id in list(texts.keys())[:max_docs]:
        text = texts[doc_id]
        entities = gold_by_doc[doc_id]

        windows = make_unique_loss_windows(text, entities)

        for window in windows:
            ids = window["input_ids"]
            labels = window["labels"]

            for token_id, label_id in zip(ids, labels):
                if int(label_id) == IGNORE_INDEX:
                    continue

                tag = id2label[int(label_id)]
                token_counts[tag] += 1

                if tag != "O":
                    entity_token_counts[tag] += 1

        # Report entities for which no token boundary overlaps the gold span.
        offsets = tokenizer.encode(
            text, add_special_tokens=False
        ).offsets

        for ent in entities:
            start, end, label = int(ent[0]), int(ent[1]), str(ent[2])

            overlap_count = sum(
                1 for s, e in offsets
                if e > s and s < end and e > start
            )

            if overlap_count == 0:
                problems.append((doc_id, start, end, label, text[start:end]))

    print("Token labels, including O:")
    for tag, count in token_counts.most_common():
        print(f"{tag:15s} {count:,}")

    print("\nEntity-token counts:")
    for tag, count in sorted(entity_token_counts.items()):
        print(f"{tag:15s} {count:,}")

    print("\nEntities with no overlapping tokenizer token:", len(problems))
    for problem in problems[:10]:
        print(problem)

inspect_training_labels(train_text_1, train_gold_1)

In [ ]:
# Save current diagnostic baseline

from pathlib import Path
import json

report_path = Path("/kaggle/working/ner_diagnostic_baseline.json")

baseline = {
    "validation_macro_f1": float(diagnostic_macro_f1),
    "thresholds": {
        str(k): float(v)
        for k, v in diagnostic_thresholds.items()
    },
    "per_label": rows,
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(baseline, f, indent=2)

print("Saved baseline:", report_path)

In [ ]:
# EXPERIMENT 3A — Are the missing classes predicted at all?

import pandas as pd
import numpy as np
from collections import Counter

thresholds = (
    LABEL_THRESHOLDS
    if "LABEL_THRESHOLDS" in globals()
    else label_thresholds
)

raw_counts = Counter()
surviving_counts = Counter()
confidence_by_label = {label: [] for label in ENTITY_LABELS}

for doc_id, spans in confidence_predictions.items():
    for start, end, label, confidence in spans:
        raw_counts[label] += 1
        confidence_by_label[label].append(float(confidence))

        if confidence >= thresholds.get(label, 0.55):
            surviving_counts[label] += 1

gold_counts = Counter()
for entities in val_gold.values():
    for entity in entities:
        gold_counts[str(entity[2])] += 1

rows = []

for label in ENTITY_LABELS:
    confs = confidence_by_label[label]

    rows.append({
        "label": label,
        "gold": gold_counts[label],
        "raw_predictions": raw_counts[label],
        "after_threshold": surviving_counts[label],
        "threshold": thresholds.get(label, 0.55),
        "mean_confidence": round(float(np.mean(confs)), 3) if confs else 0.0,
        "median_confidence": round(float(np.median(confs)), 3) if confs else 0.0,
        "p90_confidence": round(float(np.percentile(confs, 90)), 3) if confs else 0.0,
        "max_confidence": round(max(confs), 3) if confs else 0.0,
    })

raw_report = pd.DataFrame(rows)
display(raw_report.sort_values("raw_predictions").reset_index(drop=True))

In [ ]:
# EXPERIMENT 4A — Token-level BIO prediction distribution
from collections import Counter
import torch
import pandas as pd

ner_model.eval()

predicted_bio_counts = Counter()
gold_bio_counts = Counter()

with torch.no_grad():
    for doc_id, text in val_texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids = enc.ids
        offsets = enc.offsets

        if not ids:
            continue

        logit_sum = torch.zeros(
            (len(ids), len(BIO_LABELS)), dtype=torch.float32
        )
        token_count = torch.zeros(len(ids), dtype=torch.float32)

        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = ner_model(input_ids, attention_mask)

            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            chunk_len = end - start
            logit_sum[start:end] += logits[:chunk_len]
            token_count[start:end] += 1

            if end == len(ids):
                break

        averaged_logits = (
            logit_sum / token_count.clamp_min(1).unsqueeze(-1)
        )
        predicted_ids = averaged_logits.argmax(dim=-1).tolist()

        for idx in predicted_ids:
            predicted_bio_counts[BIO_LABELS[idx]] += 1

        # Convert gold spans to token-level tags using overlapping tokens.
        token_tags = ["O"] * len(ids)

        for entity in val_gold[doc_id]:
            es, ee, label = int(entity[0]), int(entity[1]), str(entity[2])

            overlapping = [
                i for i, (s, e) in enumerate(offsets)
                if e > s and s < ee and e > es
            ]

            for j, i in enumerate(overlapping):
                token_tags[i] = ("B-" if j == 0 else "I-") + label

        for tag in token_tags:
            gold_bio_counts[tag] += 1

rows = [
    {
        "BIO tag": tag,
        "gold_tokens": gold_bio_counts[tag],
        "predicted_tokens": predicted_bio_counts[tag],
    }
    for tag in BIO_LABELS
]

display(pd.DataFrame(rows))

In [ ]:
# EXPERIMENT 5A — Inspect label assignment for one training document

doc_id = next(iter(train_text_1))
text = train_text_1[doc_id]
entities = train_gold_1[doc_id]

enc = tokenizer.encode(text, add_special_tokens=False)
offsets = enc.offsets

print("Document:", doc_id)
print("Text length:", len(text))
print("Gold entity count:", len(entities))

# Show examples where entity starts are inside token offsets.
shown = 0

for entity in entities:
    start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

    overlapping = [
        (i, s, e, text[s:e])
        for i, (s, e) in enumerate(offsets)
        if e > s and s < end and e > start
    ]

    if overlapping and (
        overlapping[0][1] != start or overlapping[-1][2] != end
    ):
        print("\nGold:", (start, end, label, text[start:end]))
        print("Overlapping tokens:", overlapping)

        shown += 1
        if shown >= 10:
            break

In [ ]:
# EXPERIMENT 6A — Count invalid BIO transitions

invalid = Counter()
total_i = Counter()

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids

    if not ids:
        continue

    # Reuse the existing confidence predictions' decoded spans only for
    # reference; obtain token predictions with the same sliding-window logic.
    logits_sum = torch.zeros(
        (len(ids), len(BIO_LABELS)), dtype=torch.float32
    )
    counts = torch.zeros(len(ids), dtype=torch.float32)

    ner_model.eval()
    with torch.no_grad():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = ner_model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            logits_sum[start:end] += logits[:end-start]
            counts[start:end] += 1

            if end == len(ids):
                break

    pred_ids = (logits_sum / counts.clamp_min(1).unsqueeze(-1)).argmax(-1)
    tags = [BIO_LABELS[i] for i in pred_ids.tolist()]

    previous = "O"

    for tag in tags:
        if tag.startswith("I-"):
            label = tag[2:]
            total_i[label] += 1

            if previous not in (f"B-{label}", f"I-{label}"):
                invalid[label] += 1

        previous = tag

print("Invalid I-tag transitions:")
for label in ENTITY_LABELS:
    print(
        f"{label:12s} invalid={invalid[label]:6d} "
        f"total_I={total_i[label]:6d} "
        f"rate={invalid[label] / max(total_i[label], 1):.1%}"
    )

In [ ]:
def predict_doc_constrained_with_confidence(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return []

    logit_sum = torch.zeros(
        (len(ids), len(BIO_LABELS)), dtype=torch.float32
    )
    token_count = torch.zeros(len(ids), dtype=torch.float32)

    ner_model.eval()

    with torch.no_grad():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = ner_model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            chunk_len = end - start
            logit_sum[start:end] += logits[:chunk_len]
            token_count[start:end] += 1

            if end == len(ids):
                break

    averaged_logits = (
        logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    )
    probabilities = torch.softmax(averaged_logits, dim=-1)

    pred_ids = bio_viterbi_decode(averaged_logits)
    spans = decode_spans(text, offsets, pred_ids)

    # Convert predicted IDs to a tensor so indexed selection works.
    pred_ids_tensor = torch.tensor(pred_ids, dtype=torch.long)

    results = []

    for start, end, label in spans:
        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < end and e > start
        ]

        if token_indices:
            selected_probs = probabilities[
                token_indices,
                pred_ids_tensor[token_indices]
            ]
            confidence = float(selected_probs.mean().item())
        else:
            confidence = 0.0

        results.append((start, end, label, confidence))

    return results


print("Corrected constrained predictor ready.")

In [ ]:
# EXPERIMENT 7B — Validation comparison

constrained_predictions = {
    doc_id: predict_doc_constrained_with_confidence(text)
    for doc_id, text in val_texts.items()
}

# Existing unconstrained predictions, using the same thresholds.
unconstrained_filtered = filter_by_label_thresholds(
    confidence_predictions, thresholds
)
unconstrained_trimmed = trim_predictions(
    val_texts, unconstrained_filtered
)

# New constrained predictions.
constrained_filtered = {
    doc_id: [
        (start, end, label)
        for start, end, label, confidence in spans
        if confidence >= thresholds.get(label, 0.55)
    ]
    for doc_id, spans in constrained_predictions.items()
}

constrained_trimmed = trim_predictions(
    val_texts, constrained_filtered
)

baseline_f1, baseline_per_label = exact_span_scores(
    val_gold, unconstrained_trimmed
)
constrained_f1, constrained_per_label = exact_span_scores(
    val_gold, constrained_trimmed
)

print(f"Unconstrained macro-F1: {baseline_f1:.4f}")
print(f"Constrained macro-F1:   {constrained_f1:.4f}")
print(f"Difference:             {constrained_f1 - baseline_f1:+.4f}")
print(
    "Constrained prediction count:",
    sum(len(v) for v in constrained_trimmed.values())
)

comparison_rows = []
for label in ENTITY_LABELS:
    comparison_rows.append({
        "label": label,
        "baseline_F1": round(baseline_per_label[label]["f1"], 4),
        "constrained_F1": round(constrained_per_label[label]["f1"], 4),
        "change": round(
            constrained_per_label[label]["f1"]
            - baseline_per_label[label]["f1"], 4
        ),
    })

display(
    pd.DataFrame(comparison_rows)
    .sort_values("change")
    .reset_index(drop=True)
)

In [ ]:
# EXPERIMENT 7C — Compare overall scores fairly

baseline_f1, baseline_per_label = exact_span_scores(
    val_gold, unconstrained_trimmed
)

constrained_f1, constrained_per_label = exact_span_scores(
    val_gold, constrained_trimmed
)

print(f"Unconstrained macro-F1: {baseline_f1:.4f}")
print(f"Constrained macro-F1:   {constrained_f1:.4f}")
print(f"Absolute improvement:  {constrained_f1 - baseline_f1:+.4f}")

print(
    "Unconstrained entities:",
    sum(len(v) for v in unconstrained_trimmed.values())
)
print(
    "Constrained entities:",
    sum(len(v) for v in constrained_trimmed.values())
)

In [ ]:
# EXPERIMENT 7D — Tune per-label confidence thresholds
# Use this after Cell 11.

threshold_grid = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]

constrained_thresholds = dict(thresholds)

def score_constrained_thresholds(threshold_map):
    filtered = {
        doc_id: [
            (start, end, label)
            for start, end, label, confidence in spans
            if confidence >= threshold_map.get(label, 0.55)
        ]
        for doc_id, spans in constrained_predictions.items()
    }

    trimmed = trim_predictions(val_texts, filtered)
    score, per_label = exact_span_scores(val_gold, trimmed)

    return score, per_label, sum(len(v) for v in trimmed.values())

best_score, _, _ = score_constrained_thresholds(constrained_thresholds)

for label in ENTITY_LABELS:
    chosen_threshold = constrained_thresholds[label]
    chosen_score = best_score

    for threshold in threshold_grid:
        candidate = dict(constrained_thresholds)
        candidate[label] = threshold

        score, _, _ = score_constrained_thresholds(candidate)

        if score > chosen_score:
            chosen_score = score
            chosen_threshold = threshold

    constrained_thresholds[label] = chosen_threshold
    best_score = chosen_score

final_score, final_per_label, final_count = (
    score_constrained_thresholds(constrained_thresholds)
)

print(f"Best constrained validation macro-F1: {final_score:.4f}")
print("Predicted entities:", final_count)
print("Thresholds:", constrained_thresholds)

display(pd.DataFrame([
    {
        "label": label,
        "threshold": constrained_thresholds[label],
        "F1": round(final_per_label[label]["f1"], 4),
    }
    for label in ENTITY_LABELS
]).sort_values("F1").reset_index(drop=True))

In [ ]:
# EXPERIMENT 8A — Compare gold entity distributions across folds

from collections import Counter
import pandas as pd

fold1_counts = Counter(str(e[2]) for doc in train_gold_1.values() for e in doc)
fold2_counts = Counter(str(e[2]) for doc in train_gold_2.values() for e in doc)

distribution_rows = []

for label in ENTITY_LABELS:
    n1 = fold1_counts[label]
    n2 = fold2_counts[label]

    distribution_rows.append({
        "label": label,
        "fold1_entities": n1,
        "fold2_entities": n2,
        "fold1_share": round(n1 / max(sum(fold1_counts.values()), 1), 4),
        "fold2_share": round(n2 / max(sum(fold2_counts.values()), 1), 4),
        "fold2_to_fold1_ratio": round(n2 / n1, 2) if n1 else None,
    })

display(
    pd.DataFrame(distribution_rows)
    .sort_values("fold2_to_fold1_ratio", na_position="first")
    .reset_index(drop=True)
)

In [ ]:
# EXPERIMENT 8B — Gold entity counts and BIO beginning-token counts

import pandas as pd
from collections import Counter

bio_begin_counts = Counter()

for window in train_windows_v2:
    for tag_id in window["labels"]:
        tag_id = int(tag_id)

        if tag_id == IGNORE_INDEX:
            continue

        tag = BIO_LABELS[tag_id]
        if tag.startswith("B-"):
            bio_begin_counts[tag[2:]] += 1

rows = []

for label in ENTITY_LABELS:
    rows.append({
        "label": label,
        "gold_entities_fold1": fold1_counts[label],
        "B_tokens_in_training_windows": bio_begin_counts[label],
    })

display(pd.DataFrame(rows).sort_values("B_tokens_in_training_windows"))

In [ ]:
# EXPERIMENT 9A — Calculate moderate class weights

import torch
import numpy as np
from collections import Counter

tag_counts = Counter()

for window in train_windows_v2:
    for tag_id in window["labels"]:
        tag_id = int(tag_id)
        if tag_id != IGNORE_INDEX:
            tag_counts[tag_id] += 1

counts = np.array(
    [max(tag_counts.get(i, 0), 1) for i in range(len(BIO_LABELS))],
    dtype=np.float64
)

# Square-root inverse-frequency weights, normalized to mean 1.
class_weights = 1.0 / np.sqrt(counts)
class_weights = class_weights / class_weights.mean()
class_weights = np.clip(class_weights, 0.5, 3.0)

# Keep O at a modest weight so the model does not overfit entity tokens.
class_weights[BIO_LABELS.index("O")] = 0.5

# Renormalize non-O weights to mean 1.
non_o = np.array([i for i, tag in enumerate(BIO_LABELS) if tag != "O"])
class_weights[non_o] /= class_weights[non_o].mean()

class_weights = torch.tensor(
    class_weights, dtype=torch.float32, device=device
)

for i, tag in enumerate(BIO_LABELS):
    print(f"{tag:15s} weight={class_weights[i].item():.3f} count={counts[i]:.0f}")

In [ ]:
# EXPERIMENT 9B — Inspect available training objects

print("Model class:", type(ner_model).__name__)
print("Device:", device)
print("Training windows:", len(train_windows_v2))
print("Validation documents:", len(val_texts))
print("BIO labels:", len(BIO_LABELS))

print("\nFirst training window keys:", train_windows_v2[0].keys())
print("First window length:", len(train_windows_v2[0]["input_ids"]))
print("First label sequence length:", len(train_windows_v2[0]["labels"]))

print("\nCurrent model forward signature:")
import inspect
print(inspect.signature(ner_model.forward))

In [ ]:
# EXPERIMENT 9C — Inspect training-related definitions

import inspect

for name in [
    "LegalNERModel",
    "make_unique_loss_windows",
    "make_contained_loss_windows",
    "exact_span_scores",
    "predict_doc_constrained_with_confidence",
]:
    obj = globals().get(name)

    if obj is not None:
        print(f"\n{'=' * 20} {name} {'=' * 20}")
        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Source unavailable; object:", obj)

In [ ]:
# EXPERIMENT C — Fresh model + moderate class weighting
# Does not overwrite the current champion checkpoint.

import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path
from collections import Counter
from transformers import GPT2Config, GPT2LMHeadModel
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. Reconstruct the Phase 1 language model from the supplied checkpoint.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

phase1_ckpt = torch.load(
    "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt",
    map_location="cpu",
    weights_only=False
)

cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

fresh_lm_C = GPT2LMHeadModel(gpt2_config)
fresh_lm_C.load_state_dict(
    phase1_ckpt["model_state_dict"],
    strict=True
)

# Reset the NER head and transformer to the Phase 1 initialization.
ner_model = LegalNERModel(
    pretrained_lm=fresh_lm_C,
    num_labels=len(BIO_LABELS)
).to(device)

# 2. Calculate capped square-root inverse-frequency weights.
IGNORE_INDEX = -100
tag_counts_C = Counter()

for window in train_windows_v2:
    for tag_id in window["labels"]:
        tag_id = int(tag_id)
        if tag_id != IGNORE_INDEX:
            tag_counts_C[tag_id] += 1

freq_C = torch.tensor(
    [max(tag_counts_C.get(i, 0), 1) for i in range(len(BIO_LABELS))],
    dtype=torch.float32
)

weights_C = 1.0 / torch.sqrt(freq_C)
weights_C = weights_C / weights_C.mean()
weights_C = weights_C.clamp(0.5, 3.0)

# Explicitly reduce the dominance of the O class.
weights_C[BIO_LABELS.index("O")] = 0.5

non_o_ids = [
    i for i, tag in enumerate(BIO_LABELS) if tag != "O"
]
weights_C[non_o_ids] /= weights_C[non_o_ids].mean()

weights_C = weights_C.to(device)

# 3. Training configuration.
MAX_EPOCHS_C = 5
BATCH_SIZE_C = 2
LEARNING_RATE_C = 1e-5
WEIGHT_DECAY_C = 0.01
GRAD_CLIP_C = 1.0

criterion_C = nn.CrossEntropyLoss(
    weight=weights_C,
    ignore_index=IGNORE_INDEX
)

optimizer_C = torch.optim.AdamW(
    ner_model.parameters(),
    lr=LEARNING_RATE_C,
    weight_decay=WEIGHT_DECAY_C
)

SAVE_DIR_C = Path("/kaggle/working/phase2_ner_experiment_C")
SAVE_DIR_C.mkdir(parents=True, exist_ok=True)

print("Fresh Phase 1 initialization loaded.")
print("Training windows:", len(train_windows_v2))
print("Validation documents:", len(val_texts))
print("Device:", device)
print("Learning rate:", LEARNING_RATE_C)
print("Epochs:", MAX_EPOCHS_C)
print("Checkpoint directory:", SAVE_DIR_C)

In [ ]:
# EXPERIMENT C — Data loader and training function

def collate_ner_windows_C(batch):
    ids_list = [
        torch.as_tensor(item["input_ids"], dtype=torch.long)
        for item in batch
    ]

    labels_list = [
        torch.as_tensor(item["labels"], dtype=torch.long)
        for item in batch
    ]

    input_ids = pad_sequence(
        ids_list,
        batch_first=True,
        padding_value=0
    )

    labels = pad_sequence(
        labels_list,
        batch_first=True,
        padding_value=IGNORE_INDEX
    )

    attention_mask = torch.zeros_like(input_ids)

    for i, ids in enumerate(ids_list):
        attention_mask[i, :len(ids)] = 1

    return input_ids, attention_mask, labels


train_loader_C = DataLoader(
    train_windows_v2,
    batch_size=BATCH_SIZE_C,
    shuffle=True,
    collate_fn=collate_ner_windows_C
)


def train_one_epoch_C():
    ner_model.train()
    total_loss = 0.0
    total_batches = 0

    for input_ids, attention_mask, labels in train_loader_C:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer_C.zero_grad(set_to_none=True)

        output = ner_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(output, dict):
            logits = output["logits"]
        elif hasattr(output, "logits"):
            logits = output.logits
        elif isinstance(output, (tuple, list)):
            logits = output[0]
        else:
            logits = output

        loss = criterion_C(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss encountered: {loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            ner_model.parameters(),
            GRAD_CLIP_C
        )

        optimizer_C.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)


print("Training batches:", len(train_loader_C))
print("Training function ready.")

In [ ]:
# EXPERIMENT C — Train and evaluate each epoch

history_C = []
best_f1_C = -1.0
best_epoch_C = None

# Use the previously tuned thresholds for a controlled comparison.
thresholds_C = dict(constrained_thresholds)

def evaluate_constrained_C():
    ner_model.eval()

    confidence_preds = {
        doc_id: predict_doc_constrained_with_confidence(text)
        for doc_id, text in val_texts.items()
    }

    filtered = {
        doc_id: [
            (start, end, label)
            for start, end, label, confidence in spans
            if confidence >= thresholds_C.get(label, 0.55)
        ]
        for doc_id, spans in confidence_preds.items()
    }

    trimmed = trim_predictions(val_texts, filtered)

    score, per_label = exact_span_scores(val_gold, trimmed)

    return score, per_label, trimmed


for epoch in range(1, MAX_EPOCHS_C + 1):
    train_loss = train_one_epoch_C()

    val_f1, val_per_label, val_preds = evaluate_constrained_C()

    pred_count = sum(len(v) for v in val_preds.values())

    print(f"\nEpoch {epoch}/{MAX_EPOCHS_C}")
    print(f"Training loss: {train_loss:.4f}")
    print(f"Constrained validation macro-F1: {val_f1:.4f}")
    print(f"Predicted spans: {pred_count}")

    for label in ENTITY_LABELS:
        c = val_per_label[label]
        print(
            f"{label:12s} "
            f"F1={c['f1']:.4f} "
            f"gold={c['gold']:4d} "
            f"pred={c['pred']:5d} "
            f"correct={c['tp']:4d}"
        )

    history_C.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "constrained_macro_f1": float(val_f1),
        "predicted_spans": int(pred_count),
    })

    checkpoint_C = {
        "epoch": epoch,
        "model_state_dict": ner_model.state_dict(),
        "optimizer_state_dict": optimizer_C.state_dict(),
        "train_loss": float(train_loss),
        "constrained_macro_f1": float(val_f1),
        "bio_labels": BIO_LABELS,
        "entity_labels": ENTITY_LABELS,
        "thresholds": thresholds_C,
        "phase1_checkpoint": str(
            "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
        ),
    }

    torch.save(
        checkpoint_C,
        SAVE_DIR_C / f"epoch_{epoch}.pt"
    )

    if val_f1 > best_f1_C:
        best_f1_C = val_f1
        best_epoch_C = epoch

        torch.save(
            checkpoint_C,
            SAVE_DIR_C / "best_model.pt"
        )

        print("Saved new best checkpoint.")

with open(SAVE_DIR_C / "history.json", "w") as f:
    json.dump(history_C, f, indent=2)

print("\nExperiment C complete.")
print("Best epoch:", best_epoch_C)
print("Best constrained validation macro-F1:", round(best_f1_C, 4))
print("Checkpoint:", SAVE_DIR_C / "best_model.pt")

In [ ]:
# CELL 1 — Clean failed Experiment C and free RAM/GPU memory
# Preserves original checkpoints, dataset files, and prediction.json.

import os
import gc
import shutil
import torch
from pathlib import Path

experiment_dir = Path("/kaggle/working/phase2_ner_experiment_C")

if experiment_dir.exists():
    shutil.rmtree(experiment_dir)
    print("Deleted temporary Experiment C directory.")
else:
    print("Experiment C directory already absent.")

# Remove large experiment-only objects if they exist.
for name in [
    "constrained_predictions",
    "champion_constrained_predictions",
    "history_C",
    "val_preds",
    "logit_sum",
    "averaged_logits",
    "probabilities",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# Check remaining working-directory storage.
usage = shutil.disk_usage("/kaggle/working")
print(f"Working storage used: {(usage.total - usage.free) / 1024**3:.2f} GB")
print(f"Working storage free: {usage.free / 1024**3:.2f} GB")

print("Cleanup complete.")

In [ ]:
# CELL 2 — Restore champion checkpoint

import torch

CHAMPION_PATH = (
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
)

assert os.path.exists(CHAMPION_PATH), (
    f"Champion checkpoint not found: {CHAMPION_PATH}"
)

champion_ckpt = torch.load(
    CHAMPION_PATH,
    map_location="cpu",
    weights_only=False
)

assert champion_ckpt["bio_labels"] == BIO_LABELS

ner_model.load_state_dict(
    champion_ckpt["model_state_dict"],
    strict=True
)

ner_model = ner_model.to(device).eval()

print("Champion restored.")
print("Checkpoint:", CHAMPION_PATH)
print("Epoch:", champion_ckpt.get("epoch"))
print("Recorded validation F1:", champion_ckpt.get("val_exact_span_macro_f1"))
print("Device:", device)

In [ ]:
# CELL 3 — Predict fold 2 using the restored champion

champion_constrained_predictions = {}

for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
    champion_constrained_predictions[doc_id] = (
        predict_doc_constrained_with_confidence(text)
    )

    if i % 5 == 0 or i == len(val_texts):
        print(f"Processed {i}/{len(val_texts)} validation documents")

print(
    "Raw constrained spans:",
    sum(len(v) for v in champion_constrained_predictions.values())
)

In [ ]:
# CELL 4 — Coordinate search for per-label thresholds

import pandas as pd

threshold_grid = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]

champion_thresholds = {
    label: 0.45 for label in ENTITY_LABELS
}

def score_champion_thresholds(threshold_map):
    filtered = {
        doc_id: [
            (start, end, label)
            for start, end, label, confidence in spans
            if confidence >= threshold_map.get(label, 0.45)
        ]
        for doc_id, spans in champion_constrained_predictions.items()
    }

    trimmed = trim_predictions(val_texts, filtered)
    score, per_label = exact_span_scores(val_gold, trimmed)

    return score, per_label, sum(len(v) for v in trimmed.values())


best_score, _, _ = score_champion_thresholds(champion_thresholds)

for label in ENTITY_LABELS:
    chosen_threshold = champion_thresholds[label]
    chosen_score = best_score

    for threshold in threshold_grid:
        candidate = dict(champion_thresholds)
        candidate[label] = threshold

        score, _, _ = score_champion_thresholds(candidate)

        if score > chosen_score:
            chosen_score = score
            chosen_threshold = threshold

    champion_thresholds[label] = chosen_threshold
    best_score = chosen_score

champion_best_score, champion_per_label, champion_count = (
    score_champion_thresholds(champion_thresholds)
)

print(f"Champion constrained macro-F1: {champion_best_score:.4f}")
print("Predicted spans:", champion_count)
print("Thresholds:", champion_thresholds)

display(pd.DataFrame([
    {
        "label": label,
        "threshold": champion_thresholds[label],
        "F1": round(champion_per_label[label]["f1"], 4),
        "gold": champion_per_label[label]["gold"],
        "predicted": champion_per_label[label]["pred"],
        "correct": champion_per_label[label]["tp"],
    }
    for label in ENTITY_LABELS
]).sort_values("F1").reset_index(drop=True))

In [ ]:
# CELL 5 — Save champion threshold configuration

import json

THRESHOLD_PATH = "/kaggle/working/champion_thresholds.json"

with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "validation_macro_f1": float(champion_best_score),
        "thresholds": champion_thresholds,
        "checkpoint": CHAMPION_PATH,
        "decoder": "BIO-constrained Viterbi",
    }, f, indent=2)

print("Saved:", THRESHOLD_PATH)
print("Best development macro-F1:", round(champion_best_score, 4))

In [ ]:
import os, json

assert os.path.exists(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
)
assert os.path.exists(
    "/kaggle/working/champion_thresholds.json"
)

with open("/kaggle/working/champion_thresholds.json") as f:
    champion_thresholds = json.load(f)

print("Champion preserved.")
print("Thresholds:", champion_thresholds)

In [ ]:
import os
import gc
import torch
from transformers import GPT2Config, GPT2LMHeadModel

# Preserve the champion; do not load its weights into this experiment.
champion_path = "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
phase1_path = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
experiment_dir = "/kaggle/working/phase2_ner_experiment_D"

os.makedirs(experiment_dir, exist_ok=True)

assert os.path.exists(champion_path), "Champion checkpoint is missing."
assert os.path.exists(phase1_path), "Phase 1 checkpoint is missing."

# Release the previous model from GPU memory.
if "ner_model" in globals():
    del ner_model
gc.collect()
torch.cuda.empty_cache()

# Load original Phase 1 weights.
phase1_ckpt = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)

cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

phase1_fresh = GPT2LMHeadModel(gpt2_config)
phase1_fresh.load_state_dict(
    phase1_ckpt["model_state_dict"],
    strict=True
)

ner_model_D = LegalNERModel(
    pretrained_lm=phase1_fresh,
    num_labels=len(BIO_LABELS)
).to(device)

print("Experiment D initialized from Phase 1.")
print("Parameters:", sum(p.numel() for p in ner_model_D.parameters()))
print("Device:", device)
print("Champion remains untouched:", os.path.exists(champion_path))

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

assert len(train_windows_v2) > 0, "Training windows are missing."

# Confirm the windows contain token IDs and labels.
print("Training windows:", len(train_windows_v2))
print("Example keys:", train_windows_v2[0].keys())
print("Label ignore index:", IGNORE_INDEX)

optimizer_D = AdamW(
    ner_model_D.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

# Standard, unweighted cross-entropy.
loss_fn_D = torch.nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

print("Unweighted loss configured.")

In [ ]:
# Inspect available collator functions without modifying anything.
for name in [
    "collate_fn",
    "collate_batch",
    "collate_windows",
    "ner_collate_fn",
    "collate_fn_ner",
]:
    obj = globals().get(name)
    if callable(obj):
        print(f"Found {name}: {obj}")

print("\nTraining window shapes:")
print("input_ids:", len(train_windows_v2[0]["input_ids"]))
print("labels:", len(train_windows_v2[0]["labels"]))
print("Input/label lengths match:",
      len(train_windows_v2[0]["input_ids"]) ==
      len(train_windows_v2[0]["labels"]))

In [ ]:
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    train_windows_v2,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_windows,
)

batch = next(iter(debug_loader))

print("Batch type:", type(batch))
print("Batch length:", len(batch))

for i, item in enumerate(batch):
    print(
        f"Item {i}: type={type(item)}, "
        f"shape={getattr(item, 'shape', None)}, "
        f"dtype={getattr(item, 'dtype', None)}"
    )

In [ ]:
ner_model_D.eval()

batch = next(iter(debug_loader))
input_ids, labels, attention_mask = [
    x.to(device) for x in batch
]

with torch.no_grad():
    outputs = ner_model_D(input_ids, attention_mask)

print("Output type:", type(outputs))

if isinstance(outputs, (tuple, list)):
    print("Number of outputs:", len(outputs))
    for i, item in enumerate(outputs):
        if torch.is_tensor(item):
            print(f"outputs[{i}] shape: {tuple(item.shape)}")
        else:
            print(f"outputs[{i}] type: {type(item)}")
elif torch.is_tensor(outputs):
    print("Output shape:", tuple(outputs.shape))

print("Input shape:", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))
print("Expected logits shape: [2, 512, number_of_BIO_labels]")

In [ ]:
import os
import gc
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"
os.makedirs(EXP_D_DIR, exist_ok=True)

BATCH_SIZE_D = 2
EPOCHS_D = 5

train_loader_D = DataLoader(
    train_windows_v2,
    batch_size=BATCH_SIZE_D,
    shuffle=True,
    collate_fn=collate_windows,
)

optimizer_D = AdamW(
    ner_model_D.parameters(),
    lr=1e-5,
    weight_decay=0.01,
)

loss_fn_D = torch.nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

history_D = []

for epoch in range(1, EPOCHS_D + 1):
    ner_model_D.train()
    total_loss = 0.0
    total_batches = 0

    progress = tqdm(
        train_loader_D,
        desc=f"Experiment D — Epoch {epoch}/{EPOCHS_D}"
    )

    for batch in progress:
        input_ids, labels, attention_mask = [
            x.to(device) for x in batch
        ]

        optimizer_D.zero_grad(set_to_none=True)

        # The model returns logits directly: [B, L, C].
        logits = ner_model_D(input_ids, attention_mask)

        assert logits.ndim == 3, f"Unexpected logits shape: {logits.shape}"
        assert logits.shape[:2] == labels.shape, (
            f"Logits {logits.shape} and labels {labels.shape} do not align"
        )

        loss = loss_fn_D(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            ner_model_D.parameters(), 1.0
        )
        optimizer_D.step()

        total_loss += loss.item()
        total_batches += 1
        progress.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_loss / max(total_batches, 1)

    checkpoint_path = os.path.join(
        EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt"
    )

    torch.save(
        {
            "model_state_dict": ner_model_D.state_dict(),
            "epoch": epoch,
            "mean_train_loss": mean_loss,
            "phase1_checkpoint": phase1_path,
            "train_fold": 1,
            "validation_fold": 2,
            "loss": "unweighted_cross_entropy",
            "learning_rate": 1e-5,
            "bio_labels": BIO_LABELS,
            "entity_labels": ENTITY_LABELS,
        },
        checkpoint_path,
    )

    history_D.append({
        "epoch": epoch,
        "mean_train_loss": mean_loss,
        "checkpoint": checkpoint_path,
    })

    print(
        f"Epoch {epoch}: mean loss={mean_loss:.4f}\n"
        f"Saved: {checkpoint_path}"
    )

    gc.collect()
    torch.cuda.empty_cache()

print("\nExperiment D training finished.")
print(history_D)

In [ ]:
import os
import shutil
import torch

# Persistent checkpoint staging directory
PERSIST_DIR = "/kaggle/working/cs781_persistent_checkpoints"
os.makedirs(PERSIST_DIR, exist_ok=True)

files_to_save = {
    "champion_best_ner_model.pt":
        "/kaggle/working/phase2_ner_corrected/best_ner_model.pt",
    "experiment_D_initial_state.pt":
        "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt",
    "champion_thresholds.json":
        "/kaggle/working/champion_thresholds.json",
}

for dst_name, src in files_to_save.items():
    assert os.path.isfile(src), f"Missing source file: {src}"
    shutil.copy2(src, os.path.join(PERSIST_DIR, dst_name))
    print(f"Saved: {os.path.join(PERSIST_DIR, dst_name)}")

# Save the current Experiment D model initialization as well
torch.save(
    {
        "model_state_dict": ner_model_D.state_dict(),
        "bio_labels": BIO_LABELS,
        "entity_labels": ENTITY_LABELS,
        "source": "Phase 1 initialization for Experiment D",
    },
    os.path.join(PERSIST_DIR, "experiment_D_model_init.pt"),
)

print("\nExperiment D initialization saved.")
print("Files:", os.listdir(PERSIST_DIR))
print("NOTE: /kaggle/working is session-persistent only within this runtime.")
print("To preserve files across sessions, create/save a Kaggle Dataset")
print("from this folder using Kaggle's dataset versioning workflow.")

In [ ]:
import os
import json
import subprocess
from pathlib import Path

DATA_DIR = Path("/kaggle/working/cs781_persistent_checkpoints")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Ensure the staged files are present before publishing.
required_files = [
    "champion_best_ner_model.pt",
    "experiment_D_initial_state.pt",
    "champion_thresholds.json",
    "experiment_D_model_init.pt",
]

missing = [f for f in required_files if not (DATA_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing staged files: {missing}. Run the checkpoint-staging cell first."
    )

metadata_path = DATA_DIR / "dataset-metadata.json"
metadata = {
    "title": "CS781 Persistent NER Checkpoints",
    "id": "oipranesh/cs781-persistent-ner-checkpoints",
    "licenses": [{"name": "CC0-1.0"}],
    "description": (
        "CS781 legal NER project checkpoints and validation thresholds."
    ),
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# Check authentication and CLI availability.
check = subprocess.run(
    ["kaggle", "datasets", "create", "-h"],
    capture_output=True,
    text=True
)
if check.returncode != 0:
    raise RuntimeError(check.stderr or check.stdout)

# Publish privately: do NOT add --public.
result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(DATA_DIR)],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Dataset creation failed. If the dataset ID already exists, "
        "use the version-update command instead."
    )

print("Dataset creation command succeeded.")
print("Expected URL: https://www.kaggle.com/datasets/oipranesh/cs781-persistent-ner-checkpoints")

In [ ]:

import os
import gc
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"
os.makedirs(EXP_D_DIR, exist_ok=True)

BATCH_SIZE_D = 2
EPOCHS_D = 5

train_loader_D = DataLoader(
    train_windows_v2,
    batch_size=BATCH_SIZE_D,
    shuffle=True,
    collate_fn=collate_windows,
)

optimizer_D = AdamW(
    ner_model_D.parameters(),
    lr=1e-5,
    weight_decay=0.01,
)

loss_fn_D = torch.nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

history_D = []

for epoch in range(1, EPOCHS_D + 1):
    ner_model_D.train()
    total_loss = 0.0
    total_batches = 0

    progress = tqdm(
        train_loader_D,
        desc=f"Experiment D — Epoch {epoch}/{EPOCHS_D}"
    )

    for input_ids, labels, attention_mask in progress:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        attention_mask = attention_mask.to(device)

        optimizer_D.zero_grad(set_to_none=True)

        # Model returns logits directly: [B, L, C].
        logits = ner_model_D(input_ids, attention_mask)

        assert logits.shape[:2] == labels.shape, (
            f"Shape mismatch: logits={logits.shape}, labels={labels.shape}"
        )

        loss = loss_fn_D(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            ner_model_D.parameters(), 1.0
        )
        optimizer_D.step()

        total_loss += loss.item()
        total_batches += 1
        progress.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_loss / max(total_batches, 1)

    checkpoint_path = os.path.join(
        EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt"
    )

    torch.save(
        {
            "model_state_dict": ner_model_D.state_dict(),
            "epoch": epoch,
            "mean_train_loss": mean_loss,
            "phase1_checkpoint": phase1_path,
            "train_fold": 1,
            "validation_fold": 2,
            "loss": "unweighted_cross_entropy",
            "learning_rate": 1e-5,
            "bio_labels": BIO_LABELS,
            "entity_labels": ENTITY_LABELS,
        },
        checkpoint_path,
    )

    history_D.append({
        "epoch": epoch,
        "mean_train_loss": mean_loss,
        "checkpoint": checkpoint_path,
    })

    print(f"Epoch {epoch}: mean loss={mean_loss:.4f}")
    print(f"Saved: {checkpoint_path}")

    gc.collect()
    torch.cuda.empty_cache()

print("\nExperiment D training finished.")
print(history_D)


In [ ]:

import os
import torch

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"

for epoch in range(1, 6):
    path = os.path.join(
        EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt"
    )
    if os.path.exists(path):
        ckpt = torch.load(
            path, map_location="cpu", weights_only=False
        )
        print(
            f"Epoch {epoch}: "
            f"loss={ckpt['mean_train_loss']:.4f}, "
            f"size={os.path.getsize(path) / 1e6:.1f} MB"
        )
    else:
        print(f"Epoch {epoch}: checkpoint missing")


In [ ]:

import os
import torch

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"
CHAMPION_PATH = "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"

assert os.path.exists(CHAMPION_PATH), "Champion checkpoint missing."
assert "val_texts" in globals(), "Validation texts are missing."
assert "val_gold" in globals(), "Validation gold annotations are missing."
assert "predict_doc_constrained_with_confidence" in globals(), \
    "Constrained prediction function is missing."
assert "exact_span_scores" in globals(), "Scoring function is missing."

for epoch in range(1, 6):
    path = os.path.join(EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt")
    assert os.path.exists(path), f"Missing checkpoint: {path}"

print("All five Experiment D checkpoints found.")
print("Champion checkpoint found.")
print("Validation documents:", len(val_texts))
print("Ready for controlled evaluation.")

In [ ]:

import os
import gc
import pandas as pd
import torch

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"

# Save Experiment D's current in-memory weights for restoration.
original_weights_D = {
    k: v.detach().cpu().clone()
    for k, v in ner_model_D.state_dict().items()
}

# The prediction function references the global variable ner_model.
ner_model = ner_model_D

def tune_thresholds_D(predictions, gold):
    grid = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]
    thresholds = {label: 0.45 for label in ENTITY_LABELS}

    for _ in range(2):
        changed = False

        for label in ENTITY_LABELS:
            best_score = -1.0
            best_threshold = thresholds[label]

            for threshold in grid:
                candidate = dict(thresholds)
                candidate[label] = threshold

                filtered = {
                    doc_id: [
                        (s, e, lab)
                        for s, e, lab, conf in spans
                        if conf >= candidate[lab]
                    ]
                    for doc_id, spans in predictions.items()
                }

                score, _ = exact_span_scores(gold, filtered)

                if score > best_score:
                    best_score = score
                    best_threshold = threshold

            if thresholds[label] != best_threshold:
                changed = True
            thresholds[label] = best_threshold

        if not changed:
            break

    filtered = {
        doc_id: [
            (s, e, label)
            for s, e, label, conf in spans
            if conf >= thresholds[label]
        ]
        for doc_id, spans in predictions.items()
    }

    score, per_label = exact_span_scores(gold, filtered)
    count = sum(len(spans) for spans in filtered.values())

    return score, thresholds, per_label, count


results_D = []

for epoch in range(1, 6):
    path = os.path.join(EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    ner_model_D.load_state_dict(ckpt["model_state_dict"])
    ner_model_D.to(device)
    ner_model_D.eval()

    # Keep the predictor's global model reference synchronized.
    ner_model = ner_model_D

    predictions = {
        doc_id: predict_doc_constrained_with_confidence(text)
        for doc_id, text in val_texts.items()
    }

    score, thresholds, per_label, count = tune_thresholds_D(
        predictions, val_gold
    )

    results_D.append({
        "epoch": epoch,
        "train_loss": ckpt["mean_train_loss"],
        "macro_f1": score,
        "predicted_spans": count,
        "thresholds": thresholds,
        "per_label": per_label,
        "checkpoint": path,
    })

    print(
        f"Epoch {epoch}: loss={ckpt['mean_train_loss']:.3e}, "
        f"macro-F1={score:.4f}, predicted spans={count}"
    )

    del predictions
    gc.collect()
    torch.cuda.empty_cache()

# Restore Experiment D's original in-memory weights.
ner_model_D.load_state_dict(original_weights_D)
ner_model = ner_model_D
ner_model.to(device)
ner_model.eval()

summary_D = pd.DataFrame([
    {
        "epoch": r["epoch"],
        "train_loss": r["train_loss"],
        "validation_macro_f1": r["macro_f1"],
        "predicted_spans": r["predicted_spans"],
    }
    for r in results_D
]).sort_values("validation_macro_f1", ascending=False)

display(summary_D)

best_D = max(results_D, key=lambda r: r["macro_f1"])
print("\nBest Experiment D epoch:", best_D["epoch"])
print("Best Experiment D macro-F1:", round(best_D["macro_f1"], 4))
print("Champion reference: 0.1663")


In [ ]:

import torch

batch = collate_windows([train_windows_v2[0], train_windows_v2[1]])

input_ids, attention_mask, labels = batch

print("Input IDs:", tuple(input_ids.shape))
print("Attention mask:", tuple(attention_mask.shape))
print("Labels:", tuple(labels.shape))

print("\nGold label counts in batch:")
valid = labels[labels != IGNORE_INDEX]
counts = torch.bincount(valid, minlength=len(BIO_LABELS))
for i, count in enumerate(counts.tolist()):
    if count:
        print(f"{BIO_LABELS[i]:12s}: {count}")

assert input_ids.shape == attention_mask.shape == labels.shape
assert torch.all((attention_mask == 0) | (attention_mask == 1))
assert torch.any(labels == 0), "Expected O labels in the batch."
assert torch.any(labels == IGNORE_INDEX), "Expected ignored overlap positions."

print("\nBatch order verified.")


In [ ]:

import torch
from transformers import GPT2Config, GPT2LMHeadModel

# Reload the original Phase 1 checkpoint.
phase1_ckpt = torch.load(
    phase1_path, map_location="cpu", weights_only=False
)
cfg = phase1_ckpt["model_config"]

config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

fresh_lm = GPT2LMHeadModel(config)
fresh_lm.load_state_dict(phase1_ckpt["model_state_dict"], strict=True)

ner_model_D = LegalNERModel(
    pretrained_lm=fresh_lm,
    num_labels=len(BIO_LABELS)
).to(device)

ner_model = ner_model_D

optimizer_D = torch.optim.AdamW(
    ner_model_D.parameters(),
    lr=1e-5,
    weight_decay=0.01,
)

loss_fn_D = torch.nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

print("Experiment D reinitialized from Phase 1.")
print("Champion preserved:", os.path.exists(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
))


In [ ]:

import os
import gc
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"
os.makedirs(EXP_D_DIR, exist_ok=True)

train_loader_D = DataLoader(
    train_windows_v2,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_windows,
)

history_D = []

for epoch in range(1, 6):
    ner_model_D.train()
    total_loss = 0.0
    total_batches = 0

    for input_ids, attention_mask, labels in tqdm(
        train_loader_D, desc=f"Corrected Experiment D — Epoch {epoch}/5"
    ):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer_D.zero_grad(set_to_none=True)

        logits = ner_model_D(input_ids, attention_mask)
        loss = loss_fn_D(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(ner_model_D.parameters(), 1.0)
        optimizer_D.step()

        total_loss += loss.item()
        total_batches += 1

    mean_loss = total_loss / total_batches

    path = os.path.join(EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt")
    torch.save({
        "model_state_dict": ner_model_D.state_dict(),
        "epoch": epoch,
        "mean_train_loss": mean_loss,
        "phase1_checkpoint": phase1_path,
        "train_fold": 1,
        "validation_fold": 2,
        "loss": "unweighted_cross_entropy",
        "learning_rate": 1e-5,
        "bio_labels": BIO_LABELS,
        "entity_labels": ENTITY_LABELS,
    }, path)

    history_D.append({
        "epoch": epoch,
        "mean_train_loss": mean_loss,
        "checkpoint": path,
    })

    print(f"Epoch {epoch}: loss={mean_loss:.6f}")
    print(f"Saved: {path}")

    gc.collect()
    torch.cuda.empty_cache()

print("Corrected Experiment D training complete.")
print(history_D)


In [ ]:

import os
import gc
import pandas as pd
import torch
from tqdm.auto import tqdm

EXP_D_DIR = "/kaggle/working/phase2_ner_experiment_D"
GRID_D = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]

def tune_thresholds_D(predictions, gold):
    thresholds = {label: 0.45 for label in ENTITY_LABELS}

    for _ in range(2):
        changed = False

        for label in ENTITY_LABELS:
            best_score = -1.0
            best_threshold = thresholds[label]

            for threshold in GRID_D:
                candidate = dict(thresholds)
                candidate[label] = threshold

                filtered = {
                    doc_id: [
                        (s, e, lab)
                        for s, e, lab, conf in spans
                        if conf >= candidate[lab]
                    ]
                    for doc_id, spans in predictions.items()
                }

                score, _ = exact_span_scores(gold, filtered)
                if score > best_score:
                    best_score = score
                    best_threshold = threshold

            if best_threshold != thresholds[label]:
                changed = True
            thresholds[label] = best_threshold

        if not changed:
            break

    filtered = {
        doc_id: [
            (s, e, lab)
            for s, e, lab, conf in spans
            if conf >= thresholds[lab]
        ]
        for doc_id, spans in predictions.items()
    }

    score, per_label = exact_span_scores(gold, filtered)
    count = sum(len(spans) for spans in filtered.values())
    return score, thresholds, per_label, count


results_D = []

# The prediction helper refers to the global variable ner_model.
ner_model = ner_model_D

for epoch in range(1, 6):
    path = os.path.join(EXP_D_DIR, f"experiment_D_epoch_{epoch}.pt")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    ner_model_D.load_state_dict(ckpt["model_state_dict"])
    ner_model_D.to(device)
    ner_model_D.eval()
    ner_model = ner_model_D

    predictions = {}
    for doc_id, text in tqdm(
        val_texts.items(),
        total=len(val_texts),
        desc=f"Epoch {epoch} validation",
    ):
        predictions[doc_id] = predict_doc_constrained_with_confidence(text)

    score, thresholds, per_label, count = tune_thresholds_D(
        predictions, val_gold
    )

    results_D.append({
        "epoch": epoch,
        "train_loss": ckpt["mean_train_loss"],
        "macro_f1": score,
        "predicted_spans": count,
        "thresholds": thresholds,
        "per_label": per_label,
        "checkpoint": path,
    })

    print(
        f"Epoch {epoch}: train_loss={ckpt['mean_train_loss']:.4f}, "
        f"macro-F1={score:.4f}, spans={count}"
    )

    del predictions
    gc.collect()
    torch.cuda.empty_cache()

summary_D = pd.DataFrame([
    {
        "epoch": r["epoch"],
        "train_loss": r["train_loss"],
        "validation_macro_f1": r["macro_f1"],
        "predicted_spans": r["predicted_spans"],
    }
    for r in results_D
]).sort_values("validation_macro_f1", ascending=False)

display(summary_D)

best_D = max(results_D, key=lambda r: r["macro_f1"])
print("\nBest epoch:", best_D["epoch"])
print("Best Experiment D macro-F1:", round(best_D["macro_f1"], 4))
print("Champion reference: 0.1663")


In [ ]:

import os
import json
import torch

CHAMPION_PATH = "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
THRESHOLD_PATH = "/kaggle/working/champion_thresholds.json"

champion = torch.load(
    CHAMPION_PATH,
    map_location="cpu",
    weights_only=False
)

ner_model.load_state_dict(champion["model_state_dict"])
ner_model.to(device)
ner_model.eval()

with open(THRESHOLD_PATH) as f:
    champion_config = json.load(f)

print("Champion restored:", CHAMPION_PATH)
print("Checkpoint epoch:", champion.get("epoch"))
print("Recorded training-validation F1:",
      champion.get("val_exact_span_macro_f1"))
print("Tuned development macro-F1:",
      champion_config["validation_macro_f1"])
print("Decoder:", champion_config["decoder"])
print("Thresholds:", champion_config["thresholds"])
print("Model ready for inference.")


In [ ]:

import os
import json
import gc
import torch
from tqdm.auto import tqdm

TEST_TEXT_PATH = (
    "/kaggle/working/phase2_lner/Phase_2_LNER/Test/fold3-text.json"
)
OUTPUT_PATH = "/kaggle/working/prediction_champion.json"
THRESHOLD_PATH = "/kaggle/working/champion_thresholds.json"

assert os.path.exists(TEST_TEXT_PATH), f"Missing test file: {TEST_TEXT_PATH}"
assert os.path.exists(CHAMPION_PATH), "Champion checkpoint is missing."

with open(TEST_TEXT_PATH, "r", encoding="utf-8") as f:
    test_texts = json.load(f)

with open(THRESHOLD_PATH, "r", encoding="utf-8") as f:
    threshold_config = json.load(f)

thresholds = threshold_config["thresholds"]

# Ensure the inference function uses the restored champion.
ner_model.eval()
predictions = {}

for doc_id, text in tqdm(
    test_texts.items(),
    total=len(test_texts),
    desc="Predicting fold 3",
):
    spans_with_confidence = predict_doc_constrained_with_confidence(text)

    # Keep spans that pass their label-specific confidence threshold.
    kept = [
        [int(start), int(end), label]
        for start, end, label, confidence in spans_with_confidence
        if confidence >= thresholds.get(label, 1.0)
    ]

    # Apply the same punctuation trimming used during development.
    trimmed = []
    for start, end, label in kept:
        if label in {"DATE", "PREC", "CASENO", "STAT"}:
            while end > start and text[end - 1] in set(".,;:!?)]}"):
                end -= 1

        if end > start:
            trimmed.append([start, end, label])

    predictions[doc_id] = trimmed

    if len(predictions) % 10 == 0:
        print(
            f"Processed {len(predictions)}/{len(test_texts)} documents; "
            f"spans so far: {sum(len(x) for x in predictions.values())}"
        )

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)

print("\nPrediction file saved:", OUTPUT_PATH)
print("Documents:", len(predictions))
print("Total predicted spans:", sum(len(x) for x in predictions.values()))
print("Example document:", next(iter(predictions.items())))
print("File size: {:.2f} MB".format(os.path.getsize(OUTPUT_PATH) / 1e6))

gc.collect()
torch.cuda.empty_cache()


In [ ]:

import json
import os

OUTPUT_PATH = "/kaggle/working/prediction_champion.json"

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    check = json.load(f)

assert isinstance(check, dict), "Output must be a dictionary."
assert len(check) == 35, f"Expected 35 documents, found {len(check)}"

for doc_id, spans in check.items():
    assert isinstance(spans, list), f"Invalid spans for {doc_id}"
    for span in spans:
        assert len(span) == 3, f"Invalid span: {span}"
        start, end, label = span
        assert isinstance(start, int) and isinstance(end, int)
        assert start >= 0 and end > start
        assert label in ENTITY_LABELS, f"Unknown label: {label}"

print("Prediction JSON format verified.")
print("Documents:", len(check))
print("Total spans:", sum(len(v) for v in check.values()))
print("Saved file:", OUTPUT_PATH)


In [ ]:

import torch
import gc
from collections import defaultdict, Counter
from tqdm.auto import tqdm

# Ensure the inference helper uses the restored champion.
ner_model.eval()

confidence_diag = {}

for doc_id, text in tqdm(
    val_texts.items(),
    total=len(val_texts),
    desc="Analyzing champion confidence",
):
    confidence_diag[doc_id] = predict_doc_constrained_with_confidence(text)

threshold_grid = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.45]

rows = []

for label in ENTITY_LABELS:
    spans = [
        (s, e, lab, conf)
        for doc_spans in confidence_diag.values()
        for s, e, lab, conf in doc_spans
        if lab == label
    ]

    row = {
        "label": label,
        "raw_spans": len(spans),
        "max_confidence": round(max(
            (x[3] for x in spans), default=0.0
        ), 3),
    }

    for threshold in threshold_grid:
        row[f">={threshold:.2f}"] = sum(
            conf >= threshold for _, _, _, conf in spans
        )

    rows.append(row)

display(pd.DataFrame(rows))
print("Documents analyzed:", len(confidence_diag))
print(
    "Raw constrained spans:",
    sum(len(v) for v in confidence_diag.values())
)


In [ ]:

import pandas as pd

GRID = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.45, 0.55, 0.65]

def filter_predictions_by_thresholds(predictions, thresholds):
    filtered = {}

    for doc_id, spans in predictions.items():
        text = val_texts[doc_id]
        kept = []

        for start, end, label, confidence in spans:
            if confidence < thresholds.get(label, 1.0):
                continue

            # Match the punctuation trimming used for the champion.
            if label in {"DATE", "PREC", "CASENO", "STAT"}:
                while end > start and text[end - 1] in set(".,;:!?)]}"):
                    end -= 1

            if end > start:
                kept.append((start, end, label))

        filtered[doc_id] = kept

    return filtered


# Start from a low threshold for all labels.
candidate_thresholds = {label: 0.10 for label in ENTITY_LABELS}

for pass_num in range(3):
    for label in ENTITY_LABELS:
        best_score = -1.0
        best_threshold = candidate_thresholds[label]

        for threshold in GRID:
            trial = dict(candidate_thresholds)
            trial[label] = threshold

            filtered = filter_predictions_by_thresholds(
                confidence_diag, trial
            )
            score, _ = exact_span_scores(val_gold, filtered)

            if score > best_score:
                best_score = score
                best_threshold = threshold

        candidate_thresholds[label] = best_threshold

    filtered = filter_predictions_by_thresholds(
        confidence_diag, candidate_thresholds
    )
    score, per_label = exact_span_scores(val_gold, filtered)

    print(
        f"Pass {pass_num + 1}: macro-F1={score:.4f}, "
        f"spans={sum(len(v) for v in filtered.values())}"
    )

print("\nOptimized thresholds:")
print(candidate_thresholds)

print("\nPer-label validation scores:")
rows = []
for label, metrics in per_label.items():
    rows.append({
        "label": label,
        "F1": round(metrics["f1"], 4),
        "gold": metrics["gold"],
        "predicted": metrics["pred"],
        "correct": metrics["tp"],
        "threshold": candidate_thresholds[label],
    })

display(pd.DataFrame(rows).sort_values("F1", ascending=False))

print("Final development macro-F1:", score)
print("Champion reference: 0.1663")


In [ ]:

from collections import Counter, defaultdict
import pandas as pd

WEAK_LABELS = {"APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "JUDGE"}

# Gold spans indexed by document.
gold_by_doc = {}
for doc_id, entities in val_gold.items():
    gold_by_doc[doc_id] = [
        (int(e[0]), int(e[1]), e[2])
        for e in entities
    ]

# Count exact-span predictions by label and near-miss boundaries.
exact_by_label = Counter()
gold_by_label = Counter()
pred_by_label = Counter()
boundary_near_misses = Counter()
wrong_label_same_boundary = Counter()

for doc_id, text in val_texts.items():
    gold = gold_by_doc[doc_id]
    preds = confidence_diag[doc_id]

    # Use a permissive threshold for diagnosis, not as a final prediction.
    preds = [
        (s, e, lab, conf)
        for s, e, lab, conf in preds
        if conf >= 0.01
    ]

    gold_by_label.update(lab for _, _, lab in gold)
    pred_by_label.update(lab for _, _, lab, _ in preds)

    gold_set = set(gold)
    pred_set = {(s, e, lab) for s, e, lab, _ in preds}
    exact_by_label.update(lab for s, e, lab in pred_set & gold_set)

    for gs, ge, glab in gold:
        if glab not in WEAK_LABELS:
            continue

        for ps, pe, plab, conf in preds:
            if plab not in WEAK_LABELS:
                continue

            # Same boundary but wrong label.
            if (gs, ge) == (ps, pe) and glab != plab:
                wrong_label_same_boundary[(glab, plab)] += 1

            # Overlapping span with boundary mismatch.
            overlap = max(0, min(ge, pe) - max(gs, ps))
            if overlap > 0 and (gs, ge, glab) != (ps, pe, plab):
                boundary_near_misses[glab] += 1

rows = []
for label in sorted(WEAK_LABELS):
    rows.append({
        "label": label,
        "gold": gold_by_label[label],
        "raw_pred_at_0.01": pred_by_label[label],
        "exact_correct": exact_by_label[label],
    })

display(pd.DataFrame(rows).sort_values("gold", ascending=False))

print("\nWrong-label confusions at identical boundaries:")
for (gold_label, pred_label), count in wrong_label_same_boundary.most_common(20):
    print(f"Gold {gold_label:10s} -> predicted {pred_label:10s}: {count}")

print("\nOverlapping but non-exact candidates by gold label:")
for label, count in boundary_near_misses.most_common():
    print(label, count)


In [ ]:

from collections import Counter
import pandas as pd

def count_gold_labels(gold_docs):
    counts = Counter()
    for entities in gold_docs.values():
        for entity in entities:
            counts[entity[2]] += 1
    return counts

train_counts = count_gold_labels(train_gold)
val_counts = count_gold_labels(val_gold)

rows = []
for label in ENTITY_LABELS:
    rows.append({
        "label": label,
        "fold1_train": train_counts[label],
        "fold2_validation": val_counts[label],
        "train_to_validation_ratio": round(
            train_counts[label] / max(val_counts[label], 1), 2
        ),
    })

display(pd.DataFrame(rows).sort_values("fold1_train"))


In [ ]:
import inspect

print(inspect.getsource(LegalNERModel.forward))
print("\n--- Training loss / loop references ---")
for name in ["train_one_epoch", "train_model", "criterion"]:
    obj = globals().get(name)
    if obj is not None:
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(obj))
        except (TypeError, OSError):
            print(obj)

In [ ]:

import torch
import pandas as pd

# Confirm the labels and training counts are available.
print("BIO_LABELS:", BIO_LABELS)
print("ENTITY_LABELS:", ENTITY_LABELS)

# Count token-level labels from the existing training windows.
from collections import Counter

token_counts = Counter()
for item in train_windows_v2:
    for tag_id in item["labels"]:
        if tag_id >= 0:  # exclude IGNORE_INDEX (-100)
            token_counts[int(tag_id)] += 1

# Start with equal weights.
class_weights = torch.ones(len(BIO_LABELS), dtype=torch.float32)

# Modestly increase loss for the underperforming entity types.
# These are BIO tag names, matching the label list.
target_labels = {"WIT", "A.COUNSEL", "R.COUNSEL", "JUDGE"}

for i, tag in enumerate(BIO_LABELS):
    if tag != "O":
        entity_type = tag[2:]  # B-APP -> APP, I-APP -> APP
        if entity_type in target_labels:
            class_weights[i] = 1.5

# Keep O at 1.0 and cap all weights at 1.5.
rows = []
for i, tag in enumerate(BIO_LABELS):
    rows.append({
        "tag": tag,
        "token_count": token_counts[i],
        "proposed_weight": float(class_weights[i]),
    })

display(pd.DataFrame(rows))
print("\nWeight range:", class_weights.min().item(), "to", class_weights.max().item())


In [ ]:

import torch
import torch.nn as nn
import copy

# 1. Rebuild the Phase 1 language model from its original checkpoint.
cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

phase1_fresh_weighted = GPT2LMHeadModel(gpt2_config)
phase1_fresh_weighted.load_state_dict(
    phase1_ckpt["model_state_dict"], strict=True
)

# 2. Initialize a new NER model with a fresh classification head.
weighted_model = LegalNERModel(
    pretrained_lm=phase1_fresh_weighted,
    num_labels=len(BIO_LABELS),
).to(device)

# 3. Configure modest class weights.
weights = class_weights.to(device)

weighted_criterion = nn.CrossEntropyLoss(
    weight=weights,
    ignore_index=-100,
)

print("Fresh weighted model initialized.")
print("Device:", device)
print("Loss:", weighted_criterion)
print("Weights:", weights.detach().cpu().tolist())
print("Champion checkpoint remains unchanged:", CHAMPION_PATH)


In [ ]:

import os
import shutil

ROOT = "/kaggle/working"

total, used, free = shutil.disk_usage(ROOT)

print(f"Filesystem total: {total / 1024**3:.2f} GiB")
print(f"Filesystem used:  {used / 1024**3:.2f} GiB")
print(f"Filesystem free:  {free / 1024**3:.2f} GiB")

print("\nLargest files under /kaggle/working:")
files = []

for root, dirs, filenames in os.walk(ROOT):
    for filename in filenames:
        path = os.path.join(root, filename)
        try:
            size = os.path.getsize(path)
            files.append((size, path))
        except OSError:
            pass

for size, path in sorted(files, reverse=True)[:25]:
    print(f"{size / 1024**2:8.1f} MiB  {path}")

print(f"\nTotal files found: {len(files)}")
print(f"Total working-directory size: "
      f"{sum(size for size, _ in files) / 1024**3:.2f} GiB")


In [ ]:

import os
import shutil

# Explicitly selected disposable directories from the storage audit.
DELETE_DIRS = [
    "/kaggle/working/phase2_ner_experiment_B",
    "/kaggle/working/phase2_ner_experiment",
]

# Duplicate/obsolete checkpoints; keep the current champion.
DELETE_FILES = [
    "/kaggle/working/phase2_ner_corrected/evaluated_ner_model_v2.pt",
    "/kaggle/working/phase2_ner/best_ner_model.pt",
]

# Safety checks: never delete the champion or submitted prediction.
PROTECTED = [
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt",
    "/kaggle/working/prediction_champion.json",
    "/kaggle/working/champion_thresholds.json",
]

for path in PROTECTED:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Protected file missing; stopping: {path}")

freed_bytes = 0

for path in DELETE_DIRS:
    if os.path.isdir(path):
        size = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(path)
            for f in files
        )
        shutil.rmtree(path)
        freed_bytes += size
        print(f"Removed directory: {path} ({size / 1024**3:.2f} GiB)")
    else:
        print(f"Already absent: {path}")

for path in DELETE_FILES:
    if os.path.isfile(path):
        size = os.path.getsize(path)
        os.remove(path)
        freed_bytes += size
        print(f"Removed file: {path} ({size / 1024**2:.1f} MiB)")
    else:
        print(f"Already absent: {path}")

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nEstimated space freed: {freed_bytes / 1024**3:.2f} GiB")
print(f"Disk free now: {free / 1024**3:.2f} GiB")
print("Champion exists:", os.path.exists(PROTECTED[0]))
print("Submitted prediction exists:", os.path.exists(PROTECTED[1]))
print("Threshold file exists:", os.path.exists(PROTECTED[2]))


In [ ]:

import os
import torch

WEIGHTED_DIR = "/kaggle/working/phase2_ner_weighted"
os.makedirs(WEIGHTED_DIR, exist_ok=True)

WEIGHTED_BEST_PATH = os.path.join(
    WEIGHTED_DIR, "best_weighted_model.pt"
)

# Fresh optimizer: do not reuse the champion or a previous experiment's optimizer.
weighted_optimizer = torch.optim.AdamW(
    weighted_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Experiment directory:", WEIGHTED_DIR)
print("Best checkpoint path:", WEIGHTED_BEST_PATH)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Training batches:", len(train_loader))
print("Champion still exists:", os.path.exists(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
))


In [ ]:

import os
import json
import gc
import torch
import pandas as pd

# Use one fixed threshold configuration for every weighted-model epoch.
if "candidate_thresholds" in globals():
    EVAL_THRESHOLDS = dict(candidate_thresholds)
else:
    with open("/kaggle/working/champion_thresholds.json", "r") as f:
        EVAL_THRESHOLDS = json.load(f)["thresholds"]

assert len(val_texts) == len(val_gold), "Validation data mismatch"
assert os.path.exists(CHAMPION_PATH), "Champion checkpoint is missing"

# Keep the champion model object available and never save over its checkpoint.
champion_model_reference = ner_model
weighted_history = []
weighted_best_f1 = -1.0
weighted_best_epoch = None

def evaluate_weighted_model(model):
    """Evaluate with the notebook's constrained decoder and exact-span scorer."""
    global ner_model

    previous_model = ner_model
    try:
        # The existing predictor reads the global variable ner_model.
        ner_model = model
        ner_model.eval()

        predictions = {}
        with torch.inference_mode():
            for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
                predictions[doc_id] = (
                    predict_doc_constrained_with_confidence(text)
                )

        filtered = filter_predictions_by_thresholds(
            predictions, EVAL_THRESHOLDS
        )
        score, per_label = exact_span_scores(val_gold, filtered)

        return float(score), per_label, sum(
            len(spans) for spans in filtered.values()
        )
    finally:
        # Always restore the champion as the notebook's global model.
        ner_model = previous_model
        ner_model.eval()


# The previous cell already trained epoch 1 before hitting NameError.
# Evaluate that current state first, then train epochs 2-5.
for epoch in range(1, MAX_WEIGHTED_EPOCHS + 1):
    if epoch > 1:
        train_loss = train_weighted_one_epoch(
            weighted_model,
            weighted_optimizer,
            weighted_criterion,
        )
    else:
        # The first epoch's loss was computed by the previous cell.
        train_loss = None

    val_f1, per_label, pred_count = evaluate_weighted_model(weighted_model)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_macro_f1": val_f1,
        "predicted_spans": pred_count,
    }
    weighted_history.append(row)

    loss_text = f"{train_loss:.5f}" if train_loss is not None else "already trained"
    print(
        f"Epoch {epoch}/{MAX_WEIGHTED_EPOCHS} | "
        f"loss={loss_text} | "
        f"fold2_macro_f1={val_f1:.5f} | "
        f"predictions={pred_count}"
    )

    if val_f1 > weighted_best_f1:
        weighted_best_f1 = val_f1
        weighted_best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": weighted_model.state_dict(),
                "validation_macro_f1": val_f1,
                "class_weights": weights.detach().cpu(),
                "bio_labels": BIO_LABELS,
                "thresholds": EVAL_THRESHOLDS,
                "decoder": "BIO-constrained Viterbi",
            },
            WEIGHTED_BEST_PATH,
        )
        print("  Saved best weighted checkpoint.")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nWeighted experiment history:")
display(pd.DataFrame(weighted_history))

print(f"Best weighted F1: {weighted_best_f1:.5f}")
print(f"Best epoch: {weighted_best_epoch}")
print("Champion reference: approximately 0.16651")
print("Champion checkpoint preserved:", os.path.exists(CHAMPION_PATH))
print("Weighted checkpoint saved:", os.path.exists(WEIGHTED_BEST_PATH))
print("Global ner_model restored to champion:", ner_model is champion_model_reference)

In [ ]:

import torch
import pandas as pd

# Evaluate the current champion.
champion_f1, champion_per_label, champion_pred_count = (
    evaluate_weighted_model(ner_model)
)

# Load the best weighted checkpoint into the separate weighted model.
weighted_ckpt = torch.load(
    WEIGHTED_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)
weighted_model.load_state_dict(weighted_ckpt["model_state_dict"])
weighted_model.to(device).eval()

weighted_f1, weighted_per_label, weighted_pred_count = (
    evaluate_weighted_model(weighted_model)
)

print(f"Champion macro-F1: {champion_f1:.5f}")
print(f"Weighted macro-F1: {weighted_f1:.5f}")
print(f"Champion predicted spans: {champion_pred_count}")
print(f"Weighted predicted spans: {weighted_pred_count}")

rows = []
for label in ENTITY_LABELS:
    c = champion_per_label[label]
    w = weighted_per_label[label]
    rows.append({
        "label": label,
        "champion_F1": c["f1"],
        "weighted_F1": w["f1"],
        "delta": w["f1"] - c["f1"],
        "champion_TP": c["tp"],
        "weighted_TP": w["tp"],
        "champion_pred": c["pred"],
        "weighted_pred": w["pred"],
    })

display(pd.DataFrame(rows).sort_values("delta", ascending=False))


In [ ]:

import torch
import pandas as pd

weighted_model.eval()
weighted_diag = {}

with torch.inference_mode():
    for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
        weighted_diag[doc_id] = predict_doc_constrained_with_confidence(text)

        if i % 10 == 0:
            print(f"Processed {i}/{len(val_texts)} validation documents")

print("Weighted validation predictions cached:", len(weighted_diag))
print("Total raw candidate spans:", sum(len(x) for x in weighted_diag.values()))


In [ ]:

# Force the existing predictor to use the weighted model, then restore champion.
_previous_ner_model = ner_model
try:
    ner_model = weighted_model
    ner_model.eval()
    weighted_diag = {}

    with torch.inference_mode():
        for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
            weighted_diag[doc_id] = predict_doc_constrained_with_confidence(text)

            if i % 10 == 0:
                print(f"Processed {i}/{len(val_texts)} validation documents")
finally:
    ner_model = _previous_ner_model
    ner_model.eval()

print("Weighted predictions cached:", len(weighted_diag))
print("Total raw candidate spans:", sum(len(x) for x in weighted_diag.values()))


In [ ]:

import pandas as pd

# Try a modest grid; allow lower thresholds for weak labels.
threshold_grid = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20,
                  0.25, 0.30, 0.35, 0.45, 0.55, 0.65]

# Start with the same thresholds used for the champion.
weighted_thresholds = dict(EVAL_THRESHOLDS)

def score_weighted_thresholds(thresholds):
    filtered = filter_predictions_by_thresholds(
        weighted_diag, thresholds
    )
    score, per_label = exact_span_scores(val_gold, filtered)
    count = sum(len(spans) for spans in filtered.values())
    return float(score), per_label, count

# Coordinate search: optimize one label at a time.
for pass_no in range(3):
    changed = False

    for label in ENTITY_LABELS:
        best_score = -1.0
        best_threshold = weighted_thresholds[label]

        for threshold in threshold_grid:
            trial = dict(weighted_thresholds)
            trial[label] = threshold
            score, _, _ = score_weighted_thresholds(trial)

            if score > best_score:
                best_score = score
                best_threshold = threshold

        if weighted_thresholds[label] != best_threshold:
            weighted_thresholds[label] = best_threshold
            changed = True

    current_score, current_per_label, current_count = (
        score_weighted_thresholds(weighted_thresholds)
    )
    print(
        f"Pass {pass_no + 1}: macro-F1={current_score:.5f}, "
        f"predictions={current_count}"
    )

    if not changed:
        break

print("\nChosen weighted thresholds:")
print(weighted_thresholds)

rows = []
for label in ENTITY_LABELS:
    m = current_per_label[label]
    rows.append({
        "label": label,
        "F1": m["f1"],
        "gold": m["gold"],
        "pred": m["pred"],
        "TP": m["tp"],
        "threshold": weighted_thresholds[label],
    })

display(pd.DataFrame(rows).sort_values("F1", ascending=False))

print(f"\nBest tuned weighted F1: {current_score:.5f}")
print("Champion reference: 0.16651")


In [ ]:

from collections import Counter
import json

TRAIN_TEXT_PATH = (
    "/kaggle/working/phase2_lner/Phase_2_LNER/Train/fold1-text.json"
)
TRAIN_GOLD_PATH = (
    "/kaggle/working/phase2_lner/Phase_2_LNER/Train/fold1-gold.json"
)

with open(TRAIN_TEXT_PATH, "r", encoding="utf-8") as f:
    train_text_docs = json.load(f)

with open(TRAIN_GOLD_PATH, "r", encoding="utf-8") as f:
    train_gold_docs = json.load(f)

def check_entity_alignment(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = enc.offsets
    results = Counter()

    for ent in entities:
        start, end, label = ent[:3]

        # Ignore empty/invalid annotations.
        if start < 0 or end <= start or end > len(text):
            results["invalid_span"] += 1
            continue

        # A valid entity should align with the first and last
        # overlapping token boundaries after whitespace trimming.
        overlapping = [
            (s, e) for s, e in offsets
            if e > start and s < end
        ]

        if not overlapping:
            results["no_overlapping_tokens"] += 1
            continue

        token_start = overlapping[0][0]
        token_end = overlapping[-1][1]

        if text[start:end].strip() == text[token_start:token_end].strip():
            results["exact_text_alignment"] += 1
        else:
            results["boundary_mismatch"] += 1

    return results

totals = Counter()
by_label = Counter()

for doc_id, entities in train_gold_docs.items():
    text = train_text_docs[doc_id]
    counts = check_entity_alignment(text, entities)
    totals.update(counts)

    for ent in entities:
        start, end, label = ent[:3]
        if start < 0 or end <= start or end > len(text):
            by_label[(label, "invalid_span")] += 1
            continue

        overlapping = [
            (s, e) for s, e in tokenizer.encode(
                text, add_special_tokens=False
            ).offsets
            if e > start and s < end
        ]

        if not overlapping:
            by_label[(label, "no_overlapping_tokens")] += 1
        elif text[start:end].strip() == text[
            overlapping[0][0]:overlapping[-1][1]
        ].strip():
            by_label[(label, "exact_text_alignment")] += 1
        else:
            by_label[(label, "boundary_mismatch")] += 1

print("Overall alignment diagnostics:")
print(dict(totals))

print("\nAlignment by entity label:")
for label in ENTITY_LABELS:
    print(
        label,
        {
            kind: by_label[(label, kind)]
            for kind in [
                "exact_text_alignment",
                "boundary_mismatch",
                "no_overlapping_tokens",
                "invalid_span",
            ]
        },
    )


In [ ]:

import pandas as pd

# Best known validation-tuned thresholds for each model.
champion_thresholds_fusion = {
    "APP": 0.15, "RESP": 0.01, "A.COUNSEL": 0.01,
    "R.COUNSEL": 0.01, "JUDGE": 0.25, "WIT": 0.01,
    "AUTH": 0.35, "COURT": 0.45, "STAT": 0.30,
    "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35,
}

# These are the thresholds optimized for the weighted model.
weighted_thresholds_fusion = dict(weighted_thresholds)

champion_preds = filter_predictions_by_thresholds(
    confidence_diag, champion_thresholds_fusion
)
weighted_preds = filter_predictions_by_thresholds(
    weighted_diag, weighted_thresholds_fusion
)

# Replace only APP and JUDGE with the weighted model's predictions.
# Keep every other label from the champion.
REPLACE_LABELS = {"APP", "JUDGE"}
fused_preds = {}

for doc_id in val_texts:
    kept = [
        span for span in champion_preds[doc_id]
        if span[2] not in REPLACE_LABELS
    ]
    added = [
        span for span in weighted_preds[doc_id]
        if span[2] in REPLACE_LABELS
    ]
    fused_preds[doc_id] = kept + added

champion_score, champion_metrics = exact_span_scores(
    val_gold, champion_preds
)
weighted_score, weighted_metrics = exact_span_scores(
    val_gold, weighted_preds
)
fused_score, fused_metrics = exact_span_scores(
    val_gold, fused_preds
)

print(f"Champion F1: {champion_score:.5f}")
print(f"Weighted F1: {weighted_score:.5f}")
print(f"Label-wise fused F1: {fused_score:.5f}")
print(
    "Fused predicted spans:",
    sum(len(v) for v in fused_preds.values())
)

rows = []
for label in ENTITY_LABELS:
    rows.append({
        "label": label,
        "champion_F1": champion_metrics[label]["f1"],
        "weighted_F1": weighted_metrics[label]["f1"],
        "fused_F1": fused_metrics[label]["f1"],
        "fused_TP": fused_metrics[label]["tp"],
        "fused_pred": fused_metrics[label]["pred"],
    })

display(pd.DataFrame(rows).sort_values("fused_F1", ascending=False))


In [ ]:

import pandas as pd

BASE_REPLACE = {"APP", "JUDGE"}
CANDIDATES = [
    "RESP", "A.COUNSEL", "R.COUNSEL", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

def make_fused_predictions(replace_labels):
    result = {}
    for doc_id in val_texts:
        kept = [
            s for s in champion_preds[doc_id]
            if s[2] not in replace_labels
        ]
        added = [
            s for s in weighted_preds[doc_id]
            if s[2] in replace_labels
        ]
        result[doc_id] = kept + added
    return result

rows = []
for candidate in CANDIDATES:
    replace_set = BASE_REPLACE | {candidate}
    preds = make_fused_predictions(replace_set)
    score, metrics = exact_span_scores(val_gold, preds)

    rows.append({
        "additional_label": candidate,
        "macro_F1": score,
        "change_vs_base_fusion": score - 0.17410,
        "candidate_label_F1": metrics[candidate]["f1"],
        "candidate_label_TP": metrics[candidate]["tp"],
        "candidate_label_predictions": metrics[candidate]["pred"],
    })

display(
    pd.DataFrame(rows)
    .sort_values("macro_F1", ascending=False)
    .reset_index(drop=True)
)

print("Reference: APP + JUDGE fusion = 0.17410")


In [ ]:

import pandas as pd

# Keep champion predictions for all labels except APP and JUDGE.
FUSION_LABELS = {"APP", "JUDGE"}
grid = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20,
        0.25, 0.30, 0.35, 0.45, 0.55, 0.65]

def evaluate_app_judge_fusion(app_threshold, judge_threshold):
    # Start with the champion's currently selected predictions.
    result = {}

    for doc_id in val_texts:
        kept = [
            s for s in champion_preds[doc_id]
            if s[2] not in FUSION_LABELS
        ]

        additions = []
        for s in weighted_diag[doc_id]:
            start, end, label, confidence = s
            if label == "APP" and confidence >= app_threshold:
                additions.append((start, end, label))
            elif label == "JUDGE" and confidence >= judge_threshold:
                additions.append((start, end, label))

        result[doc_id] = kept + additions

    score, metrics = exact_span_scores(val_gold, result)
    return float(score), metrics, result

results = []

for app_t in grid:
    for judge_t in grid:
        score, metrics, _ = evaluate_app_judge_fusion(app_t, judge_t)
        results.append({
            "APP_threshold": app_t,
            "JUDGE_threshold": judge_t,
            "macro_F1": score,
            "APP_F1": metrics["APP"]["f1"],
            "JUDGE_F1": metrics["JUDGE"]["f1"],
        })

threshold_results = pd.DataFrame(results).sort_values(
    "macro_F1", ascending=False
).reset_index(drop=True)

display(threshold_results.head(15))
print("Current fusion reference: 0.17410")


In [ ]:

import inspect

print(inspect.getsource(predict_doc_constrained_with_confidence))


In [ ]:

import torch

def get_averaged_logits_for_model(model, text):
    """Reproduce the notebook's existing overlapping-window aggregation."""
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return torch.empty((0, len(BIO_LABELS))), offsets

    logit_sum = torch.zeros((len(ids), len(BIO_LABELS)), dtype=torch.float32)
    token_count = torch.zeros(len(ids), dtype=torch.float32)

    model.eval()
    with torch.inference_mode():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            chunk_len = end - start
            logit_sum[start:end] += logits[:chunk_len]
            token_count[start:end] += 1

            if end == len(ids):
                break

    averaged_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    return averaged_logits, offsets


def predict_logit_ensemble(text, weighted_alpha):
    """
    weighted_alpha=0.0 means champion only.
    weighted_alpha=0.2 means 80% champion + 20% weighted model.
    """
    champion_logits, offsets = get_averaged_logits_for_model(ner_model, text)

    if champion_logits.shape[0] == 0:
        return []

    weighted_logits, weighted_offsets = get_averaged_logits_for_model(
        weighted_model, text
    )
    if offsets != weighted_offsets:
        raise ValueError("Tokenizer offsets differ between models.")

    blended_logits = (
        (1.0 - weighted_alpha) * champion_logits
        + weighted_alpha * weighted_logits
    )

    probabilities = torch.softmax(blended_logits, dim=-1)
    pred_ids = bio_viterbi_decode(blended_logits)
    spans = decode_spans(text, offsets, pred_ids)
    pred_ids_tensor = torch.tensor(pred_ids, dtype=torch.long)

    results = []
    for start, end, label in spans:
        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < end and e > start
        ]

        if token_indices:
            selected_probs = probabilities[
                token_indices, pred_ids_tensor[token_indices]
            ]
            confidence = float(selected_probs.mean().item())
        else:
            confidence = 0.0

        results.append((start, end, label, confidence))

    return results

print("Ensemble predictor defined.")
print("Champion checkpoint:", CHAMPION_PATH)
print("Weighted checkpoint:", WEIGHTED_BEST_PATH)


In [ ]:

import pandas as pd

ensemble_results = []
ensemble_predictions = {}

for alpha in [0.0, 0.1, 0.2, 0.3]:
    print(f"Evaluating weighted contribution alpha={alpha:.1f} ...")

    raw_predictions = {}
    for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
        raw_predictions[doc_id] = predict_logit_ensemble(text, alpha)

    filtered_predictions = filter_predictions_by_thresholds(
        raw_predictions, champion_thresholds_fusion
    )
    score, per_label = exact_span_scores(val_gold, filtered_predictions)

    ensemble_predictions[alpha] = filtered_predictions
    ensemble_results.append({
        "weighted_alpha": alpha,
        "champion_alpha": 1.0 - alpha,
        "macro_F1": float(score),
        "predicted_spans": sum(
            len(spans) for spans in filtered_predictions.values()
        ),
    })

    print(f"  macro-F1={score:.5f}")

display(pd.DataFrame(ensemble_results))
print("Champion baseline: 0.16651")
print("Label-wise fusion baseline: 0.17410")


In [ ]:

def show_label_examples(label, limit=12):
    gold = {}
    for doc_id, entities in val_gold.items():
        gold[doc_id] = {
            (int(e[0]), int(e[1]), e[2])
            for e in entities
        }

    rows = []
    for doc_id, text in val_texts.items():
        gold_label_spans = {
            (s, e, lab) for s, e, lab in gold[doc_id]
            if lab == label
        }

        champion_label_spans = {
            (s, e, lab) for s, e, lab in champion_preds[doc_id]
            if lab == label
        }

        weighted_label_spans = {
            (s, e, lab) for s, e, lab in weighted_preds[doc_id]
            if lab == label
        }

        for span in sorted(gold_label_spans | champion_label_spans | weighted_label_spans):
            s, e, lab = span
            if span in gold_label_spans and span in weighted_label_spans:
                status = "weighted_exact_correct"
            elif span in gold_label_spans and span in champion_label_spans:
                status = "champion_exact_correct"
            elif span in gold_label_spans:
                status = "gold_only"
            elif span in weighted_label_spans and span in champion_label_spans:
                status = "both_predicted_false_positive"
            elif span in weighted_label_spans:
                status = "weighted_only_false_positive"
            else:
                status = "champion_only_false_positive"

            rows.append({
                "doc_id": doc_id,
                "label": label,
                "span_text": text[s:e].replace("\n", " ")[:180],
                "status": status,
                "start": s,
                "end": e,
            })

    return pd.DataFrame(rows).head(limit)

print("APP examples")
display(show_label_examples("APP", limit=15))

print("JUDGE examples")
display(show_label_examples("JUDGE", limit=15))


In [ ]:

import pandas as pd

# Keep champion's original predictions for all labels except APP and JUDGE.
# Test only stricter weighted-model confidence thresholds.
app_grid = [0.35, 0.45, 0.55, 0.65]
judge_grid = [0.20, 0.30, 0.40, 0.50]

rows = []
best_selective = None

for app_t in app_grid:
    for judge_t in judge_grid:
        fused = {}

        for doc_id in val_texts:
            kept = [
                s for s in champion_preds[doc_id]
                if s[2] not in {"APP", "JUDGE"}
            ]

            added = []
            for s, e, label, confidence in weighted_diag[doc_id]:
                if label == "APP" and confidence >= app_t:
                    added.append((s, e, label))
                elif label == "JUDGE" and confidence >= judge_t:
                    added.append((s, e, label))

            fused[doc_id] = kept + added

        score, metrics = exact_span_scores(val_gold, fused)

        row = {
            "APP_threshold": app_t,
            "JUDGE_threshold": judge_t,
            "macro_F1": float(score),
            "APP_F1": metrics["APP"]["f1"],
            "JUDGE_F1": metrics["JUDGE"]["f1"],
            "APP_TP": metrics["APP"]["tp"],
            "JUDGE_TP": metrics["JUDGE"]["tp"],
        }
        rows.append(row)

        if best_selective is None or score > best_selective["score"]:
            best_selective = {
                "score": float(score),
                "predictions": fused,
                "app_threshold": app_t,
                "judge_threshold": judge_t,
            }

display(
    pd.DataFrame(rows)
    .sort_values("macro_F1", ascending=False)
    .reset_index(drop=True)
)

print("Current fusion reference: 0.17410")
print("Best tested score:", best_selective["score"])
print("Best APP threshold:", best_selective["app_threshold"])
print("Best JUDGE threshold:", best_selective["judge_threshold"])


In [ ]:

import pandas as pd

rows = []

for doc_id, text in val_texts.items():
    gold_judges = [
        (int(e[0]), int(e[1]), e[2])
        for e in val_gold[doc_id]
        if e[2] == "JUDGE"
    ]

    weighted_judges = [
        (int(s), int(e), label, float(conf))
        for s, e, label, conf in weighted_diag[doc_id]
        if label == "JUDGE" and conf >= 0.30
    ]

    for ps, pe, label, conf in weighted_judges:
        exact = any(ps == gs and pe == ge for gs, ge, _ in gold_judges)
        overlaps = [
            (gs, ge) for gs, ge, _ in gold_judges
            if ps < ge and pe > gs
        ]

        if exact:
            status = "exact"
        elif overlaps:
            status = "overlapping_boundary_error"
        else:
            status = "no_gold_overlap"

        rows.append({
            "doc_id": doc_id,
            "prediction": text[ps:pe].replace("\n", " ")[:120],
            "confidence": round(conf, 3),
            "status": status,
            "overlapping_gold": "; ".join(
                text[gs:ge].replace("\n", " ")[:120]
                for gs, ge in overlaps
            ),
            "pred_start": ps,
            "pred_end": pe,
        })

judge_error_df = pd.DataFrame(rows)
display(
    judge_error_df.sort_values(
        ["status", "confidence"],
        ascending=[True, False]
    )
)

print("\nCounts by status:")
print(judge_error_df["status"].value_counts())


In [ ]:

import pandas as pd

def evaluate_judge_oracle(predictions):
    """
    Diagnostic only: replace a predicted JUDGE span with an overlapping
    gold JUDGE span when exactly one gold span overlaps it.
    Never use this oracle procedure for final/test predictions.
    """
    repaired = {}
    repairs = 0
    ambiguous = 0

    for doc_id, spans in predictions.items():
        text = val_texts[doc_id]
        gold_judges = [
            (int(e[0]), int(e[1]), e[2])
            for e in val_gold[doc_id]
            if e[2] == "JUDGE"
        ]

        output = []
        for s, e, label in spans:
            if label != "JUDGE":
                output.append((s, e, label))
                continue

            overlaps = [
                (gs, ge, glabel)
                for gs, ge, glabel in gold_judges
                if s < ge and e > gs
            ]

            if len(overlaps) == 1 and (s, e, label) != overlaps[0]:
                gs, ge, _ = overlaps[0]
                output.append((gs, ge, "JUDGE"))
                repairs += 1
            else:
                if len(overlaps) > 1:
                    ambiguous += 1
                output.append((s, e, label))

        repaired[doc_id] = output

    score, metrics = exact_span_scores(val_gold, repaired)
    return score, metrics, repairs, ambiguous

# Evaluate the current APP + JUDGE fusion predictions.
oracle_score, oracle_metrics, repairs, ambiguous = (
    evaluate_judge_oracle(fused_preds)
)

base_score, base_metrics = exact_span_scores(val_gold, fused_preds)

print(f"Current fusion macro-F1: {base_score:.5f}")
print(f"Oracle-repaired macro-F1: {oracle_score:.5f}")
print(f"Potentially repaired spans: {repairs}")
print(f"Ambiguous overlaps left unchanged: {ambiguous}")
print("\nJUDGE metrics:")
print("Current:", base_metrics["JUDGE"])
print("Oracle :", oracle_metrics["JUDGE"])


In [ ]:

import pandas as pd

rows = []

for doc_id, text in val_texts.items():
    gold_judges = [
        (int(e[0]), int(e[1]))
        for e in val_gold[doc_id]
        if e[2] == "JUDGE"
    ]

    for s, e, label, conf in weighted_diag[doc_id]:
        if label != "JUDGE" or conf < 0.30:
            continue

        overlaps = [
            (gs, ge) for gs, ge in gold_judges
            if s < ge and e > gs
        ]
        if not overlaps:
            continue

        left = max(0, s - 35)
        right = min(len(text), e + 35)

        rows.append({
            "doc_id": doc_id,
            "prediction": text[s:e],
            "confidence": round(conf, 3),
            "context": text[left:right].replace("\n", " "),
            "prediction_end_char": repr(text[e:e+12]),
            "prediction_start_char": repr(text[max(0,s-12):s]),
            "gold_overlap_count": len(overlaps),
        })

display(pd.DataFrame(rows).head(30))


In [ ]:
import re
import pandas as pd

def repair_judge_continuations(predictions, texts, min_conf=0.30):
    repaired = {}
    changes = []

    for doc_id, spans in predictions.items():
        text = texts[doc_id]
        output = []

        for s, e, label, conf in spans:
            if label != "JUDGE" or conf < min_conf:
                output.append((s, e, label, conf))
                continue

            extension = re.match(r"[A-Za-z]+", text[e:])

            if extension:
                extra = extension.group(0)
                old_span = text[s:e]

                if old_span and old_span[-1].isalpha() and len(extra) <= 6:
                    new_end = e + len(extra)
                    new_span = text[s:new_end]

                    if re.fullmatch(r"[A-Za-z][A-Za-z.\s,'-]*", new_span):
                        output.append((s, new_end, label, conf))
                        changes.append({
                            "doc_id": doc_id,
                            "old": old_span,
                            "new": new_span,
                            "confidence": round(float(conf), 4)
                        })
                        continue

            output.append((s, e, label, conf))

        repaired[doc_id] = output

    return repaired, pd.DataFrame(changes)


# IMPORTANT: Use raw predictions here, not weighted_filtered.
repaired_weighted_raw, repair_log = repair_judge_continuations(
    weighted_diag,
    val_texts,
    min_conf=0.30
)

print("Proposed repairs:", len(repair_log))
display(repair_log.head(30))

In [ ]:
# Original weighted predictions
weighted_filtered = filter_predictions_by_thresholds(
    weighted_diag, weighted_thresholds_fusion
)

# Repaired weighted predictions
repaired_weighted = filter_predictions_by_thresholds(
    repaired_weighted_raw, weighted_thresholds_fusion
)

# Compare JUDGE + overall macro F1
baseline_f1, baseline_per_label = exact_span_scores(val_gold, weighted_filtered)
repaired_f1, repaired_per_label = exact_span_scores(val_gold, repaired_weighted)

print(f"Baseline macro F1: {baseline_f1:.6f}")
print(f"Repaired macro F1: {repaired_f1:.6f}")
print(f"Change:            {repaired_f1 - baseline_f1:+.6f}")

print("\nJUDGE baseline:", baseline_per_label["JUDGE"])
print("JUDGE repaired:", repaired_per_label["JUDGE"])

In [ ]:
# Start with the best label-wise fusion predictions.
# Champion supplies all labels except APP and JUDGE.
# Weighted model supplies APP and JUDGE.

fused_repaired = {}

for doc_id in val_texts:
    champion_spans = champion_preds.get(doc_id, [])
    weighted_spans = repaired_weighted.get(doc_id, [])

    fused_repaired[doc_id] = [
        (s, e, label)
        for s, e, label in champion_spans
        if label not in {"APP", "JUDGE"}
    ] + [
        (s, e, label)
        for s, e, label in weighted_spans
        if label in {"APP", "JUDGE"}
    ]

fusion_baseline_f1, fusion_baseline_metrics = exact_span_scores(
    val_gold, fused_preds
)

fusion_repaired_f1, fusion_repaired_metrics = exact_span_scores(
    val_gold, fused_repaired
)

print(f"Original fusion macro F1: {fusion_baseline_f1:.6f}")
print(f"Repaired fusion macro F1: {fusion_repaired_f1:.6f}")
print(f"Change:                   {fusion_repaired_f1 - fusion_baseline_f1:+.6f}")

print("\nJUDGE baseline:", fusion_baseline_metrics["JUDGE"])
print("JUDGE repaired:", fusion_repaired_metrics["JUDGE"])

In [ ]:
print("Repaired fusion — per-label metrics")
for label, metrics in fusion_repaired_metrics.items():
    print(
        f"{label:10s} "
        f"F1={metrics['f1']:.4f} "
        f"TP={metrics['tp']:3d} "
        f"Pred={metrics['pred']:3d} "
        f"Gold={metrics['gold']:3d}"
    )

In [ ]:
# Tune thresholds on cached predictions only.
# This does not retrain the model or overwrite the submission.

import itertools

base_thresholds = {
    "APP": 0.35,
    "RESP": 0.01,
    "A.COUNSEL": 0.01,
    "R.COUNSEL": 0.01,
    "JUDGE": 0.30,
    "WIT": 0.01,
    "AUTH": 0.35,
    "COURT": 0.45,
    "STAT": 0.30,
    "PREC": 0.35,
    "DATE": 0.55,
    "CASENO": 0.35,
}

# Keep the established thresholds for strong labels.
# Search a modest grid for the most overpredicted labels.
search_grid = {
    "PREC":  [0.35, 0.45, 0.55, 0.65],
    "DATE":  [0.45, 0.55, 0.65, 0.75],
    "CASENO":[0.25, 0.35, 0.45, 0.55],
    "AUTH":  [0.25, 0.35, 0.45, 0.55],
    "STAT":  [0.25, 0.30, 0.40, 0.50],
}

best_score = -1
best_thresholds = None
results = []

for values in itertools.product(*search_grid.values()):
    thresholds = base_thresholds.copy()

    for label, value in zip(search_grid.keys(), values):
        thresholds[label] = value

    # Champion predictions for labels it supplies.
    champ = filter_predictions_by_thresholds(
        confidence_diag, thresholds
    )

    # Weighted predictions for APP and JUDGE.
    weighted = filter_predictions_by_thresholds(
        repaired_weighted_raw, thresholds
    )

    fused = {}
    for doc_id in val_texts:
        fused[doc_id] = [
            (s, e, label)
            for s, e, label in champ.get(doc_id, [])
            if label not in {"APP", "JUDGE"}
        ] + [
            (s, e, label)
            for s, e, label in weighted.get(doc_id, [])
            if label in {"APP", "JUDGE"}
        ]

    score, metrics = exact_span_scores(val_gold, fused)
    results.append((score, thresholds.copy(), metrics))

    if score > best_score:
        best_score = score
        best_thresholds = thresholds.copy()

print(f"Best macro F1: {best_score:.6f}")
print("Best thresholds:")
for label, value in best_thresholds.items():
    print(f"  {label:10s}: {value:.2f}")

results.sort(key=lambda x: x[0], reverse=True)
print("\nTop 5 scores:")
for score, thresholds, _ in results[:5]:
    print(f"{score:.6f} | " + ", ".join(
        f"{label}={thresholds[label]:.2f}"
        for label in search_grid
    ))

In [ ]:
import pandas as pd

def inspect_label_errors(label, predictions, gold_by_doc, texts, limit=15):
    false_positives = []
    false_negatives = []

    for doc_id, text in texts.items():
        pred_spans = {
            (int(s), int(e), lab)
            for s, e, lab in predictions.get(doc_id, [])
            if lab == label
        }

        gold_spans = set()
        for item in gold_by_doc.get(doc_id, []):
            s, e, lab = item[:3]  # Ignore the optional fourth field (entity text)
            if lab == label:
                gold_spans.add((int(s), int(e), lab))

        for s, e, lab in pred_spans - gold_spans:
            overlaps = [
                (gs, ge, gl)
                for gs, ge, gl in gold_spans
                if max(s, gs) < min(e, ge)
            ]
            false_positives.append({
                "doc_id": doc_id,
                "prediction": text[s:e],
                "context": text[max(0, s-60):min(len(text), e+60)],
                "overlapping_gold": [
                    text[gs:ge] for gs, ge, _ in overlaps
                ],
            })

        for s, e, lab in gold_spans - pred_spans:
            overlaps = [
                (ps, pe, pl)
                for ps, pe, pl in pred_spans
                if max(s, ps) < min(e, pe)
            ]
            false_negatives.append({
                "doc_id": doc_id,
                "gold": text[s:e],
                "context": text[max(0, s-60):min(len(text), e+60)],
                "overlapping_predictions": [
                    text[ps:pe] for ps, pe, _ in overlaps
                ],
            })

    print(f"\n=== {label}: FALSE POSITIVES ===")
    display(pd.DataFrame(false_positives).head(limit))

    print(f"\n=== {label}: FALSE NEGATIVES ===")
    display(pd.DataFrame(false_negatives).head(limit))


inspect_label_errors("PREC", fused_repaired, val_gold, val_texts)
inspect_label_errors("APP", fused_repaired, val_gold, val_texts)

In [ ]:
import pandas as pd

def per_document_label_errors(label, predictions, gold_by_doc, texts):
    rows = []

    for doc_id, text in texts.items():
        pred = {
            (int(s), int(e), lab)
            for s, e, lab in predictions.get(doc_id, [])
            if lab == label
        }

        gold = {
            (int(item[0]), int(item[1]), item[2])
            for item in gold_by_doc.get(doc_id, [])
            if item[2] == label
        }

        tp = len(pred & gold)
        rows.append({
            "doc_id": doc_id,
            "gold": len(gold),
            "pred": len(pred),
            "tp": tp,
            "recall": tp / len(gold) if gold else None,
            "precision": tp / len(pred) if pred else None,
            "text_length": len(text),
        })

    return pd.DataFrame(rows).sort_values(
        ["gold", "pred"], ascending=False
    )

for label in ["APP", "PREC"]:
    print(f"\n=== {label}: worst document-level recall ===")
    df = per_document_label_errors(
        label, fused_repaired, val_gold, val_texts
    )
    display(df[df["gold"] > 0].sort_values("recall").head(15))

In [ ]:
import pandas as pd

def overlap_diagnostic(label, predictions, gold_by_doc, texts):
    rows = []

    for doc_id, text in texts.items():
        pred = [
            (int(s), int(e))
            for s, e, lab in predictions.get(doc_id, [])
            if lab == label
        ]

        gold = [
            (int(item[0]), int(item[1]))
            for item in gold_by_doc.get(doc_id, [])
            if item[2] == label
        ]

        exact = sum(1 for g in gold if g in pred)
        overlapping_gold = sum(
            1 for gs, ge in gold
            if any(max(gs, ps) < min(ge, pe) for ps, pe in pred)
        )

        rows.append({
            "doc_id": doc_id,
            "gold": len(gold),
            "pred": len(pred),
            "exact_matches": exact,
            "gold_with_any_overlap": overlapping_gold,
            "boundary_error_only": overlapping_gold - exact,
            "completely_missed": len(gold) - overlapping_gold,
        })

    return pd.DataFrame(rows)

for label in ["APP", "PREC"]:
    df = overlap_diagnostic(label, fused_repaired, val_gold, val_texts)

    print(f"\n=== {label}: aggregate overlap diagnostic ===")
    print(df[[
        "gold", "pred", "exact_matches",
        "gold_with_any_overlap", "boundary_error_only",
        "completely_missed"
    ]].sum())

    print(f"\n=== {label}: documents with most boundary errors ===")
    display(
        df.sort_values("boundary_error_only", ascending=False).head(10)
    )

In [ ]:
import pandas as pd

def inspect_boundary_pairs(label, predictions, gold_by_doc, texts, limit=40):
    rows = []

    for doc_id, text in texts.items():
        pred = [
            (int(s), int(e))
            for s, e, lab in predictions.get(doc_id, [])
            if lab == label
        ]
        gold = [
            (int(item[0]), int(item[1]))
            for item in gold_by_doc.get(doc_id, [])
            if item[2] == label
        ]

        for gs, ge in gold:
            overlaps = [
                (ps, pe)
                for ps, pe in pred
                if max(gs, ps) < min(ge, pe)
            ]

            # Show only gold entities with no exact prediction.
            if (gs, ge) not in pred and overlaps:
                for ps, pe in overlaps:
                    rows.append({
                        "doc_id": doc_id,
                        "gold_text": text[gs:ge],
                        "pred_text": text[ps:pe],
                        "gold_start": gs,
                        "pred_start": ps,
                        "start_delta": ps - gs,
                        "gold_end": ge,
                        "pred_end": pe,
                        "end_delta": pe - ge,
                        "context": text[max(0, gs-45):min(len(text), ge+45)]
                    })

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"No overlapping boundary errors found for {label}.")
        return df

    print(f"{label}: {len(df)} overlapping prediction/gold pairs")
    display(df.head(limit))
    print("\nBoundary delta summary:")
    display(df[["start_delta", "end_delta"]].describe())
    return df

prec_boundary_pairs = inspect_boundary_pairs(
    "PREC", fused_repaired, val_gold, val_texts, limit=40
)

In [ ]:
import re
from collections import Counter

prec_texts = [
    text[s:e]
    for doc_id, spans in fused_repaired.items()
    for s, e, label in spans
    if label == "PREC"
    for text in [val_texts[doc_id]]
]

suffix_patterns = Counter()

for span in prec_texts:
    checks = {
        "ends_with_supra": bool(re.search(r"\(\s*supra\s*\)\s*$", span, re.I)),
        "ends_with_citation_bracket": bool(re.search(r"\[[^\]]{0,50}\]\s*$", span)),
        "ends_with_report_citation": bool(re.search(
            r"\b\d{4}\s*\([^)]{0,20}\)\s*[A-Z][A-Za-z.() ]{0,20}\s+\d+\s*$",
            span
        )),
        "contains_vs": bool(re.search(r"\b(vs?\.?|versus)\b", span, re.I)),
        "very_long": len(span) > 100,
    }

    for name, matched in checks.items():
        if matched:
            suffix_patterns[name] += 1

print("PREC predictions:", len(prec_texts))
print("Pattern counts:")
for name, count in suffix_patterns.most_common():
    print(f"{name:28s} {count:5d}")

In [ ]:
import re
import pandas as pd

def trim_prec_suffixes(predictions, texts):
    repaired = {}
    changes = []

    for doc_id, spans in predictions.items():
        text = texts[doc_id]
        output = []

        for s, e, label in spans:
            if label != "PREC":
                output.append((s, e, label))
                continue

            original = text[s:e]
            candidate = original

            # Remove a trailing "(supra)" marker.
            candidate = re.sub(
                r"\s*\(\s*supra\s*\)\s*$", "", candidate, flags=re.I
            )

            # Remove a trailing bracketed law-report citation.
            candidate = re.sub(
                r"\s*\[(?:AIR|SCC|SCR|All ER|WLR)\b[^\]]{0,80}\]\s*$",
                "", candidate, flags=re.I
            )

            # Remove a trailing law-report reference such as
            # "2017 (3) CLJ (Cal) 593".
            candidate = re.sub(
                r"\s+\b\d{4}\s*\([^)]{1,20}\)\s*"
                r"[A-Z][A-Za-z.() ]{0,20}\s+\d+\s*$",
                "", candidate
            )

            candidate = candidate.rstrip(" ,;:")
            new_end = s + len(candidate)

            # Only accept non-empty trims that actually shorten the span.
            if candidate and len(candidate) < len(original):
                output.append((s, new_end, label))
                changes.append({
                    "doc_id": doc_id,
                    "old": original,
                    "new": candidate
                })
            else:
                output.append((s, e, label))

        repaired[doc_id] = output

    return repaired, pd.DataFrame(changes)


prec_suffix_repaired, prec_suffix_log = trim_prec_suffixes(
    fused_repaired, val_texts
)

print("Changed spans:", len(prec_suffix_log))
display(prec_suffix_log.head(25))

new_f1, new_metrics = exact_span_scores(val_gold, prec_suffix_repaired)

print(f"\nBaseline fusion F1: {fusion_repaired_f1:.6f}")
print(f"Suffix-repair F1:  {new_f1:.6f}")
print(f"Change:            {new_f1 - fusion_repaired_f1:+.6f}")

print("\nPREC baseline:", fusion_repaired_metrics["PREC"])
print("PREC repaired:", new_metrics["PREC"])

In [ ]:
required_vars = [
    "confidence_diag",
    "weighted_diag",
    "champion_thresholds_fusion",
    "weighted_thresholds_fusion",
    "repaired_weighted_raw",
    "fused_repaired",
    "val_texts",
    "val_gold",
    "tokenizer",
    "ner_model",
    "device",
]

missing = [name for name in required_vars if name not in globals()]

if missing:
    print("Missing variables:", missing)
else:
    print("All required variables are available.")
    print("Validation documents:", len(val_texts))
    print("Baseline macro F1:", f"{fusion_repaired_f1:.6f}")

In [ ]:
import torch
import pandas as pd

PREC_LABELS = {
    "B-PREC": BIO_LABELS.index("B-PREC"),
    "I-PREC": BIO_LABELS.index("I-PREC"),
    "O": BIO_LABELS.index("O"),
}

def get_document_token_probabilities(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return offsets, torch.empty((0, len(BIO_LABELS)))

    logit_sum = torch.zeros((len(ids), len(BIO_LABELS)))
    token_count = torch.zeros(len(ids))

    ner_model.eval()
    with torch.no_grad():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = ner_model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            logit_sum[start:end] += logits[:end-start]
            token_count[start:end] += 1

            if end == len(ids):
                break

    averaged_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(averaged_logits, dim=-1)


prec_token_cache = {}

for i, (doc_id, text) in enumerate(val_texts.items(), 1):
    offsets, probs = get_document_token_probabilities(text)
    prec_token_cache[doc_id] = (offsets, probs)

    if i % 5 == 0 or i == len(val_texts):
        print(f"Processed {i}/{len(val_texts)} validation documents")

print("Token-probability cache ready:", len(prec_token_cache), "documents")

In [ ]:
import re
import pandas as pd

# Baseline: current best repaired fusion
PREC_BASELINE_F1 = 0.176267

def get_prec_boundary_candidates(doc_id, s, e, max_shift=12):
    text = val_texts[doc_id]
    offsets, probs = prec_token_cache[doc_id]

    candidates = {(s, e)}

    # Consider token boundaries near the current start/end.
    start_positions = {
        offsets[i][0]
        for i, (a, b) in enumerate(offsets)
        if b > s - max_shift and a < s + max_shift
    }
    end_positions = {
        offsets[i][1]
        for i, (a, b) in enumerate(offsets)
        if b > e - max_shift and a < e + max_shift
    }

    for ns in start_positions:
        for ne in end_positions:
            if ns < ne and abs(ns - s) <= max_shift and abs(ne - e) <= max_shift:
                candidates.add((ns, ne))

    # Prefer boundaries that do not begin/end in whitespace.
    return [
        (ns, ne)
        for ns, ne in candidates
        if ns < ne
        and not text[ns:ns+1].isspace()
        and not text[ne-1:ne].isspace()
    ]


def candidate_prec_score(doc_id, s, e, ns, ne):
    text = val_texts[doc_id]
    offsets, probs = prec_token_cache[doc_id]

    # A small boundary adjustment only; don't alter the entity's core.
    if abs(ns - s) > 12 or abs(ne - e) > 12:
        return -1e9

    covered = [
        i for i, (a, b) in enumerate(offsets)
        if b > ns and a < ne
    ]
    if not covered:
        return -1e9

    # Reward PREC evidence; penalize O evidence at the boundaries.
    p_prec = probs[covered][:, [
        PREC_LABELS["B-PREC"], PREC_LABELS["I-PREC"]
    ]].sum(dim=-1)

    p_o = probs[covered, PREC_LABELS["O"]]
    score = float(torch.log(p_prec.clamp_min(1e-8)).mean())
    score -= 0.25 * float(torch.log(p_o.clamp_min(1e-8)).mean())

    # Slight penalty for unnecessarily changing existing boundaries.
    score -= 0.01 * (abs(ns - s) + abs(ne - e))
    return score


# Generate boundary candidate sets for existing PREC predictions.
prec_candidates = {}

for doc_id, spans in fused_repaired.items():
    for s, e, label in spans:
        if label != "PREC":
            continue

        key = (doc_id, s, e)
        candidates = get_prec_boundary_candidates(doc_id, s, e)
        scored = [
            (
                candidate_prec_score(doc_id, s, e, ns, ne),
                ns, ne
            )
            for ns, ne in candidates
        ]
        prec_candidates[key] = sorted(scored, reverse=True)

print("PREC spans:", len(prec_candidates))
print("Candidate boundary sets prepared.")

In [ ]:
import pandas as pd

# Start from the current best fusion.
prec_boundary_test = {
    doc_id: list(spans)
    for doc_id, spans in fused_repaired.items()
}

# Replace each PREC span with its highest-scoring candidate boundary.
for doc_id, spans in prec_boundary_test.items():
    updated = []

    for s, e, label in spans:
        if label != "PREC":
            updated.append((s, e, label))
            continue

        candidates = prec_candidates.get((doc_id, s, e), [])

        if candidates:
            score, new_s, new_e = candidates[0]
            updated.append((new_s, new_e, label))
        else:
            updated.append((s, e, label))

    prec_boundary_test[doc_id] = updated

# Evaluate against the existing validation baseline.
candidate_f1, candidate_metrics = exact_span_scores(
    val_gold, prec_boundary_test
)

print(f"Baseline macro F1: {fusion_repaired_f1:.6f}")
print(f"Candidate macro F1: {candidate_f1:.6f}")
print(f"Change:             {candidate_f1 - fusion_repaired_f1:+.6f}")

print("\nPREC baseline:", fusion_repaired_metrics["PREC"])
print("PREC candidate:", candidate_metrics["PREC"])

In [ ]:
import os
import json
from pathlib import Path

TEST_PATH = "/kaggle/working/phase2_lner/Phase_2_LNER/Test/fold3-text.json"
OUTPUT_PATH = "/kaggle/working/prediction.json"

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

# Normalize the expected doc_id -> text mapping.
if isinstance(test_data, dict):
    test_texts = {}
    for doc_id, value in test_data.items():
        if isinstance(value, str):
            test_texts[str(doc_id)] = value
        elif isinstance(value, dict) and isinstance(value.get("text"), str):
            test_texts[str(doc_id)] = value["text"]
        else:
            raise ValueError(
                f"Unexpected test record for doc_id={doc_id}: "
                f"{type(value).__name__}"
            )
else:
    raise ValueError(
        f"Expected a JSON object mapping document IDs to text, "
        f"got {type(test_data).__name__}. Inspect test_data before proceeding."
    )

print("Test documents:", len(test_texts))
print("First document ID:", next(iter(test_texts)))
print("First document length:", len(next(iter(test_texts.values()))))
print("Output path:", OUTPUT_PATH)

In [ ]:
import torch

assert "weighted_model" in globals(), (
    "weighted_model is missing. Load the saved weighted checkpoint first."
)
assert "predict_doc_constrained_with_confidence" in globals()

# Save the current model reference so it can be restored.
_original_ner_model = ner_model

test_champion_diag = {}
test_weighted_diag = {}

try:
    # Champion predictions
    ner_model = _original_ner_model
    ner_model.eval()

    for i, (doc_id, text) in enumerate(test_texts.items(), 1):
        test_champion_diag[doc_id] = (
            predict_doc_constrained_with_confidence(text)
        )

        if i % 5 == 0 or i == len(test_texts):
            print(f"Champion: {i}/{len(test_texts)}")

    # Weighted-model predictions
    ner_model = weighted_model
    ner_model.eval()

    for i, (doc_id, text) in enumerate(test_texts.items(), 1):
        test_weighted_diag[doc_id] = (
            predict_doc_constrained_with_confidence(text)
        )

        if i % 5 == 0 or i == len(test_texts):
            print(f"Weighted: {i}/{len(test_texts)}")

finally:
    # Restore the original model even if inference raises an error.
    ner_model = _original_ner_model

print("\nInference completed.")
print("Champion documents:", len(test_champion_diag))
print("Weighted documents:", len(test_weighted_diag))

In [ ]:
# The previously validated JUDGE continuation repair.
# This function accepts raw four-tuples: (start, end, label, confidence).
repaired_test_weighted_raw, test_repair_log = repair_judge_continuations(
    test_weighted_diag,
    test_texts,
    min_conf=0.30
)

print("Test JUDGE continuation repairs:", len(test_repair_log))
if not test_repair_log.empty:
    display(test_repair_log.head(20))

# Apply the previously selected per-label confidence thresholds.
test_champion_filtered = filter_predictions_by_thresholds(
    test_champion_diag,
    champion_thresholds_fusion
)

test_weighted_filtered = filter_predictions_by_thresholds(
    repaired_test_weighted_raw,
    weighted_thresholds_fusion
)

# Label-wise fusion: weighted model supplies APP and JUDGE;
# champion supplies all other labels.
REPLACE_LABELS = {"APP", "JUDGE"}

final_predictions = {}

for doc_id in test_texts:
    champion_spans = test_champion_filtered.get(doc_id, [])
    weighted_spans = test_weighted_filtered.get(doc_id, [])

    fused = [
        (int(s), int(e), label)
        for s, e, label in champion_spans
        if label not in REPLACE_LABELS
    ]

    fused += [
        (int(s), int(e), label)
        for s, e, label in weighted_spans
        if label in REPLACE_LABELS
    ]

    # Deterministic ordering by position.
    final_predictions[doc_id] = sorted(
        set(fused),
        key=lambda x: (x[0], x[1], x[2])
    )

print("Prediction documents:", len(final_predictions))
print("Total predicted spans:", sum(
    len(spans) for spans in final_predictions.values()
))

In [ ]:
def filter_predictions_by_thresholds_for_texts(predictions, thresholds, texts):
    filtered = {}

    for doc_id, spans in predictions.items():
        text = texts[doc_id]
        kept = []

        for s, e, label, conf in spans:
            if conf >= thresholds.get(label, 1.0):
                # Keep only valid, non-empty text spans.
                if 0 <= s < e <= len(text):
                    kept.append((int(s), int(e), label))

        filtered[doc_id] = kept

    return filtered


# Filter using TEST texts rather than val_texts.
test_champion_filtered = filter_predictions_by_thresholds_for_texts(
    test_champion_diag,
    champion_thresholds_fusion,
    test_texts
)

test_weighted_filtered = filter_predictions_by_thresholds_for_texts(
    repaired_test_weighted_raw,
    weighted_thresholds_fusion,
    test_texts
)

print("Champion filtered:", len(test_champion_filtered))
print("Weighted filtered:", len(test_weighted_filtered))

In [ ]:
REPLACE_LABELS = {"APP", "JUDGE"}

final_predictions = {}

for doc_id in test_texts:
    fused = [
        (int(s), int(e), label)
        for s, e, label in test_champion_filtered.get(doc_id, [])
        if label not in REPLACE_LABELS
    ]

    fused += [
        (int(s), int(e), label)
        for s, e, label in test_weighted_filtered.get(doc_id, [])
        if label in REPLACE_LABELS
    ]

    final_predictions[doc_id] = sorted(
        set(fused),
        key=lambda x: (x[0], x[1], x[2])
    )

print("Test documents:", len(final_predictions))
print("Total spans:", sum(len(v) for v in final_predictions.values()))

In [ ]:
import os
import json
import torch
import pandas as pd

def filter_predictions_by_thresholds_for_texts(
    predictions, thresholds, texts
):
    filtered = {}

    for doc_id, spans in predictions.items():
        text = texts[doc_id]
        kept = []

        for s, e, label, conf in spans:
            s, e = int(s), int(e)

            if (
                conf >= thresholds.get(label, 1.0)
                and 0 <= s < e <= len(text)
            ):
                kept.append((s, e, label))

        filtered[doc_id] = kept

    return filtered


assert set(test_champion_diag) == set(test_texts), \
    "Champion predictions do not cover exactly the test documents."

assert set(test_weighted_diag) == set(test_texts), \
    "Weighted predictions do not cover exactly the test documents."

# Apply the same validated JUDGE continuation repair.
repaired_test_weighted_raw, test_repair_log = repair_judge_continuations(
    test_weighted_diag,
    test_texts,
    min_conf=0.30
)

print("JUDGE continuation repairs:", len(test_repair_log))
if not test_repair_log.empty:
    display(test_repair_log)

test_champion_filtered = filter_predictions_by_thresholds_for_texts(
    test_champion_diag,
    champion_thresholds_fusion,
    test_texts
)

test_weighted_filtered = filter_predictions_by_thresholds_for_texts(
    repaired_test_weighted_raw,
    weighted_thresholds_fusion,
    test_texts
)

print("Champion predictions filtered:", len(test_champion_filtered))
print("Weighted predictions filtered:", len(test_weighted_filtered))

In [ ]:
REPLACE_LABELS = {"APP", "JUDGE"}

final_predictions = {}

for doc_id in test_texts:
    champion_spans = test_champion_filtered.get(doc_id, [])
    weighted_spans = test_weighted_filtered.get(doc_id, [])

    fused = [
        (int(s), int(e), label)
        for s, e, label in champion_spans
        if label not in REPLACE_LABELS
    ]

    fused.extend(
        (int(s), int(e), label)
        for s, e, label in weighted_spans
        if label in REPLACE_LABELS
    )

    # Remove duplicate spans and sort by character position.
    final_predictions[doc_id] = sorted(
        set(fused),
        key=lambda x: (x[0], x[1], x[2])
    )

print("Test documents:", len(final_predictions))
print("Total predicted spans:", sum(
    len(spans) for spans in final_predictions.values()
))

In [ ]:
OUTPUT_PATH = "/kaggle/working/prediction.json"

submission = {
    doc_id: [
        [int(s), int(e), label]
        for s, e, label in spans
    ]
    for doc_id, spans in final_predictions.items()
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False)

print("Saved:", OUTPUT_PATH)
print("File size:", os.path.getsize(OUTPUT_PATH), "bytes")

In [ ]:
VALID_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL",
    "JUDGE", "WIT", "AUTH", "COURT",
    "STAT", "PREC", "DATE", "CASENO"
}

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    checked = json.load(f)

errors = []
label_counts = {label: 0 for label in sorted(VALID_LABELS)}
total_spans = 0

missing_docs = set(test_texts) - set(checked)
extra_docs = set(checked) - set(test_texts)

if missing_docs:
    errors.append(f"Missing documents: {len(missing_docs)}")
if extra_docs:
    errors.append(f"Unexpected documents: {len(extra_docs)}")

for doc_id, text in test_texts.items():
    spans = checked.get(doc_id)

    if not isinstance(spans, list):
        errors.append(f"{doc_id}: prediction value is not a list")
        continue

    seen = set()

    for i, span in enumerate(spans):
        if not isinstance(span, list) or len(span) != 3:
            errors.append(f"{doc_id}, span {i}: expected [start, end, label]")
            continue

        s, e, label = span

        if (
            not isinstance(s, int)
            or isinstance(s, bool)
            or not isinstance(e, int)
            or isinstance(e, bool)
        ):
            errors.append(f"{doc_id}, span {i}: offsets must be integers")
            continue

        if not (0 <= s < e <= len(text)):
            errors.append(
                f"{doc_id}, span {i}: invalid offsets [{s}, {e}) "
                f"for text length {len(text)}"
            )

        if label not in VALID_LABELS:
            errors.append(f"{doc_id}, span {i}: invalid label {label!r}")

        key = (s, e, label)
        if key in seen:
            errors.append(f"{doc_id}: duplicate span {key}")
        seen.add(key)

        if label in label_counts:
            label_counts[label] += 1

        total_spans += 1

print("Documents in test set:", len(test_texts))
print("Documents in prediction.json:", len(checked))
print("Total predicted spans:", total_spans)
print("\nPer-label prediction counts:")
for label, count in label_counts.items():
    print(f"{label:10s}: {count}")

print("\nValidation errors:", len(errors))

if errors:
    for error in errors[:30]:
        print("ERROR:", error)
    raise ValueError("Validation failed. Do not submit this file yet.")

assert os.path.exists(OUTPUT_PATH)
assert os.path.getsize(OUTPUT_PATH) > 0

print("\nSUCCESS: prediction.json passed all structural checks.")
print("File:", OUTPUT_PATH)

In [ ]:
for doc_id in list(test_texts.keys())[:3]:
    text = test_texts[doc_id]

    print(f"\nDocument ID: {doc_id}")
    print("Text preview:", repr(text[:200]))

    for s, e, label in checked[doc_id][:10]:
        print(f"  [{s}, {e}, {label}] {text[s:e]!r}")

In [ ]:
import re
import pandas as pd

suspicious = []

for doc_id, spans in checked.items():
    text = test_texts[doc_id]

    for s, e, label in spans:
        entity = text[s:e]
        reasons = []

        if label == "PREC" and re.fullmatch(
            r"(THE )?(SUPREME COURT|HIGH COURT)( OF INDIA)?",
            entity.strip(), re.I
        ):
            reasons.append("court name predicted as PREC")

        if label == "JUDGE" and re.search(
            r"\b(AIR|SCC|SCR)\b", entity, re.I
        ):
            reasons.append("citation predicted as JUDGE")

        if label == "DATE" and re.fullmatch(r"\d{1,2}", entity.strip()):
            reasons.append("bare number predicted as DATE")

        if label == "CASENO" and entity.strip().endswith(
            ("ARIS", "ACT", "SCR")
        ):
            reasons.append("possible partial/overextended case number")

        if reasons:
            suspicious.append({
                "doc_id": doc_id,
                "label": label,
                "span": entity,
                "reasons": "; ".join(reasons)
            })

suspicious_df = pd.DataFrame(suspicious)

print("Suspicious spans:", len(suspicious_df))
display(suspicious_df.head(50))

In [ ]:
import re

# Start from the current best validated fusion.
date_filter_test = {
    doc_id: list(spans)
    for doc_id, spans in fused_repaired.items()
}

removed_dates = []

for doc_id, spans in date_filter_test.items():
    text = val_texts[doc_id]
    kept = []

    for s, e, label in spans:
        entity = text[s:e].strip()

        # Experimental rule: remove only DATE spans containing
        # one or two digits and nothing else.
        if label == "DATE" and re.fullmatch(r"\d{1,2}", entity):
            removed_dates.append((doc_id, s, e, entity))
            continue

        kept.append((s, e, label))

    date_filter_test[doc_id] = kept

filtered_f1, filtered_metrics = exact_span_scores(
    val_gold, date_filter_test
)

print("Baseline macro F1:", f"{fusion_repaired_f1:.6f}")
print("Filtered macro F1:", f"{filtered_f1:.6f}")
print("Change:", f"{filtered_f1 - fusion_repaired_f1:+.6f}")
print("DATE spans removed:", len(removed_dates))
print("DATE baseline:", fusion_repaired_metrics["DATE"])
print("DATE filtered:", filtered_metrics["DATE"])

In [ ]:
import os
import re
import json
import shutil

OUTPUT_PATH = "/kaggle/working/prediction.json"
BACKUP_PATH = "/kaggle/working/prediction_before_date_filter.json"

# Back up the current submission without overwriting an earlier backup.
if os.path.exists(OUTPUT_PATH) and not os.path.exists(BACKUP_PATH):
    shutil.copy2(OUTPUT_PATH, BACKUP_PATH)

# Apply the validated DATE rule to test predictions.
filtered_test_predictions = {}
removed_date_spans = []

for doc_id, spans in checked.items():
    text = test_texts[str(doc_id)]
    kept = []

    for s, e, label in spans:
        entity = text[int(s):int(e)].strip()

        if label == "DATE" and re.fullmatch(r"\d{1,2}", entity):
            removed_date_spans.append(
                (str(doc_id), int(s), int(e), entity)
            )
            continue

        kept.append([int(s), int(e), label])

    filtered_test_predictions[str(doc_id)] = kept

# Save updated prediction.json.
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(filtered_test_predictions, f, ensure_ascii=False)

print("Saved:", OUTPUT_PATH)
print("Bare-number DATE spans removed:", len(removed_date_spans))
print("Documents:", len(filtered_test_predictions))
print("Total remaining spans:", sum(
    len(spans) for spans in filtered_test_predictions.values()
))
print("Previous submission backup:", BACKUP_PATH if os.path.exists(BACKUP_PATH) else "not needed")

In [ ]:
# Weak-label diagnostic — read-only
import pandas as pd

WEAK_LABELS = [
    "RESP", "A.COUNSEL", "R.COUNSEL",
    "WIT", "APP", "PREC"
]

rows = []

for label in WEAK_LABELS:
    m = fusion_repaired_metrics[label]
    rows.append({
        "label": label,
        "gold": m["gold"],
        "predicted": m["pred"],
        "exact_matches": m["tp"],
        "precision": round(m["tp"] / m["pred"], 4) if m["pred"] else 0.0,
        "recall": round(m["tp"] / m["gold"], 4) if m["gold"] else 0.0,
        "F1": round(m["f1"], 4),
    })

display(pd.DataFrame(rows).sort_values("F1"))

In [ ]:
# Find existing validation prediction variables without changing anything
for name in sorted(globals()):
    if any(k in name.lower() for k in [
        "fusion", "champion_preds", "weighted_preds"
    ]):
        obj = globals()[name]
        if isinstance(obj, dict):
            print(
                f"{name}: {len(obj)} documents; "
                f"first value type={type(next(iter(obj.values()), None)).__name__}"
            )

In [ ]:
# Reconstruct the validated label-wise fusion.
# Champion supplies all labels except APP and JUDGE.
# Weighted model supplies APP and JUDGE.

REPLACE_LABELS = {"APP", "JUDGE"}

fusion_repaired = {}

for doc_id in val_texts:
    champion_spans = champion_preds.get(doc_id, [])
    weighted_spans = weighted_preds.get(doc_id, [])

    combined = [
        (s, e, label)
        for s, e, label in champion_spans
        if label not in REPLACE_LABELS
    ]

    combined += [
        (s, e, label)
        for s, e, label in weighted_spans
        if label in REPLACE_LABELS
    ]

    fusion_repaired[doc_id] = sorted(set(combined))

print("Validation documents:", len(fusion_repaired))
print("Total predicted spans:", sum(len(v) for v in fusion_repaired.values()))
print("Example:", next(iter(fusion_repaired.items())))

In [ ]:
# Read-only diagnostic: inspect exact matches, missed spans, and false positives.

WEAK_LABELS = [
    "RESP", "A.COUNSEL", "R.COUNSEL",
    "WIT", "APP", "PREC"
]

def normalize_span(item):
    # Supports either [start, end, label] or [start, end, label, confidence].
    return (int(item[0]), int(item[1]), item[2])

for label in WEAK_LABELS:
    print("\n" + "=" * 100)
    print(f"LABEL: {label}")

    shown_fn = 0
    shown_fp = 0

    for doc_id, text in val_texts.items():
        gold_spans = {
            normalize_span(x)
            for x in val_gold.get(doc_id, [])
            if x[2] == label
        }
        pred_spans = {
            normalize_span(x)
            for x in fusion_repaired.get(doc_id, [])
            if x[2] == label
        }

        missed = sorted(gold_spans - pred_spans)
        false_positives = sorted(pred_spans - gold_spans)

        for s, e, _ in missed:
            if shown_fn >= 5:
                break
            print(f"  MISSED [{s}:{e}]: {text[s:e]!r}")
            shown_fn += 1

        for s, e, _ in false_positives:
            if shown_fp >= 5:
                break
            print(f"  FALSE POS [{s}:{e}]: {text[s:e]!r}")
            shown_fp += 1

        if shown_fn >= 5 and shown_fp >= 5:
            break

    print(f"Examples shown: {shown_fn} missed, {shown_fp} false positives")

In [ ]:
# Diagnose predictions overlapping missed gold entities.
# Read-only: no model, checkpoint, or file changes.

WEAK_LABELS = [
    "RESP", "A.COUNSEL", "R.COUNSEL",
    "WIT", "APP"
]

for target_label in WEAK_LABELS:
    print("\n" + "=" * 90)
    print("TARGET LABEL:", target_label)

    shown = 0

    for doc_id, text in val_texts.items():
        gold_spans = [
            (int(x[0]), int(x[1]), x[2])
            for x in val_gold.get(doc_id, [])
            if x[2] == target_label
        ]

        pred_spans = [
            (int(x[0]), int(x[1]), x[2])
            for x in fusion_repaired.get(doc_id, [])
        ]

        pred_set = set(pred_spans)

        for gs, ge, glabel in gold_spans:
            if (gs, ge, glabel) in pred_set:
                continue

            overlapping = [
                (ps, pe, plabel, text[ps:pe])
                for ps, pe, plabel in pred_spans
                if max(gs, ps) < min(ge, pe)
            ]

            if shown < 8:
                print(f"\nGOLD: {text[gs:ge]!r} [{gs}:{ge}]")
                print("OVERLAPPING PREDICTIONS:", overlapping or "None")
                shown += 1

            if shown >= 8:
                break

        if shown >= 8:
            break

In [ ]:
# Inspect existing training/model objects without changing anything.

for name in [
    "LegalNERModel",
    "train_loader",
    "train_dataset",
    "train_model",
    "criterion",
    "optimizer",
    "BIO_LABELS",
    "ENTITY_LABELS",
    "phase1_ckpt",
    "weighted_model",
    "ner_model",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")
    else:
        print(f"{name}: not currently defined")

In [ ]:
import torch
from collections import Counter

print("Number of training batches:", len(train_loader))
print("Loss function:", criterion)
print("Optimizer:", type(optimizer).__name__)

batch = next(iter(train_loader))

print("\nBatch structure:")
print("Number of elements:", len(batch))

for i, item in enumerate(batch):
    if torch.is_tensor(item):
        print(
            f"batch[{i}]: shape={tuple(item.shape)}, "
            f"dtype={item.dtype}"
        )
    else:
        print(f"batch[{i}]: type={type(item).__name__}")

print("\nBIO labels:")
for i, label in enumerate(BIO_LABELS):
    print(i, label)

In [ ]:
from collections import Counter

token_counts = Counter()

for batch in train_loader:
    input_ids, attention_mask, labels = batch

    labels = labels.detach().cpu().reshape(-1).tolist()

    for label_id in labels:
        if label_id != -100:
            token_counts[BIO_LABELS[label_id]] += 1

print("Supervised token counts:")
for label in BIO_LABELS:
    print(f"{label:12s} {token_counts[label]:>8,}")

print("\nEntity token totals:")
for entity in ENTITY_LABELS:
    total = token_counts[f"B-{entity}"] + token_counts[f"I-{entity}"]
    print(f"{entity:12s} {total:>8,}")

In [ ]:
import inspect
import torch

print("Device:", device)
print("Model class:", LegalNERModel.__name__)
print("Model forward signature:", inspect.signature(LegalNERModel.forward))

print("\nTraining loader batch size:", train_loader.batch_size)
print("Training batches:", len(train_loader))

print("\nPhase 1 checkpoint keys:")
print(list(phase1_ckpt.keys()))

print("\nExisting model parameter count:")
print(f"{sum(p.numel() for p in ner_model.parameters()):,}")

In [ ]:
import inspect
from collections import Counter

# 1. Identify the existing window-building functions.
for name in [
    "make_unique_loss_windows",
    "make_contained_loss_windows",
    "collate_windows",
    "collate_ner_windows",
    "audit_alignment",
]:
    fn = globals().get(name)

    if callable(fn):
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(fn)[:5000])
        except (OSError, TypeError):
            print("Source unavailable:", fn)

# 2. Compare entity-token counts across the current training windows.
print("\n--- Training-window BIO counts ---")

counts = Counter()

for window in train_windows_v2:
    for tag_id in window["labels"]:
        tag_id = int(tag_id)

        if tag_id != -100:
            counts[BIO_LABELS[tag_id]] += 1

for label in [
    "APP", "RESP", "A.COUNSEL",
    "R.COUNSEL", "WIT", "PREC"
]:
    b_count = counts[f"B-{label}"]
    i_count = counts[f"I-{label}"]

    print(
        f"{label:12s} "
        f"B={b_count:5,d} "
        f"I={i_count:5,d} "
        f"total={b_count + i_count:6,d}"
    )

In [ ]:
from collections import Counter, defaultdict

WEAK_LABELS = {
    "RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "APP"
}

def audit_weak_entity_alignment(texts, gold_by_doc, fold_name):
    stats = Counter()
    examples = []
    overlap_examples = []

    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        entities = [
            (int(x[0]), int(x[1]), str(x[2]))
            for x in gold_by_doc[doc_id]
        ]

        # Check whether two annotations overlap in character space.
        ordered = sorted(entities, key=lambda x: (x[0], x[1]))

        for i, a in enumerate(ordered):
            for b in ordered[i + 1:]:
                if b[0] >= a[1]:
                    break

                stats["overlapping_entity_pairs"] += 1

                if len(overlap_examples) < 10:
                    overlap_examples.append((
                        doc_id,
                        text[a[0]:a[1]], a[2],
                        text[b[0]:b[1]], b[2]
                    ))

        # Check the weak-label spans against token boundaries.
        for s, e, label in entities:
            if label not in WEAK_LABELS:
                continue

            stats[f"{label}_entities"] += 1

            overlapping = [
                (i, a, b)
                for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]

            if not overlapping:
                stats[f"{label}_no_token"] += 1
                continue

            first_i, first_s, first_e = overlapping[0]
            last_i, last_s, last_e = overlapping[-1]

            exact_start = first_s == s
            exact_end = last_e == e

            stats[f"{label}_exact_boundary"] += int(
                exact_start and exact_end
            )
            stats[f"{label}_boundary_mismatch"] += int(
                not (exact_start and exact_end)
            )

            if not (exact_start and exact_end) and len(examples) < 15:
                examples.append((
                    doc_id, label, text[s:e],
                    (first_s, first_e), (last_s, last_e)
                ))

    print(f"\n=== {fold_name} ===")

    for label in sorted(WEAK_LABELS):
        n = stats[f"{label}_entities"]
        print(
            f"{label:12s} entities={n:4d}, "
            f"exact boundaries={stats[f'{label}_exact_boundary']:4d}, "
            f"mismatches={stats[f'{label}_boundary_mismatch']:4d}, "
            f"no token={stats[f'{label}_no_token']:3d}"
        )

    print("\nOverlapping gold annotation pairs:",
          stats["overlapping_entity_pairs"])

    print("\nBoundary mismatch examples:")
    for item in examples:
        print(item)

    print("\nOverlapping annotation examples:")
    for item in overlap_examples:
        print(item)


audit_weak_entity_alignment(train_texts, train_gold, "Training data")

In [ ]:
# Read-only tokenizer boundary diagnostic.
# Uses the document and entity examples from the audit.

doc_id = "115651329"
text = train_texts[doc_id]

enc = tokenizer.encode(text, add_special_tokens=False)
offsets = enc.offsets
tokens = enc.tokens

examples = [
    ("STATE OF HARYANA", 136, 153),
    ("Jagdish", 636, 644),
    ("Mange Ram", 694, 704),
    ("Jage Ram", 791, 800),
]

for entity_text, start, end in examples:
    print("\n" + "=" * 70)
    print("Gold text:", repr(text[start:end]))
    print("Expected:", repr(entity_text))
    print("Offsets:", start, end)

    overlapping = [
        (i, offsets[i][0], offsets[i][1], tokens[i])
        for i, (a, b) in enumerate(offsets)
        if b > a and a < end and b > start
    ]

    print("Overlapping tokens:")
    for item in overlapping:
        print(item)

    if overlapping:
        first = overlapping[0]
        last = overlapping[-1]
        print("First token starts at gold boundary:", first[1] == start)
        print("Last token ends at gold boundary:", last[2] == end)
        print("Combined token text:", repr(text[first[1]:last[2]]))

In [ ]:
# Verify BIO tags generated by make_unique_loss_windows.
# Read-only: does not change training data or model weights.

doc_id = "115651329"
text = train_texts[doc_id]
entities = train_gold[doc_id]

enc = tokenizer.encode(text, add_special_tokens=False)
offsets = enc.offsets
token_ids = enc.ids

# Reproduce the current tag-generation logic exactly.
tags = [label2id["O"]] * len(token_ids)

for entity in entities:
    start, end, label = int(entity[0]), int(entity[1]), str(entity[2])

    overlapping = [
        i for i, (a, b) in enumerate(offsets)
        if b > a and a < end and b > start
    ]

    if not overlapping:
        continue

    tags[overlapping[0]] = label2id[f"B-{label}"]

    for i in overlapping[1:]:
        tags[i] = label2id[f"I-{label}"]

# Display tokens around selected entities.
for entity_text in [
    "STATE OF HARYANA",
    "Jagdish",
    "Mange Ram",
    "Jage Ram",
]:
    # Find a gold entity matching the requested text after trimming whitespace.
    matches = [
        (int(x[0]), int(x[1]), str(x[2]))
        for x in entities
        if text[int(x[0]):int(x[1])].strip() == entity_text
    ]

    for start, end, label in matches[:2]:
        print("\n", "=" * 70)
        print("Entity:", repr(text[start:end]), "| label:", label)

        for i, (a, b) in enumerate(offsets):
            if b > a and a < end and b > start:
                print(
                    f"token={i:4d}",
                    f"offset=({a},{b})",
                    f"text={text[a:b]!r}",
                    f"BIO={BIO_LABELS[tags[i]]}"
                )

In [ ]:
# Read-only diagnostic: inspect raw token predictions on gold weak entities.
# Uses the current champion model and existing validation data.

import torch
from collections import Counter, defaultdict

WEAK_LABELS = [
    "RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "APP"
]

bio_id_to_label = {i: label for i, label in enumerate(BIO_LABELS)}

def inspect_gold_token_predictions(text, entities, max_entities_per_label=4):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    # Run the model over the document using the same window settings.
    logit_sum = torch.zeros((len(ids), len(BIO_LABELS)))
    token_count = torch.zeros(len(ids))

    ner_model.eval()

    with torch.no_grad():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = ner_model(input_ids, attention_mask)
            logits = output[0] if isinstance(output, tuple) else output
            logits = logits[0].detach().float().cpu()

            logit_sum[start:end] += logits[:end - start]
            token_count[start:end] += 1

            if end == len(ids):
                break

    averaged_logits = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    probs = torch.softmax(averaged_logits, dim=-1)
    pred_ids = averaged_logits.argmax(dim=-1).tolist()

    counts = defaultdict(Counter)
    shown = Counter()

    for entity in entities:
        s, e, label = int(entity[0]), int(entity[1]), str(entity[2])

        if label not in WEAK_LABELS:
            continue

        token_indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]

        if not token_indices:
            continue

        predicted_labels = [bio_id_to_label[pred_ids[i]] for i in token_indices]
        gold_bio = [f"B-{label}"] + [f"I-{label}"] * (len(token_indices) - 1)

        # Most likely class at the first token and average gold-class probability.
        first_i = token_indices[0]
        first_pred = bio_id_to_label[pred_ids[first_i]]
        gold_probs = [float(probs[i, BIO_LABELS.index(g)] ) for i, g in zip(token_indices, gold_bio)]

        counts[label]["entities"] += 1
        counts[label]["first_token_correct"] += int(first_pred == f"B-{label}")
        counts[label]["all_tokens_correct"] += int(predicted_labels == gold_bio)

        if shown[label] < max_entities_per_label:
            print(
                f"\n{label}: {text[s:e]!r}\n"
                f"  tokens: {[text[offsets[i][0]:offsets[i][1]] for i in token_indices]}\n"
                f"  gold BIO: {gold_bio}\n"
                f"  predicted BIO: {predicted_labels}\n"
                f"  first prediction: {first_pred}\n"
                f"  mean gold-tag probability: {sum(gold_probs)/len(gold_probs):.3f}"
            )
            shown[label] += 1

    return counts

weak_token_counts = defaultdict(Counter)

for doc_id, text in val_texts.items():
    doc_counts = inspect_gold_token_predictions(
        text, val_gold.get(doc_id, []), max_entities_per_label=2
    )

    for label, counter in doc_counts.items():
        weak_token_counts[label].update(counter)

print("\n\n=== SUMMARY ===")
for label in WEAK_LABELS:
    c = weak_token_counts[label]
    n = c["entities"]
    print(
        f"{label:12s} "
        f"entities={n:4d}, "
        f"first-token correct={c['first_token_correct']:4d}, "
        f"all-token correct={c['all_tokens_correct']:4d}"
    )

In [ ]:
from collections import Counter
import torch

token_counts = Counter()
batch_count = 0
total_supervised = 0

for batch in train_loader:
    input_ids, attention_mask, labels = batch
    batch_count += 1

    valid = labels != IGNORE_INDEX
    total_supervised += int(valid.sum())

    for label_id, count in zip(*torch.unique(
        labels[valid], return_counts=True
    )):
        token_counts[BIO_LABELS[int(label_id)]] += int(count)

print("Batches:", batch_count)
print("Supervised tokens:", f"{total_supervised:,}")
print("\nActual training-loader label counts:")

for label in BIO_LABELS:
    count = token_counts[label]
    pct = 100 * count / max(total_supervised, 1)
    print(f"{label:12s} {count:9,}  ({pct:6.3f}%)")

o_count = token_counts["O"]
entity_count = sum(
    count for label, count in token_counts.items() if label != "O"
)

print("\nO fraction:", f"{o_count / max(total_supervised, 1):.2%}")
print("Entity-token fraction:",
      f"{entity_count / max(total_supervised, 1):.2%}")

In [ ]:
import inspect
import torch
import torch.nn as nn

print("Current criterion:")
print(criterion)

print("\nCurrent optimizer:")
print(type(optimizer).__name__)
print("Learning rates:", sorted({
    group["lr"] for group in optimizer.param_groups
}))
print("Weight decay:", sorted({
    group.get("weight_decay", 0)
    for group in optimizer.param_groups
}))

print("\nCriterion details:")
if isinstance(criterion, nn.CrossEntropyLoss):
    print("ignore_index:", criterion.ignore_index)
    print("reduction:", criterion.reduction)
    print("class weights:", criterion.weight)
else:
    print("Criterion is not CrossEntropyLoss")

print("\nTraining function available:", callable(globals().get("train_model")))
print("Train loader available:", "train_loader" in globals())
print("Validation texts:", len(val_texts))
print("Validation documents:", len(val_gold))

In [ ]:
import torch

print("=== Model / optimizer audit ===")

print("Champion model parameters:",
      sum(p.numel() for p in ner_model.parameters()))

print("Weighted model parameters:",
      sum(p.numel() for p in weighted_model.parameters()))

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

champion_param_ids = {id(p) for p in ner_model.parameters()}
weighted_param_ids = {id(p) for p in weighted_model.parameters()}

print("Optimizer matches champion:",
      optimizer_param_ids == champion_param_ids)

print("Optimizer matches weighted model:",
      optimizer_param_ids == weighted_param_ids)

print("\n=== Class weights ===")
for label, weight in zip(BIO_LABELS, criterion.weight.detach().cpu().tolist()):
    print(f"{label:12s} {weight:.2f}")

print("\n=== Relevant notebook functions ===")
import inspect

for name in [
    "train_one_epoch",
    "train_epoch",
    "run_training",
    "evaluate_model",
    "evaluate_ner",
]:
    fn = globals().get(name)
    if callable(fn):
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(fn)[:4000])
        except (OSError, TypeError):
            print("Source unavailable")

In [ ]:

import os
import torch

print("=== Data and model availability ===")
for name in [
    "LegalNERModel", "GPT2LMHeadModel", "GPT2Config",
    "phase1_ckpt", "train_loader", "val_texts", "val_gold",
    "BIO_LABELS", "tokenizer", "device"
]:
    print(f"{name}: {name in globals()}")

print("\n=== Training configuration ===")
print("Batches:", len(train_loader))
print("BIO labels:", len(BIO_LABELS))
print("Device:", device)

print("\n=== Checkpoint safety ===")
for path in [
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt",
    "/kaggle/working/phase2_ner_weighted/best_weighted_model.pt",
    "/kaggle/working/prediction.json",
    "/kaggle/working/prediction_champion.json",
]:
    print(f"{os.path.exists(path)}  {path}")

print("\n=== Data batch sanity ===")
batch = next(iter(train_loader))
for name, tensor in zip(
    ["input_ids", "attention_mask", "labels"], batch
):
    print(name, tuple(tensor.shape), tensor.dtype)

print("\n=== Model forward signature ===")
import inspect
print(inspect.signature(LegalNERModel.forward))


In [ ]:

import os
import torch
from torch.optim import AdamW
from transformers import GPT2Config, GPT2LMHeadModel

# Independent experiment directory
EXP_E_DIR = "/kaggle/working/phase2_ner_experiment_E"
os.makedirs(EXP_E_DIR, exist_ok=True)

# Rebuild the Phase 1 backbone from the supplied checkpoint
cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

phase1_fresh_E = GPT2LMHeadModel(gpt2_config)
phase1_fresh_E.load_state_dict(
    phase1_ckpt["model_state_dict"],
    strict=True
)

# New NER model; champion and weighted models remain untouched
model_E = LegalNERModel(
    pretrained_lm=phase1_fresh_E,
    num_labels=len(BIO_LABELS)
).to(device)

# Label-specific weights: emphasize weak classes, especially entity starts
weak_labels = {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "APP"}

class_weights_E = torch.ones(
    len(BIO_LABELS), dtype=torch.float32, device=device
)

for i, label in enumerate(BIO_LABELS):
    if label == "O":
        class_weights_E[i] = 0.5
    elif label.startswith("B-") and label[2:] in weak_labels:
        class_weights_E[i] = 8.0
    elif label.startswith("I-") and label[2:] in weak_labels:
        class_weights_E[i] = 5.0
    else:
        class_weights_E[i] = 1.0

criterion_E = torch.nn.CrossEntropyLoss(
    weight=class_weights_E,
    ignore_index=-100
)

optimizer_E = AdamW(
    model_E.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

# Verify that this optimizer belongs only to model_E
optimizer_E_ids = {
    id(p)
    for group in optimizer_E.param_groups
    for p in group["params"]
}
model_E_ids = {id(p) for p in model_E.parameters()}

assert optimizer_E_ids == model_E_ids
assert model_E is not ner_model
assert model_E is not weighted_model

print("Experiment E initialized successfully.")
print("Model parameters:", sum(p.numel() for p in model_E.parameters()))
print("Optimizer matches model_E:", optimizer_E_ids == model_E_ids)
print("Learning rate:", optimizer_E.param_groups[0]["lr"])
print("Loss weights:")
for label, weight in zip(BIO_LABELS, class_weights_E.detach().cpu().tolist()):
    print(f"{label:12s} {weight:.1f}")
print("Checkpoint directory:", EXP_E_DIR)


In [ ]:

import os
import torch

EPOCHS_E = 3
best_loss_E = float("inf")
history_E = []

for epoch in range(1, EPOCHS_E + 1):
    model_E.train()
    total_loss = 0.0
    total_batches = 0

    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer_E.zero_grad(set_to_none=True)

        output = model_E(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(output, dict):
            logits = output["logits"]
        elif hasattr(output, "logits"):
            logits = output.logits
        elif isinstance(output, (tuple, list)):
            logits = output[0]
        else:
            logits = output

        loss = criterion_E(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Experiment E non-finite loss at epoch {epoch}: "
                f"{loss.item()}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_E.parameters(), 1.0)
        optimizer_E.step()

        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    history_E.append({"epoch": epoch, "train_loss": avg_loss})

    # Save the lowest-training-loss state as a fallback.
    # Validation F1 will determine the actual best checkpoint later.
    if avg_loss < best_loss_E:
        best_loss_E = avg_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model_E.state_dict(),
                "optimizer_state_dict": optimizer_E.state_dict(),
                "train_loss": avg_loss,
                "bio_labels": BIO_LABELS,
            },
            os.path.join(EXP_E_DIR, "lowest_train_loss.pt")
        )

    print(
        f"Experiment E | Epoch {epoch}/{EPOCHS_E} | "
        f"train loss = {avg_loss:.4f}"
    )

print("\nTraining complete.")
print("History:", history_E)
print("Fallback checkpoint:", os.path.join(EXP_E_DIR, "lowest_train_loss.pt"))


In [ ]:

import inspect

print("=== Existing validation / decoding functions ===")
for name, obj in list(globals().items()):
    if callable(obj) and any(
        key in name.lower()
        for key in ["predict", "decode", "viterbi", "score", "evaluate", "f1"]
    ):
        try:
            sig = inspect.signature(obj)
        except Exception:
            sig = "(signature unavailable)"
        print(f"{name}{sig}")

print("\n=== Cached validation predictions ===")
for name in [
    "confidence_diag", "champion_preds", "weighted_diag",
    "weighted_preds", "fusion_repaired", "val_texts", "val_gold"
]:
    obj = globals().get(name)
    if obj is None:
        print(f"{name}: unavailable")
    else:
        try:
            print(f"{name}: {len(obj)} items")
        except Exception:
            print(f"{name}: available")

print("\n=== Experiment E checkpoint ===")
import os
print(os.path.exists(
    "/kaggle/working/phase2_ner_experiment_E/lowest_train_loss.pt"
))


In [ ]:

import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU total: {total_bytes / 1024**3:.2f} GiB")
    print(f"GPU free:  {free_bytes / 1024**3:.2f} GiB")

print("Experiment E on device:", next(model_E.parameters()).device)
print("Experiment E in eval mode:", not model_E.training)


In [ ]:

import inspect

for name in [
    "predict_document_spans",
    "predict_doc",
    "bio_viterbi_decode",
    "decode_bio_to_spans",
    "trim_predicted_spans",
]:
    fn = globals().get(name)
    if callable(fn):
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(fn)[:3500])
        except (OSError, TypeError):
            print("Source unavailable")


In [ ]:

import gc
import torch

# Keep Experiment E on GPU for inference.
model_E.eval()

# Move other in-memory models off GPU to free VRAM.
for name in ["ner_model", "weighted_model"]:
    obj = globals().get(name)
    if obj is not None and obj is not model_E:
        obj.to("cpu")

gc.collect()
torch.cuda.empty_cache()

free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU free after cleanup: {free_bytes / 1024**3:.2f} GiB")
print("Experiment E device:", next(model_E.parameters()).device)
print("Experiment E eval mode:", not model_E.training)


In [ ]:

import numpy as np
import torch

@torch.inference_mode()
def predict_doc_E(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids, offsets = enc.ids, enc.offsets

    if not token_ids:
        return []

    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float32
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    model_E.eval()

    for start in range(0, len(token_ids), WINDOW_STRIDE):
        end = min(start + MAX_LEN, len(token_ids))

        ids = torch.tensor(
            [token_ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(ids)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            output = model_E(input_ids=ids, attention_mask=mask)

        if isinstance(output, dict):
            logits = output["logits"]
        elif hasattr(output, "logits"):
            logits = output.logits
        elif isinstance(output, (tuple, list)):
            logits = output[0]
        else:
            logits = output

        logits = logits[0].float().cpu().numpy()
        logits_sum[start:end] += logits[:end-start]
        counts[start:end] += 1

        del ids, mask, output, logits
        if end == len(token_ids):
            break

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    tag_ids = bio_viterbi_decode(mean_logits)

    spans = decode_bio_to_spans(
        offsets, tag_ids, BIO_LABELS, text
    )
    return trim_predicted_spans(text, spans)


predictions_E = {}
for i, text in enumerate(val_texts):
    predictions_E[i] = predict_doc_E(text)

print("Experiment E documents predicted:", len(predictions_E))
print("Example:", next(iter(predictions_E.items())))
print("Total predicted spans:", sum(len(x) for x in predictions_E.values()))


In [ ]:

from collections import Counter

gold_counts_E = Counter()
pred_counts_E = Counter()
gold_total_E = 0
pred_total_E = 0

for doc_id, entities in val_gold.items():
    for entity in entities:
        label = str(entity[2])
        gold_counts_E[label] += 1
        gold_total_E += 1

for spans in predictions_E.values():
    for span in spans:
        label = str(span[2])
        pred_counts_E[label] += 1
        pred_total_E += 1

print("Gold spans:", gold_total_E)
print("Predicted spans:", pred_total_E)
print("\nPer-label counts:")
for label in [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]:
    print(
        f"{label:12s} gold={gold_counts_E[label]:4d} "
        f"predicted={pred_counts_E[label]:4d}"
    )

print("\nFirst predicted documents:")
for doc_id in list(predictions_E)[:10]:
    print(doc_id, predictions_E[doc_id][:10])


In [ ]:

import torch
import numpy as np
from collections import Counter

# Use an actual key from the validation dictionary.
first_doc_id = next(iter(val_texts))
test_text_E = val_texts[first_doc_id]

print("Validation document ID:", first_doc_id)
print("Document characters:", len(test_text_E))

enc_E = tokenizer.encode(test_text_E, add_special_tokens=False)
print("Document tokens:", len(enc_E.ids))
print("STRIDE:", STRIDE, "| WINDOW_STRIDE:", WINDOW_STRIDE, "| MAX_LEN:", MAX_LEN)

@torch.inference_mode()
def raw_logits_for_model_E(model, text, stride):
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids = enc.ids
    if not token_ids:
        return np.zeros((0, len(BIO_LABELS)), dtype=np.float32)

    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float32
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    was_training = model.training
    model.eval()
    try:
        for start in range(0, len(token_ids), stride):
            end = min(start + MAX_LEN, len(token_ids))
            x = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(x)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output = model(input_ids=x, attention_mask=mask)

            if isinstance(output, dict):
                logits = output["logits"]
            elif hasattr(output, "logits"):
                logits = output.logits
            elif isinstance(output, (tuple, list)):
                logits = output[0]
            else:
                logits = output

            logits_sum[start:end] += logits[0].float().cpu().numpy()
            counts[start:end] += 1

            del x, mask, output, logits
            if end == len(token_ids):
                break
    finally:
        model.train(was_training)

    return logits_sum / np.maximum(counts[:, None], 1)

# Inspect Experiment E's raw tag predictions.
logits_E = raw_logits_for_model_E(model_E, test_text_E, WINDOW_STRIDE)

tag_counts_E = Counter(
    BIO_LABELS[i] for i in logits_E.argmax(axis=-1)
)
print("\nExperiment E raw token tags:")
print(tag_counts_E.most_common(15))

print("\nExperiment E mean max logit:",
      float(logits_E.max(axis=-1).mean()))

# Move Experiment E off GPU before loading the champion onto it.
model_E.to("cpu")
torch.cuda.empty_cache()

ner_model.to(device)
logits_champion = raw_logits_for_model_E(
    ner_model, test_text_E, STRIDE
)

tag_counts_champion = Counter(
    BIO_LABELS[i] for i in logits_champion.argmax(axis=-1)
)
print("\nChampion raw token tags:")
print(tag_counts_champion.most_common(15))

print("\nChampion mean max logit:",
      float(logits_champion.max(axis=-1).mean()))

# Return Experiment E to GPU for consistency with the notebook.
ner_model.to("cpu")
model_E.to(device)
torch.cuda.empty_cache()


In [ ]:

import numpy as np
from collections import Counter

def bio_transition_report(logits, labels):
    raw_ids = logits.argmax(axis=-1)
    raw_tags = [labels[i] for i in raw_ids]

    invalid_starts = 0
    invalid_transitions = 0

    for i, tag in enumerate(raw_tags):
        if tag.startswith("I-"):
            label = tag[2:]
            if i == 0:
                invalid_starts += 1
            else:
                prev = raw_tags[i - 1]
                if prev not in (f"B-{label}", f"I-{label}"):
                    invalid_transitions += 1

    legal_ids = bio_viterbi_decode(logits)
    legal_tags = [labels[i] for i in legal_ids]

    raw_counts = Counter(raw_tags)
    legal_counts = Counter(legal_tags)

    print("Raw non-O tokens:", sum(v for k, v in raw_counts.items() if k != "O"))
    print("Raw invalid I-starts:", invalid_starts)
    print("Raw invalid I-transitions:", invalid_transitions)
    print("Viterbi non-O tokens:", sum(v for k, v in legal_counts.items() if k != "O"))

    print("\nRaw tags:", raw_counts.most_common(12))
    print("Viterbi tags:", legal_counts.most_common(12))

print("=== Experiment E ===")
bio_transition_report(logits_E, BIO_LABELS)

print("\n=== Champion ===")
bio_transition_report(logits_champion, BIO_LABELS)


In [ ]:

# Compare decoding paths on the same document.
# Uses the logits_E and logits_champion arrays already in memory.

def compare_decoding_paths(text, logits, model_name):
    enc = tokenizer.encode(text, add_special_tokens=False)
    offsets = enc.offsets

    raw_ids = logits.argmax(axis=-1)
    viterbi_ids = bio_viterbi_decode(logits)

    raw_spans = decode_bio_to_spans(
        offsets, raw_ids, BIO_LABELS, text
    )
    viterbi_spans = decode_bio_to_spans(
        offsets, viterbi_ids, BIO_LABELS, text
    )

    print(f"\n{model_name}")
    print("Raw argmax spans:", len(raw_spans))
    print("Viterbi spans:", len(viterbi_spans))
    print("Raw examples:", raw_spans[:10])
    print("Viterbi examples:", viterbi_spans[:10])

compare_decoding_paths(test_text_E, logits_E, "Experiment E")
compare_decoding_paths(test_text_E, logits_champion, "Champion")

print("\nPreviously generated Experiment E spans:",
      len(predictions_E.get(first_doc_id, [])))
print("Validation document key:", first_doc_id)


In [ ]:

import gc
import numpy as np
import torch

def predict_doc_E_consistent(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    token_ids, offsets = enc.ids, enc.offsets

    if not token_ids:
        return []

    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float32
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    model_E.eval()

    with torch.inference_mode():
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_LEN, len(token_ids))
            x = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(x)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output = model_E(input_ids=x, attention_mask=mask)

            if isinstance(output, dict):
                logits = output["logits"]
            elif hasattr(output, "logits"):
                logits = output.logits
            elif isinstance(output, (tuple, list)):
                logits = output[0]
            else:
                logits = output

            logits_sum[start:end] += logits[0].float().cpu().numpy()
            counts[start:end] += 1

            del x, mask, output, logits

            if end == len(token_ids):
                break

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    tag_ids = bio_viterbi_decode(mean_logits)

    spans = decode_bio_to_spans(
        offsets, tag_ids, BIO_LABELS, text
    )
    return trim_predicted_spans(text, spans)


predictions_E_consistent = {}
for doc_id, text in val_texts.items():
    predictions_E_consistent[doc_id] = predict_doc_E_consistent(text)

print("Documents:", len(predictions_E_consistent))
print("Total Experiment E spans:", sum(
    len(spans) for spans in predictions_E_consistent.values()
))
print("First document:", first_doc_id)
print("Spans for first document:",
      len(predictions_E_consistent[first_doc_id]))
print(predictions_E_consistent[first_doc_id][:10])

gc.collect()
torch.cuda.empty_cache()


In [ ]:

import numpy as np

# Ensure both prediction dictionaries use the same document IDs.
print("Validation documents:", len(val_texts))
print("Gold documents:", len(val_gold))
print("Experiment E documents:", len(predictions_E_consistent))
print("Champion documents:", len(champion_preds))

# Existing scorer expects gold and predictions as span triples.
scores_E = exact_span_scores(val_gold, predictions_E_consistent)
scores_champion = exact_span_scores(val_gold, champion_preds)

print("\n=== Experiment E exact-span scores ===")
print(scores_E)

print("\n=== Champion exact-span scores ===")
print(scores_champion)

print("\n=== Macro-F1 ===")
print("Experiment E:", exact_span_macro_f1(val_texts, val_gold))


In [ ]:

from collections import defaultdict

ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels):
    metrics = {}

    for label in labels:
        gold_set = set()
        pred_set = set()

        for doc_id, entities in gold_by_doc.items():
            for entity in entities:
                s, e, lab = int(entity[0]), int(entity[1]), str(entity[2])
                if lab == label:
                    gold_set.add((str(doc_id), s, e, lab))

        for doc_id, spans in pred_by_doc.items():
            for span in spans:
                s, e, lab = int(span[0]), int(span[1]), str(span[2])
                if lab == label:
                    pred_set.add((str(doc_id), s, e, lab))

        tp = len(gold_set & pred_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        metrics[label] = {
            "gold": len(gold_set),
            "pred": len(pred_set),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    metrics["MACRO_F1"] = sum(
        metrics[label]["f1"] for label in labels
    ) / len(labels)

    return metrics


metrics_E = calculate_exact_span_metrics(
    val_gold, predictions_E_consistent, ENTITY_LABELS
)
metrics_champion = calculate_exact_span_metrics(
    val_gold, champion_preds, ENTITY_LABELS
)

print(f"{'Label':12s} {'Gold':>6s} {'E Pred':>7s} {'E F1':>8s} "
      f"{'Champ Pred':>11s} {'Champ F1':>9s}")

for label in ENTITY_LABELS:
    e = metrics_E[label]
    c = metrics_champion[label]
    print(
        f"{label:12s} {e['gold']:6d} {e['pred']:7d} "
        f"{e['f1']:8.4f} {c['pred']:11d} {c['f1']:9.4f}"
    )

print("\nExperiment E macro-F1:", round(metrics_E["MACRO_F1"], 6))
print("Champion macro-F1:", round(metrics_champion["MACRO_F1"], 6))


In [ ]:

from collections import Counter

WEAK_LABELS = {"RESP", "A.COUNSEL", "R.COUNSEL", "WIT", "APP"}

def audit_entity_spans(texts, gold_by_doc, split_name):
    counts = Counter()
    examples = {label: [] for label in WEAK_LABELS}

    for doc_id, entities in gold_by_doc.items():
        text = texts[doc_id]

        for entity in entities:
            start, end, label = int(entity[0]), int(entity[1]), str(entity[2])
            if label not in WEAK_LABELS:
                continue

            counts[(label, "total")] += 1

            if not (0 <= start < end <= len(text)):
                counts[(label, "invalid_offsets")] += 1
                continue

            surface = text[start:end]
            if surface == surface.strip():
                counts[(label, "no_outer_whitespace")] += 1
            else:
                counts[(label, "outer_whitespace")] += 1

            if surface.strip():
                counts[(label, "nonempty")] += 1

            enc = tokenizer.encode(text, add_special_tokens=False)
            overlapping = [
                (i, a, b)
                for i, (a, b) in enumerate(enc.offsets)
                if b > start and a < end and b > a
            ]

            if overlapping:
                counts[(label, "has_overlapping_tokens")] += 1

                # Are all overlapping tokens contained in the annotation?
                if all(a >= start and b <= end for _, a, b in overlapping):
                    counts[(label, "fully_contained_tokens")] += 1
                else:
                    counts[(label, "partial_token_overlap")] += 1
            else:
                counts[(label, "no_overlapping_tokens")] += 1

            if len(examples[label]) < 4:
                examples[label].append({
                    "doc_id": doc_id,
                    "span": (start, end),
                    "surface_repr": repr(surface),
                    "token_overlaps": overlapping[:8],
                })

    print(f"\n=== {split_name} ===")
    for label in sorted(WEAK_LABELS):
        total = counts[(label, "total")]
        print(f"\n{label}: {total} gold spans")
        for key in [
            "invalid_offsets", "outer_whitespace",
            "no_overlapping_tokens", "partial_token_overlap",
            "fully_contained_tokens"
        ]:
            print(f"  {key}: {counts[(label, key)]}")

        for ex in examples[label]:
            print(" ", ex)

    return counts, examples


train_texts_E = {}
# Reconstruct the training text dictionary from the existing fold-1 JSON.
import json

with open(
    "/kaggle/working/phase2_lner/Phase_2_LNER/Train/fold1-text.json",
    "r", encoding="utf-8"
) as f:
    train_texts_E = json.load(f)

with open(
    "/kaggle/working/phase2_lner/Phase_2_LNER/Train/fold1-gold.json",
    "r", encoding="utf-8"
) as f:
    train_gold_E = json.load(f)

train_alignment_counts, train_alignment_examples = audit_entity_spans(
    train_texts_E, train_gold_E, "Fold 1 training"
)

val_alignment_counts, val_alignment_examples = audit_entity_spans(
    val_texts, val_gold, "Fold 2 validation"
)


In [ ]:

import inspect

print("=== Current training-label builder ===")
print(inspect.getsource(make_unique_loss_windows))

print("\n=== Relevant constants ===")
for name in [
    "MAX_LEN", "STRIDE", "WINDOW_STRIDE",
    "O_LABEL_ID", "LABEL2ID", "BIO_LABELS"
]:
    if name in globals():
        print(f"{name} =", globals()[name])

print("\n=== Tokenizer offset examples ===")
doc_id = "115651329"
text = train_texts_E[doc_id]
enc = tokenizer.encode(text, add_special_tokens=False)

for start, end, label, entity_text in [
    (792, 800, "APP", "Jage Ram"),
    (637, 644, "WIT", "Jagdish"),
    (137, 153, "RESP", "STATE OF HARYANA"),
]:
    print(f"\nGold: {label} [{start}:{end}] {text[start:end]!r}")
    for i, (a, b) in enumerate(enc.offsets):
        if a < end and b > start and b > a:
            print(
                f"token={i:4d}, offset=({a},{b}), "
                f"surface={text[a:b]!r}, "
                f"contained={a >= start and b <= end}"
            )


In [ ]:

from collections import Counter

def audit_entity_conflicts(texts, gold_by_doc, split_name):
    stats = Counter()
    examples = []

    for doc_id, entities in gold_by_doc.items():
        text = texts[doc_id]
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        # Count overlapping gold spans.
        normalized = []
        for ent in entities:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            normalized.append((s, e, label))

        for i, a in enumerate(normalized):
            for b in normalized[i + 1:]:
                if max(a[0], b[0]) < min(a[1], b[1]):
                    stats["overlapping_gold_pairs"] += 1
                    if len(examples) < 12:
                        examples.append({
                            "doc_id": doc_id,
                            "entity_a": (a[0], a[1], a[2], text[a[0]:a[1]]),
                            "entity_b": (b[0], b[1], b[2], text[b[0]:b[1]]),
                        })

        # Count tokens overlapped by multiple entities.
        for token_idx, (a, b) in enumerate(offsets):
            if b <= a:
                continue
            hits = [
                ent for ent in normalized
                if a < ent[1] and b > ent[0]
            ]
            if len(hits) > 1:
                stats["tokens_overlapped_by_multiple_entities"] += 1
                if len(examples) < 12:
                    examples.append({
                        "doc_id": doc_id,
                        "token": (token_idx, a, b, text[a:b]),
                        "entities": hits,
                    })

    print(f"\n=== {split_name} ===")
    print("Overlapping gold entity pairs:", stats["overlapping_gold_pairs"])
    print("Tokens overlapped by multiple entities:",
          stats["tokens_overlapped_by_multiple_entities"])
    for ex in examples:
        print(ex)

    return stats, examples

conflicts_train, examples_train = audit_entity_conflicts(
    train_texts_E, train_gold_E, "Fold 1"
)
conflicts_val, examples_val = audit_entity_conflicts(
    val_texts, val_gold, "Fold 2"
)


In [ ]:

from collections import Counter

def compare_token_label_strategies(texts, gold_by_doc, split_name):
    stats = Counter()
    examples = []

    for doc_id, entities in gold_by_doc.items():
        text = texts[doc_id]
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets

        for ent in entities:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            overlaps = [
                (i, a, b) for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]
            contained = [
                (i, a, b) for i, a, b in overlaps
                if a >= s and b <= e
            ]

            stats["entities"] += 1
            stats["overlap_tokens"] += len(overlaps)
            stats["contained_tokens"] += len(contained)

            if overlaps and not contained:
                stats["entities_without_contained_tokens"] += 1

            if overlaps and len(contained) < len(overlaps):
                stats["entities_with_boundary_tokens"] += 1
                if len(examples) < 10:
                    examples.append({
                        "doc_id": doc_id,
                        "label": label,
                        "gold": (s, e, text[s:e]),
                        "overlap_tokens": [
                            (i, a, b, text[a:b]) for i, a, b in overlaps
                        ],
                        "contained_tokens": [
                            (i, a, b, text[a:b]) for i, a, b in contained
                        ],
                    })

    print(f"\n=== {split_name} ===")
    print(dict(stats))
    for ex in examples:
        print(ex)

    return stats, examples


boundary_train_stats, boundary_train_examples = compare_token_label_strategies(
    train_texts_E, train_gold_E, "Fold 1"
)
boundary_val_stats, boundary_val_examples = compare_token_label_strategies(
    val_texts, val_gold, "Fold 2"
)


In [ ]:

from collections import Counter

def audit_weak_bio_coverage(texts, gold_by_doc, split_name):
    stats = Counter()
    examples = []

    for doc_id, entities in gold_by_doc.items():
        text = texts[doc_id]
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets
        tags = [label2id["O"]] * len(offsets)

        # Reproduce the current overlap-based label assignment.
        for ent in entities:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            overlapping = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]
            if not overlapping:
                continue
            tags[overlapping[0]] = label2id[f"B-{label}"]
            for i in overlapping[1:]:
                tags[i] = label2id[f"I-{label}"]

        for ent in entities:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            if label not in WEAK_LABELS:
                continue

            indices = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]

            stats[(label, "entities")] += 1
            if not indices:
                stats[(label, "no_tokens")] += 1
                continue

            expected = [f"B-{label}"] + [f"I-{label}"] * (len(indices) - 1)
            actual = [BIO_LABELS[tags[i]] for i in indices]

            if actual == expected:
                stats[(label, "correct_bio_sequence")] += 1
            else:
                stats[(label, "bio_sequence_mismatch")] += 1
                if len(examples) < 12:
                    examples.append({
                        "doc_id": doc_id,
                        "label": label,
                        "gold": text[s:e],
                        "token_indices": indices,
                        "expected": expected,
                        "actual": actual,
                    })

            # Count window boundaries crossed by this entity.
            window_ids = {i // WINDOW_STRIDE for i in indices}
            if len(window_ids) > 1:
                stats[(label, "crosses_window_boundary")] += 1

    print(f"\n=== {split_name} ===")
    for label in sorted(WEAK_LABELS):
        print(
            label,
            "entities =", stats[(label, "entities")],
            "correct BIO =", stats[(label, "correct_bio_sequence")],
            "mismatch =", stats[(label, "bio_sequence_mismatch")],
            "cross-window =", stats[(label, "crosses_window_boundary")],
        )

    print("\nMismatch examples:")
    for ex in examples:
        print(ex)

    return stats, examples


bio_train_stats, bio_train_examples = audit_weak_bio_coverage(
    train_texts_E, train_gold_E, "Fold 1"
)
bio_val_stats, bio_val_examples = audit_weak_bio_coverage(
    val_texts, val_gold, "Fold 2"
)


In [ ]:
print("Text/gold-related variables:")
for name, value in list(globals().items()):
    if any(k in name.lower() for k in ["fold1", "fold2", "val_text", "val_gold", "train_text", "train_gold"]):
        try:
            print(f"{name}: {type(value).__name__}, len={len(value)}")
        except Exception:
            print(f"{name}: {type(value).__name__}")

In [ ]:
weak_window_audit_fold1 = audit_actual_window_masks(
    train_text_1, train_gold_1, "Fold 1"
)

weak_window_audit_fold2 = audit_actual_window_masks(
    val_texts, val_gold, "Fold 2"
)

In [ ]:
from collections import defaultdict

weak_examples = defaultdict(list)

for split_name, texts, gold in [
    ("Fold 1", train_text_1, train_gold_1),
    ("Fold 2", val_texts, val_gold),
]:
    for doc_id, entities in gold.items():
        text = texts[doc_id]
        for ent in entities:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            if label in WEAK_LABELS and len(weak_examples[(split_name, label)]) < 8:
                left = max(0, s - 100)
                right = min(len(text), e + 100)
                weak_examples[(split_name, label)].append({
                    "doc_id": doc_id,
                    "entity": text[s:e],
                    "context": text[left:right].replace("\n", " "),
                })

for label in sorted(WEAK_LABELS):
    print(f"\n{'='*24} {label} {'='*24}")
    for split_name in ["Fold 1", "Fold 2"]:
        print(f"\n-- {split_name} --")
        for x in weak_examples[(split_name, label)][:4]:
            print(f"Entity: {x['entity']!r}")
            print(f"Context: {x['context']}\n")

In [ ]:
import re
from collections import defaultdict

# Keep all existing champion artifacts untouched.
assert "val_texts" in globals() and "val_gold" in globals()
assert "champion_preds" in globals()

RULE_LABELS = {"APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}

def normalize_span_set(predictions):
    return {
        doc_id: set(
            (int(s), int(e), str(label))
            for s, e, label, *rest in spans
            if int(s) < int(e)
        )
        for doc_id, spans in predictions.items()
    }

def gold_to_span_set(gold_by_doc):
    result = {}
    for doc_id, entities in gold_by_doc.items():
        result[doc_id] = set(
            (int(ent[0]), int(ent[1]), str(ent[2]))
            for ent in entities
        )
    return result

def score_span_predictions(gold_by_doc, predictions, labels):
    gold = gold_to_span_set(gold_by_doc)
    pred = normalize_span_set(predictions)

    rows = {}
    f1_values = []

    for label in labels:
        tp = fp = fn = 0
        for doc_id in gold:
            g = {x for x in gold[doc_id] if x[2] == label}
            p = {x for x in pred.get(doc_id, set()) if x[2] == label}
            tp += len(g & p)
            fp += len(p - g)
            fn += len(g - p)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        rows[label] = {
            "gold": tp + fn,
            "pred": tp + fp,
            "TP": tp,
            "precision": precision,
            "recall": recall,
            "F1": f1,
        }
        f1_values.append(f1)

    return rows, sum(f1_values) / len(f1_values) if f1_values else 0.0

# Baseline: champion predictions with the exact-span scorer.
baseline_rows, baseline_macro = score_span_predictions(
    val_gold, champion_preds, RULE_LABELS
)

print(f"Champion macro-F1 on these 5 labels: {baseline_macro:.4f}")
for label, row in baseline_rows.items():
    print(
        f"{label:12s} P={row['precision']:.3f} "
        f"R={row['recall']:.3f} F1={row['F1']:.3f} "
        f"pred={row['pred']} gold={row['gold']}"
    )

In [ ]:
def add_candidate(store, text, start, end, label, source):
    start, end = int(start), int(end)

    # Trim surrounding whitespace and common caption punctuation.
    while start < end and (text[start].isspace() or text[start] in ".,:;"):
        start += 1
    while end > start and (text[end - 1].isspace() or text[end - 1] in ".,:;"):
        end -= 1

    if start < end:
        store.append((start, end, label, source))


def caption_rule_candidates(text):
    candidates = []

    # Typical caption: "X ... Appellant(s) Versus Y ... Respondent(s)"
    # Keep patterns deliberately narrow for the first test.
    role_patterns = [
        (
            "APP",
            re.compile(
                r"(?P<name>[A-Z][A-Z0-9&.,'()@/\- ]{1,100}?)"
                r"\s*(?:…|\.\.\.)?\s*"
                r"(?:APPELLANT(?:S)?|PETITIONER(?:S)?)\b",
                re.I
            )
        ),
        (
            "RESP",
            re.compile(
                r"(?P<name>[A-Z][A-Z0-9&.,'()@/\- ]{1,100}?)"
                r"\s*(?:…|\.\.\.)?\s*"
                r"(?:RESPONDENT(?:S)?|OPPOSITE PARTY)\b",
                re.I
            )
        ),
    ]

    for label, pattern in role_patterns:
        for match in pattern.finditer(text):
            raw = match.group("name")
            start, end = match.span("name")

            # Reject long, likely heading/caption captures.
            raw_clean = raw.strip(" .,:;…")
            if not raw_clean or len(raw_clean) > 90:
                continue
            if len(raw_clean.split()) > 12:
                continue

            add_candidate(
                candidates, text, start, end, label, "caption_role"
            )

    # Witness references: name immediately followed by (PW-5), PW-5, LW-3, etc.
    witness_pattern = re.compile(
        r"(?P<name>"
        r"(?:[A-Z][A-Za-z.'-]*\s+){0,5}"
        r"[A-Z][A-Za-z.'-]*"
        r")\s*,?\s*\(?\s*(?:PW|LW)\s*[-–]?\s*\d+\s*\)?",
        re.I
    )

    for match in witness_pattern.finditer(text):
        raw = match.group("name").strip()
        # Avoid labeling obvious generic role words as witness names.
        if raw.lower() in {
            "witness", "prosecution witness", "learned witness"
        }:
            continue

        add_candidate(
            candidates,
            text,
            *match.span("name"),
            "WIT",
            "witness_marker",
        )

    return candidates


rule_candidates = {}
for doc_id, text in val_texts.items():
    rule_candidates[doc_id] = caption_rule_candidates(text)

print("Documents evaluated:", len(rule_candidates))
print("Total raw candidates:", sum(len(v) for v in rule_candidates.values()))

for doc_id in list(rule_candidates)[:3]:
    print("\nDocument:", doc_id)
    for s, e, label, source in rule_candidates[doc_id][:10]:
        print(label, repr(val_texts[doc_id][s:e]), (s, e), source)

In [ ]:
rule_only_preds = {
    doc_id: [
        (s, e, label)
        for s, e, label, source in spans
    ]
    for doc_id, spans in rule_candidates.items()
}

rule_rows, rule_macro = score_span_predictions(
    val_gold, rule_only_preds, RULE_LABELS
)

print(f"Rule-only macro-F1 on the 5 labels: {rule_macro:.4f}")
for label, row in rule_rows.items():
    print(
        f"{label:12s} P={row['precision']:.3f} "
        f"R={row['recall']:.3f} F1={row['F1']:.3f} "
        f"pred={row['pred']} gold={row['gold']}"
    )

In [ ]:
champion_rule_union = {}

for doc_id in val_texts:
    original = [
        (int(s), int(e), label)
        for s, e, label, *rest in champion_preds.get(doc_id, [])
    ]
    extra = [
        (s, e, label)
        for s, e, label, source in rule_candidates.get(doc_id, [])
    ]

    champion_rule_union[doc_id] = sorted(set(original + extra))

union_rows, union_macro = score_span_predictions(
    val_gold, champion_rule_union, RULE_LABELS
)

print(f"Champion baseline macro-F1: {baseline_macro:.4f}")
print(f"Rule-only macro-F1:         {rule_macro:.4f}")
print(f"Champion + rules macro-F1:  {union_macro:.4f}")

print("\nPer-label comparison")
for label in sorted(RULE_LABELS):
    print(
        f"{label:12s} "
        f"champion={baseline_rows[label]['F1']:.3f} "
        f"rules={rule_rows[label]['F1']:.3f} "
        f"union={union_rows[label]['F1']:.3f}"
    )

In [ ]:
gold_sets = gold_to_span_set(val_gold)
rule_sets = normalize_span_set(rule_only_preds)

for label in sorted(RULE_LABELS):
    print(f"\n{'='*18} {label} {'='*18}")

    false_positives = []
    false_negatives = []

    for doc_id, text in val_texts.items():
        gold_spans = {
            x for x in gold_sets[doc_id] if x[2] == label
        }
        pred_spans = {
            x for x in rule_sets.get(doc_id, set()) if x[2] == label
        }

        for s, e, _ in sorted(pred_spans - gold_spans):
            false_positives.append(
                (doc_id, text[s:e], text[max(0, s-60):min(len(text), e+60)])
            )

        for s, e, _ in sorted(gold_spans - pred_spans):
            false_negatives.append(
                (doc_id, text[s:e], text[max(0, s-60):min(len(text), e+60)])
            )

    print("False positives:", len(false_positives))
    for item in false_positives[:5]:
        print(" FP:", repr(item[1]), "|", item[2].replace("\n", " "))

    print("False negatives:", len(false_negatives))
    for item in false_negatives[:5]:
        print(" FN:", repr(item[1]), "|", item[2].replace("\n", " "))

In [ ]:
import re
from collections import defaultdict

def clean_candidate_span(text, start, end, label, source):
    # Remove only obvious boundary whitespace and punctuation.
    while start < end and (text[start].isspace() or text[start] in ".,:;…"):
        start += 1
    while end > start and (text[end-1].isspace() or text[end-1] in ".,:;…"):
        end -= 1

    if start >= end:
        return None

    candidate = text[start:end].strip()
    if not candidate or len(candidate) > 100:
        return None

    # Avoid capturing phrases rather than names.
    bad_starts = (
        "the ", "that ", "was ", "when ", "herein",
        "challenging ", "original writ", "appellants versus",
        "factual matrix", "no. "
    )
    if candidate.lower().startswith(bad_starts):
        return None

    return (start, end, label, source)


def improved_rule_candidates(text):
    out = []

    # Parse the short case-caption region before the first judgment body.
    # We require the role word to occur after the candidate, with a bounded
    # candidate and a nearby caption separator.
    header_end = len(text)
    body_markers = [
        r"\bJUDGMENT\b",
        r"\bJ U D G M E N T\b",
        r"\bJUDGEMENT\b",
    ]
    positions = [
        m.start()
        for pattern in body_markers
        if (m := re.search(pattern, text, re.I))
    ]
    if positions:
        header_end = min(positions)

    header = text[:header_end]

    # Split caption around explicit Versus / Vs. delimiter.
    versus = re.search(r"\b(?:VERSUS|VS\.?)\b", header, re.I)
    if versus:
        left = header[:versus.start()]
        right = header[versus.end():]

        # For each side, keep only text after common case-number headings.
        # We still require a role marker to avoid arbitrary long captures.
        left_start = max(
            left.rfind("\n"),
            left.rfind("PETITIONER:"),
            left.rfind("APPELLANT(S)"),
            left.rfind("APPELLANTS"),
        )
        right_start = max(
            right.rfind("\n"),
            right.rfind("RESPONDENT:"),
            right.rfind("RESPONDENTS"),
        )

        # Explicit caption role labels, where present.
        app_match = re.search(
            r"(?P<name>[A-Z][A-Z0-9&.,'()@/\- ]{1,70}?)"
            r"\s*(?:…|\.\.\.)?\s*APPELLANT(?:\(S\))?\b",
            header,
            re.I,
        )
        if app_match:
            raw = app_match.group("name")
            # Keep the final plausible name-like portion rather than a whole
            # preceding sentence.
            raw = re.split(r"[\n;]", raw)[-1]
            start = app_match.start("name") + (
                app_match.group("name").rfind(raw)
            )
            item = clean_candidate_span(
                text, start, start + len(raw), "APP", "caption_appellant"
            )
            if item:
                out.append(item)

        resp_match = re.search(
            r"(?P<name>[A-Z][A-Z0-9&.,'()@/\- ]{1,70}?)"
            r"\s*(?:…|\.\.\.)?\s*RESPONDENT(?:\(S\))?\b",
            header,
            re.I,
        )
        if resp_match:
            raw = resp_match.group("name")
            raw = re.split(r"[\n;]", raw)[-1]
            start = resp_match.start("name") + (
                resp_match.group("name").rfind(raw)
            )
            item = clean_candidate_span(
                text, start, start + len(raw), "RESP", "caption_respondent"
            )
            if item:
                out.append(item)

    # Witness marker: capture the name immediately before PW/LW marker.
    # Deliberately excludes arbitrary text across punctuation/newlines.
    witness_pattern = re.compile(
        r"(?P<name>(?:[A-Z][A-Za-z.'-]*\s+){0,4}[A-Z][A-Za-z.'-]*)"
        r"\s*,?\s*\(?\s*(?:PW|LW)\s*[-–]?\s*\d+\s*\)?",
        re.I,
    )
    for match in witness_pattern.finditer(text):
        raw = match.group("name").strip()
        if len(raw.split()) > 5:
            continue
        item = clean_candidate_span(
            text,
            match.start("name"),
            match.end("name"),
            "WIT",
            "witness_marker",
        )
        if item:
            out.append(item)

    # Deduplicate exact candidates.
    return list(dict.fromkeys(out))


rule_candidates_v2 = {
    doc_id: improved_rule_candidates(text)
    for doc_id, text in val_texts.items()
}

print("Documents:", len(rule_candidates_v2))
print("Candidates:", sum(map(len, rule_candidates_v2.values())))

for doc_id in list(rule_candidates_v2)[:5]:
    print("\nDocument:", doc_id)
    for s, e, label, source in rule_candidates_v2[doc_id][:8]:
        print(label, repr(val_texts[doc_id][s:e]), (s, e), source)

In [ ]:
rule_only_preds_v2 = {
    doc_id: [(s, e, label) for s, e, label, source in spans]
    for doc_id, spans in rule_candidates_v2.items()
}

rule_rows_v2, rule_macro_v2 = score_span_predictions(
    val_gold, rule_only_preds_v2, RULE_LABELS
)

print(f"Rule-only macro-F1 (five labels): {rule_macro_v2:.4f}")
for label, row in rule_rows_v2.items():
    print(
        f"{label:12s} P={row['precision']:.3f} "
        f"R={row['recall']:.3f} F1={row['F1']:.3f} "
        f"pred={row['pred']} gold={row['gold']}"
    )

In [ ]:
champion_rule_union_v2 = {}

for doc_id in val_texts:
    champion_spans = [
        (int(s), int(e), label)
        for s, e, label, *rest in champion_preds.get(doc_id, [])
    ]
    extra_spans = [
        (s, e, label)
        for s, e, label, source in rule_candidates_v2.get(doc_id, [])
    ]
    champion_rule_union_v2[doc_id] = sorted(
        set(champion_spans + extra_spans)
    )

union_rows_v2, union_macro_v2 = score_span_predictions(
    val_gold, champion_rule_union_v2, RULE_LABELS
)

print(f"Champion baseline (five labels): {baseline_macro:.4f}")
print(f"Corrected rule-only:              {rule_macro_v2:.4f}")
print(f"Champion + corrected rules:       {union_macro_v2:.4f}")

for label in sorted(RULE_LABELS):
    print(
        f"{label:12s} "
        f"champion={baseline_rows[label]['F1']:.3f} "
        f"rules={rule_rows_v2[label]['F1']:.3f} "
        f"union={union_rows_v2[label]['F1']:.3f}"
    )

In [ ]:
from collections import defaultdict
import numpy as np

WEAK_LABELS = {"APP", "RESP", "A.COUNSEL", "R.COUNSEL", "WIT"}

gold_counts = defaultdict(int)
candidate_counts = defaultdict(int)
kept_counts = defaultdict(int)
conf_by_label = defaultdict(list)

for doc_id, entities in val_gold.items():
    for ent in entities:
        if str(ent[2]) in WEAK_LABELS:
            gold_counts[str(ent[2])] += 1

for doc_id, spans in confidence_diag.items():
    for s, e, label, conf in spans:
        if label not in WEAK_LABELS:
            continue

        candidate_counts[label] += 1
        conf_by_label[label].append(float(conf))

        # Use the same thresholds as the champion's current evaluation.
        if conf >= champion_thresholds_fusion.get(label, 1.0):
            kept_counts[label] += 1

print("Label       Gold   Candidates   Kept   Confidence percentiles")
for label in sorted(WEAK_LABELS):
    vals = np.array(conf_by_label[label], dtype=float)
    pct = (
        np.percentile(vals, [10, 25, 50, 75, 90]).round(3).tolist()
        if len(vals) else []
    )
    print(
        f"{label:12s} {gold_counts[label]:4d} "
        f"{candidate_counts[label]:11d} {kept_counts[label]:6d}   {pct}"
    )

In [ ]:
def evaluate_weak_thresholds(confidence_diag, texts, gold_by_doc):
    results = {}

    thresholds_to_test = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60]

    for label in sorted(WEAK_LABELS):
        label_results = []

        for threshold in thresholds_to_test:
            preds = {doc_id: [] for doc_id in texts}

            for doc_id, spans in confidence_diag.items():
                for s, e, pred_label, conf in spans:
                    if pred_label != label or conf < threshold:
                        continue
                    if 0 <= int(s) < int(e) <= len(texts[doc_id]):
                        preds[doc_id].append((int(s), int(e), pred_label))

            rows, macro = score_span_predictions(
                gold_by_doc, preds, {label}
            )
            label_results.append({
                "threshold": threshold,
                "F1": rows[label]["F1"],
                "precision": rows[label]["precision"],
                "recall": rows[label]["recall"],
                "pred": rows[label]["pred"],
            })

        results[label] = sorted(
            label_results, key=lambda x: x["F1"], reverse=True
        )

    return results


weak_threshold_results = evaluate_weak_thresholds(
    confidence_diag, val_texts, val_gold
)

for label, rows in weak_threshold_results.items():
    print(f"\n{label}")
    for row in rows[:5]:
        print(row)

In [ ]:
from collections import Counter

# Inspect the existing cached model diagnostics, if available.
print("Available diagnostic variables:")
for name in [
    "confidence_diag", "champion_preds", "val_texts", "val_gold",
    "ner_model", "BIO_LABELS", "label2id"
]:
    print(f"{name}: {name in globals()}")

print("\nCandidate counts by label in constrained predictions:")
counts = Counter(
    label
    for spans in confidence_diag.values()
    for s, e, label, conf in spans
)
for label in sorted(WEAK_LABELS):
    print(f"{label:12s}: {counts[label]}")

In [ ]:
import torch
from collections import Counter

doc_id = next(iter(val_texts))
text = val_texts[doc_id]
enc = tokenizer.encode(text, add_special_tokens=False)

ids = enc.ids
offsets = enc.offsets

print("Document:", doc_id)
print("Characters:", len(text))
print("Tokens:", len(ids))
print("Raw BIO label counts will be computed on the first window.")

device = next(ner_model.parameters()).device
input_ids = torch.tensor(
    [ids[:MAX_LEN]], dtype=torch.long, device=device
)
attention_mask = torch.ones_like(input_ids)

ner_model.eval()
with torch.no_grad():
    logits = ner_model(input_ids, attention_mask)

raw_ids = logits[0].argmax(dim=-1).detach().cpu().tolist()
raw_counts = Counter(BIO_LABELS[i] for i in raw_ids)

for label in sorted(WEAK_LABELS):
    print(
        f"{label:12s} B={raw_counts[f'B-{label}']:4d} "
        f"I={raw_counts[f'I-{label}']:4d}"
    )

print("\nMost common raw tags:")
for tag, count in raw_counts.most_common(15):
    print(f"{tag:12s} {count}")

In [ ]:
import torch
import time
from collections import Counter

raw_pred_counts = Counter()
gold_token_counts = Counter()
documents_with_raw_weak_tags = Counter()

device = next(ner_model.parameters()).device
ner_model.eval()
t0 = time.time()

with torch.no_grad():
    for doc_num, (doc_id, text) in enumerate(val_texts.items(), 1):
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets

        gold_tags = [label2id["O"]] * len(ids)

        for ent in val_gold[doc_id]:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            indices = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]
            if indices:
                gold_tags[indices[0]] = label2id[f"B-{label}"]
                for i in indices[1:]:
                    gold_tags[i] = label2id[f"I-{label}"]

        gold_token_counts.update(BIO_LABELS[t] for t in gold_tags)

        # Preserve first-window ownership for overlapping tokens.
        seen = set()
        pred_tags = []

        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            if start >= end:
                break

            input_ids = torch.tensor(
                [ids[start:end]], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            logits = ner_model(input_ids, attention_mask)[0]
            local_preds = logits.argmax(dim=-1).cpu().tolist()

            for local_i, pred_id in enumerate(local_preds):
                global_i = start + local_i
                if global_i not in seen:
                    seen.add(global_i)
                    pred_tags.append((global_i, BIO_LABELS[pred_id]))

            if end == len(ids):
                break

        pred_tags.sort(key=lambda x: x[0])
        tags_only = [tag for _, tag in pred_tags]
        raw_pred_counts.update(tags_only)

        for label in WEAK_LABELS:
            if any(tag in (f"B-{label}", f"I-{label}") for tag in tags_only):
                documents_with_raw_weak_tags[label] += 1

        print(
            f"[{doc_num:02d}/{len(val_texts)}] {doc_id}: "
            f"{len(ids)} tokens, {time.time()-t0:.1f}s elapsed",
            flush=True
        )

print("\nRaw argmax predictions vs gold token tags")
print(f"{'Label':12s} {'Gold tokens':>12s} {'Pred tokens':>12s}")
for label in BIO_LABELS:
    g, p = gold_token_counts[label], raw_pred_counts[label]
    if g or p:
        print(f"{label:12s} {g:12d} {p:12d}")

print("\nDocuments with raw weak-label tokens:")
for label in sorted(WEAK_LABELS):
    print(
        f"{label:12s} "
        f"{documents_with_raw_weak_tags[label]:2d}/{len(val_texts)}"
    )

print(f"\nTotal elapsed: {time.time()-t0:.1f}s")

In [ ]:
import torch

champion_path = "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
ckpt = torch.load(champion_path, map_location="cpu", weights_only=False)

print("Checkpoint keys:")
print(list(ckpt.keys()))

for key in ["epoch", "best_f1", "val_f1", "num_labels", "label2id",
            "id2label", "bio_labels", "config", "model_config"]:
    if key in ckpt:
        value = ckpt[key]
        if isinstance(value, dict) and len(value) > 20:
            print(f"\n{key}: {list(value.items())[:8]} ...")
        else:
            print(f"\n{key}: {value}")

print("\nExpected BIO label order:")
for i, label in enumerate(BIO_LABELS):
    print(i, label)

print("\nModel classifier tensors:")
for name, tensor in ckpt["model_state_dict"].items():
    if any(x in name.lower() for x in ["classifier", "score", "out_proj"]):
        if hasattr(tensor, "shape"):
            print(name, tuple(tensor.shape))

In [ ]:
import inspect
import torch

print("=== Dataset and training settings ===")
for name in [
    "MAX_LEN", "WINDOW_STRIDE", "BATCH_SIZE",
    "IGNORE_INDEX", "BIO_LABELS", "WEAK_LABELS"
]:
    if name in globals():
        print(f"{name} = {globals()[name]}")

print("\n=== Current criterion ===")
if "criterion" in globals():
    print(criterion)
    if hasattr(criterion, "weight") and criterion.weight is not None:
        for i, w in enumerate(criterion.weight.detach().cpu().tolist()):
            print(f"{BIO_LABELS[i]:12s} {w:.3f}")
else:
    print("criterion not defined")

print("\n=== Current optimizer ===")
if "optimizer" in globals():
    print(optimizer)
    for group in optimizer.param_groups:
        print({
            "lr": group.get("lr"),
            "weight_decay": group.get("weight_decay"),
            "parameter_tensors": len(group["params"]),
        })
else:
    print("optimizer not defined")

print("\n=== Training function signature ===")
if "train_one_epoch" in globals():
    print(inspect.signature(train_one_epoch))
    print(inspect.getsource(train_one_epoch))

In [ ]:
import torch

ckpt = torch.load(
    "/kaggle/working/phase2_ner_corrected/best_ner_model.pt",
    map_location="cpu",
    weights_only=False,
)

print("Epoch:", ckpt.get("epoch"))
print("Validation exact-span macro-F1:",
      ckpt.get("val_exact_span_macro_f1"))
print("Training fold:", ckpt.get("train_fold"))
print("Validation fold:", ckpt.get("validation_fold"))
print("Phase 1 checkpoint:", ckpt.get("phase1_checkpoint"))

assert ckpt["bio_labels"] == BIO_LABELS, (
    "Checkpoint BIO labels do not match the active notebook mapping!"
)
print("BIO mapping verified.")

In [ ]:
import os
import copy
import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2LMHeadModel

EXP_F_DIR = "/kaggle/working/phase2_ner_experiment_F"
os.makedirs(EXP_F_DIR, exist_ok=True)

PHASE1_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase-1-checkpoints/checkpoint_latest.pt"
)

phase1_ckpt_F = torch.load(
    PHASE1_PATH, map_location="cpu", weights_only=False
)
cfg_F = phase1_ckpt_F["model_config"]

gpt2_config_F = GPT2Config(
    vocab_size=cfg_F["vocab_size"],
    n_positions=cfg_F["context_length"],
    n_ctx=cfg_F["context_length"],
    n_embd=cfg_F["hidden_size"],
    n_layer=cfg_F["num_layers"],
    n_head=cfg_F["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

phase1_model_F = GPT2LMHeadModel(gpt2_config_F)
phase1_model_F.load_state_dict(
    phase1_ckpt_F["model_state_dict"], strict=True
)

# Reuse the notebook's existing LegalNERModel class.
model_F = LegalNERModel(
    pretrained_lm=phase1_model_F,
    num_labels=len(BIO_LABELS)
).to(device)

# Independent optimizer; do not reuse the global optimizer.
optimizer_F = torch.optim.AdamW(
    model_F.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

# Weak entity-start tags get the strongest emphasis.
weights_F = torch.ones(len(BIO_LABELS), dtype=torch.float32)

for i, tag in enumerate(BIO_LABELS):
    if tag.startswith("B-") and tag[2:] in WEAK_LABELS:
        weights_F[i] = 8.0
    elif tag.startswith("I-") and tag[2:] in WEAK_LABELS:
        weights_F[i] = 4.0
    elif tag.startswith(("B-", "I-")):
        weights_F[i] = 3.0

# Keep O at 1.0, matching the previous baseline.
criterion_F = nn.CrossEntropyLoss(
    weight=weights_F.to(device),
    ignore_index=IGNORE_INDEX
)

print("Experiment F initialized")
print("Model parameters:", sum(p.numel() for p in model_F.parameters()))
print("Checkpoint directory:", EXP_F_DIR)
print("\nLoss weights:")
for i, tag in enumerate(BIO_LABELS):
    print(f"{tag:12s} {weights_F[i].item():.1f}")

In [ ]:
import time
import torch

def train_one_epoch_F(model):
    model.train()
    total_loss = 0.0
    total_batches = 0
    start_time = time.time()

    for batch_idx, (input_ids, attention_mask, labels) in enumerate(
        train_loader, 1
    ):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer_F.zero_grad(set_to_none=True)

        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(output, dict):
            logits = output["logits"]
        elif hasattr(output, "logits"):
            logits = output.logits
        elif isinstance(output, (tuple, list)):
            logits = output[0]
        else:
            logits = output

        loss = criterion_F(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at batch {batch_idx}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer_F.step()

        total_loss += loss.item()
        total_batches += 1

        if batch_idx % 25 == 0 or batch_idx == len(train_loader):
            print(
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"loss={loss.item():.4f} | "
                f"elapsed={time.time()-start_time:.1f}s",
                flush=True
            )

    return total_loss / max(total_batches, 1)


loss_F = train_one_epoch_F(model_F)
print(f"Experiment F epoch 1 training loss: {loss_F:.4f}")

torch.save(
    {
        "model_state_dict": model_F.state_dict(),
        "bio_labels": BIO_LABELS,
        "entity_labels": ENTITY_LABELS,
        "epoch": 1,
        "train_loss": loss_F,
        "train_fold": 1,
        "validation_fold": 2,
        "experiment": "F_weak_B_tag_weighting",
    },
    os.path.join(EXP_F_DIR, "experiment_F_epoch_1.pt")
)
print("Experiment F checkpoint saved.")

In [ ]:
import os
import torch

EXP_F_CKPT = (
    "/kaggle/working/phase2_ner_experiment_F/"
    "experiment_F_epoch_1.pt"
)

assert os.path.exists(EXP_F_CKPT), f"Missing checkpoint: {EXP_F_CKPT}"

ckpt_F = torch.load(
    EXP_F_CKPT, map_location="cpu", weights_only=False
)

assert ckpt_F["bio_labels"] == BIO_LABELS
model_F.load_state_dict(ckpt_F["model_state_dict"], strict=True)
model_F.to(device)
model_F.eval()

print("Loaded Experiment F epoch:", ckpt_F["epoch"])
print("Training loss:", ckpt_F["train_loss"])
print("Device:", next(model_F.parameters()).device)
print("Champion checkpoint remains untouched.")

In [ ]:
import torch
import numpy as np

def predict_doc_F_with_confidence(text, model):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets

    if not ids:
        return []

    logit_sum = torch.zeros(
        (len(ids), len(BIO_LABELS)), dtype=torch.float32
    )
    token_count = torch.zeros(len(ids), dtype=torch.float32)

    model.eval()

    with torch.inference_mode():
        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            chunk = ids[start:end]

            input_ids = torch.tensor(
                [chunk], dtype=torch.long, device=device
            )
            attention_mask = torch.ones_like(input_ids)

            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            if isinstance(output, dict):
                logits = output["logits"]
            elif hasattr(output, "logits"):
                logits = output.logits
            elif isinstance(output, (tuple, list)):
                logits = output[0]
            else:
                logits = output

            logits = logits[0].detach().float().cpu()
            logit_sum[start:end] += logits
            token_count[start:end] += 1

            del input_ids, attention_mask, output, logits

            if end == len(ids):
                break

    averaged_logits = (
        logit_sum / token_count.clamp_min(1).unsqueeze(-1)
    )
    probabilities = torch.softmax(averaged_logits, dim=-1)

    pred_ids = bio_viterbi_decode(averaged_logits)
    spans = decode_spans(text, offsets, pred_ids)

    pred_ids_tensor = torch.tensor(pred_ids, dtype=torch.long)
    results = []

    for start, end, label in spans:
        token_indices = [
            i for i, (s, e) in enumerate(offsets)
            if e > s and s < end and e > start
        ]

        if token_indices:
            selected_probs = probabilities[
                token_indices, pred_ids_tensor[token_indices]
            ]
            confidence = float(selected_probs.mean().item())
        else:
            confidence = 0.0

        results.append((int(start), int(end), label, confidence))

    return results


print("Experiment F inference function ready.")

In [ ]:
import time
from collections import Counter

predictions_F_raw = {}
start_time_F = time.time()

for i, (doc_id, text) in enumerate(val_texts.items(), 1):
    predictions_F_raw[doc_id] = predict_doc_F_with_confidence(
        text, model_F
    )

    if i % 5 == 0 or i == len(val_texts):
        print(
            f"[{i}/{len(val_texts)}] "
            f"elapsed={time.time() - start_time_F:.1f}s",
            flush=True
        )

print("\nTotal raw candidate spans:", sum(
    len(v) for v in predictions_F_raw.values()
))

In [ ]:
thresholds_F_comparison = dict(champion_thresholds_fusion)

predictions_F_fixed = filter_predictions_by_thresholds_for_texts(
    predictions_F_raw,
    thresholds_F_comparison,
    val_texts
)

metrics_F_fixed = calculate_exact_span_metrics(
    val_gold,
    predictions_F_fixed,
    ENTITY_LABELS
)

print("Experiment F macro-F1 at champion thresholds:",
      round(metrics_F_fixed["MACRO_F1"], 6))

for label in ENTITY_LABELS:
    m = metrics_F_fixed[label]
    print(
        f"{label:12s} "
        f"F1={m['f1']:.4f} "
        f"P={m['precision']:.4f} "
        f"R={m['recall']:.4f} "
        f"pred={m['pred']:4d} gold={m['gold']:4d}"
    )

In [ ]:
champion_metrics = calculate_exact_span_metrics(
    val_gold,
    champion_preds,
    ENTITY_LABELS
)

print(f"Champion macro-F1:     {champion_metrics['MACRO_F1']:.6f}")
print(f"Experiment F macro-F1: {metrics_F_fixed['MACRO_F1']:.6f}")
print("\nPer-label F1 comparison:")

for label in ENTITY_LABELS:
    print(
        f"{label:12s} "
        f"champion={champion_metrics[label]['f1']:.4f} "
        f"experiment_F={metrics_F_fixed[label]['f1']:.4f}"
    )

In [ ]:
from collections import Counter

print("Experiment F: predictions at champion thresholds")
print("Total spans:", sum(len(v) for v in predictions_F_fixed.values()))

for label in ENTITY_LABELS:
    pred_n = sum(
        1 for spans in predictions_F_fixed.values()
        for s, e, lab in spans if lab == label
    )
    gold_n = sum(
        1 for entities in val_gold.values()
        for ent in entities if str(ent[2]) == label
    )
    print(f"{label:12s} pred={pred_n:5d} gold={gold_n:5d}")

print("\nRaw Experiment F candidates:")
for label in ENTITY_LABELS:
    vals = [
        float(conf)
        for spans in predictions_F_raw.values()
        for s, e, lab, conf in spans if lab == label
    ]
    print(
        f"{label:12s} candidates={len(vals):5d} "
        f"mean_conf={sum(vals)/len(vals):.3f}" if vals else
        f"{label:12s} candidates=    0"
    )

In [ ]:
import time
import torch

model_F.eval()

doc_id = next(iter(val_texts))
text = val_texts[doc_id]
enc = tokenizer.encode(text, add_special_tokens=False)
ids = enc.ids

print("Test document:", doc_id)
print("Token count:", len(ids))
print("Device:", device)

start_time = time.time()

with torch.no_grad():
    end = min(MAX_LEN, len(ids))
    input_ids = torch.tensor(
        [ids[:end]], dtype=torch.long, device=device
    )
    attention_mask = torch.ones_like(input_ids)
    logits = model_F(input_ids, attention_mask)

print("One-window inference seconds:", round(time.time() - start_time, 2))
print("Logits shape:", tuple(logits.shape))

In [ ]:
print("Cached Experiment F predictions:")
for label in ["A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"]:
    n = sum(
        1 for spans in predictions_F_raw.values()
        for s, e, lab, conf in spans
        if lab == label
    )
    print(f"{label:12s}: {n} decoded spans")

print("\nRuntime device:", device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import torch
import time

if torch.cuda.is_available():
    model_F = model_F.to("cuda")
    device_F = torch.device("cuda")
else:
    device_F = torch.device("cpu")

model_F.eval()

print("Experiment F device:", next(model_F.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Quick inference benchmark
doc_id = next(iter(val_texts))
enc = tokenizer.encode(val_texts[doc_id], add_special_tokens=False)
ids = enc.ids[:MAX_LEN]

input_ids = torch.tensor([ids], dtype=torch.long, device=device_F)
attention_mask = torch.ones_like(input_ids)

with torch.no_grad():
    start = time.time()
    logits = model_F(input_ids, attention_mask)
    if device_F.type == "cuda":
        torch.cuda.synchronize()
    print("Inference seconds:", round(time.time() - start, 3))
    print("Logits shape:", tuple(logits.shape))

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict
import time

model_F.eval()
device_F = next(model_F.parameters()).device

# Sample 10 documents, spread across the validation set.
doc_items = list(val_texts.items())
sample_indices = torch.linspace(
    0, len(doc_items) - 1, min(10, len(doc_items))
).long().tolist()
sample_docs = [doc_items[i] for i in sample_indices]

bio_to_id = {label: i for i, label in enumerate(BIO_LABELS)}
target_labels = ["A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"]

class_stats = {
    label: {"gold_tokens": 0, "prob_sum": 0.0, "top1": 0}
    for label in target_labels
}

start_time = time.time()

with torch.inference_mode():
    for doc_num, (doc_id, text) in enumerate(sample_docs, 1):
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets
        if not ids:
            continue

        # Average logits from overlapping windows.
        logit_sum = torch.zeros(
            len(ids), len(BIO_LABELS), device=device_F
        )
        counts = torch.zeros(len(ids), device=device_F)

        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            if start >= end:
                break

            input_ids = torch.tensor(
                [ids[start:end]], dtype=torch.long, device=device_F
            )
            mask = torch.ones_like(input_ids)
            logits = model_F(input_ids, mask)[0]

            logit_sum[start:end] += logits
            counts[start:end] += 1

            if end == len(ids):
                break

        probs = F.softmax(
            logit_sum / counts.clamp_min(1).unsqueeze(-1), dim=-1
        )
        top_ids = probs.argmax(dim=-1)

        # Map gold character spans to overlapping tokens.
        gold_token_labels = defaultdict(set)
        for ent in val_gold[doc_id]:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            if label not in target_labels:
                continue
            for i, (a, b) in enumerate(offsets):
                if b > a and a < e and b > s:
                    gold_token_labels[i].add(label)

        for label in target_labels:
            b_id = bio_to_id[f"B-{label}"]
            i_id = bio_to_id[f"I-{label}"]
            gold_idx = [
                i for i, labs in gold_token_labels.items()
                if label in labs
            ]
            if not gold_idx:
                continue

            idx = torch.tensor(gold_idx, device=device_F)
            p = probs[idx, b_id] + probs[idx, i_id]

            class_stats[label]["gold_tokens"] += len(gold_idx)
            class_stats[label]["prob_sum"] += p.sum().item()
            class_stats[label]["top1"] += (
                ((top_ids[idx] == b_id) | (top_ids[idx] == i_id))
                .sum().item()
            )

        print(f"Processed {doc_num}/{len(sample_docs)}: {doc_id}")

print(f"\nElapsed: {time.time() - start_time:.1f}s")
print(f"{'LABEL':12s} {'GOLD TOKENS':>11s} {'MEAN GOLD P':>12s} {'TOP-1 %':>9s}")

for label in target_labels:
    s = class_stats[label]
    n = s["gold_tokens"]
    mean_p = s["prob_sum"] / n if n else 0
    top1 = 100 * s["top1"] / n if n else 0
    print(f"{label:12s} {n:11d} {mean_p:12.4f} {top1:8.2f}%")

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict

# Use the same 10 validation documents as the Experiment F diagnostic.
# Assumes `sample_docs`, `val_texts`, `val_gold`, `tokenizer`,
# `BIO_LABELS`, `ENTITY_LABELS`, `WINDOW_STRIDE`, and `MAX_LEN` exist.

champion_path = "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
champ_ckpt = torch.load(champion_path, map_location="cpu", weights_only=False)

print("Checkpoint keys:", list(champ_ckpt.keys()))

In [ ]:
import torch
from transformers import GPT2Config, GPT2LMHeadModel

champion_path = "/kaggle/working/phase2_ner_corrected/best_ner_model.pt"
phase1_path = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"

champ_ckpt = torch.load(
    champion_path, map_location="cpu", weights_only=False
)
phase1_ckpt = torch.load(
    phase1_path, map_location="cpu", weights_only=False
)

cfg = phase1_ckpt["model_config"]

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
)

phase1_lm = GPT2LMHeadModel(gpt2_config)
phase1_lm.load_state_dict(
    phase1_ckpt["model_state_dict"], strict=True
)

champion_model = LegalNERModel(
    pretrained_lm=phase1_lm,
    num_labels=len(BIO_LABELS),
).to(device_F)

champion_model.load_state_dict(
    champ_ckpt["model_state_dict"], strict=True
)
champion_model.eval()

print("Champion device:", next(champion_model.parameters()).device)
print("Saved validation F1:", champ_ckpt.get("val_exact_span_macro_f1"))
print("Experiment F device:", next(model_F.parameters()).device)

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict
import time

target_labels = ["A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"]
bio_to_id = {label: i for i, label in enumerate(BIO_LABELS)}

def gold_token_probability_stats(model, docs, labels):
    stats = {
        label: {"n": 0, "prob_sum": 0.0, "top1": 0}
        for label in labels
    }
    model.eval()
    model_device = next(model.parameters()).device

    with torch.inference_mode():
        for doc_id, text in docs:
            enc = tokenizer.encode(text, add_special_tokens=False)
            ids, offsets = enc.ids, enc.offsets
            if not ids:
                continue

            logit_sum = torch.zeros(
                len(ids), len(BIO_LABELS), device=model_device
            )
            counts = torch.zeros(len(ids), device=model_device)

            for start in range(0, len(ids), WINDOW_STRIDE):
                end = min(start + MAX_LEN, len(ids))
                if start >= end:
                    break

                x = torch.tensor(
                    [ids[start:end]], dtype=torch.long,
                    device=model_device
                )
                logits = model(x, torch.ones_like(x))[0]
                logit_sum[start:end] += logits
                counts[start:end] += 1

                if end == len(ids):
                    break

            probs = F.softmax(
                logit_sum / counts.clamp_min(1).unsqueeze(-1),
                dim=-1
            )
            top_ids = probs.argmax(dim=-1)

            gold_tokens = defaultdict(set)
            for ent in val_gold[doc_id]:
                s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
                if label not in labels:
                    continue
                for i, (a, b) in enumerate(offsets):
                    if b > a and a < e and b > s:
                        gold_tokens[i].add(label)

            for label in labels:
                b_id = bio_to_id[f"B-{label}"]
                i_id = bio_to_id[f"I-{label}"]
                idx_list = [
                    i for i, labs in gold_tokens.items()
                    if label in labs
                ]
                if not idx_list:
                    continue

                idx = torch.tensor(idx_list, device=model_device)
                p = probs[idx, b_id] + probs[idx, i_id]
                stats[label]["n"] += len(idx_list)
                stats[label]["prob_sum"] += p.sum().item()
                stats[label]["top1"] += (
                    ((top_ids[idx] == b_id) | (top_ids[idx] == i_id))
                    .sum().item()
                )

    return stats

# `sample_docs` is the same 10-document sample used for Experiment F.
t0 = time.time()
champ_stats = gold_token_probability_stats(
    champion_model, sample_docs, target_labels
)
f_stats = gold_token_probability_stats(
    model_F, sample_docs, target_labels
)

print(f"Elapsed: {time.time() - t0:.1f}s")
print(
    f"{'LABEL':12s} {'CHAMP P':>10s} {'CHAMP TOP1':>11s} "
    f"{'F P':>10s} {'F TOP1':>9s}"
)

for label in target_labels:
    c, f = champ_stats[label], f_stats[label]
    cp = c["prob_sum"] / c["n"] if c["n"] else 0
    ct = 100 * c["top1"] / c["n"] if c["n"] else 0
    fp = f["prob_sum"] / f["n"] if f["n"] else 0
    ft = 100 * f["top1"] / f["n"] if f["n"] else 0
    print(
        f"{label:12s} {cp:10.4f} {ct:10.2f}% "
        f"{fp:10.4f} {ft:8.2f}%"
    )

In [ ]:
# Inspect weak-class gold spans and their tokenizer offsets.
# Read-only: no training and no checkpoint changes.

target_labels = ["A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"]
shown = {label: 0 for label in target_labels}

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)

    for ent in val_gold[doc_id]:
        s, e, label = int(ent[0]), int(ent[1]), str(ent[2])

        if label not in target_labels or shown[label] >= 2:
            continue

        overlapping = [
            (i, a, b, enc.tokens[i])
            for i, (a, b) in enumerate(enc.offsets)
            if b > a and a < e and b > s
        ]

        print(f"\n{label} | doc={doc_id} | gold=({s},{e})")
        print("Gold text:", repr(text[s:e]))
        print("Overlapping tokens:")
        for i, a, b, tok in overlapping:
            print(
                f"  token_idx={i:4d} offsets=({a},{b}) "
                f"token={tok!r} token_text={text[a:b]!r}"
            )

        shown[label] += 1

    if all(shown[label] >= 2 for label in target_labels):
        break

In [ ]:
examples_to_check = [
    ("184861797", "JUDGE", "Hiranmay Bhattacharyya"),
    ("184861797", "A.COUNSEL", "Ghosh"),
    ("184861797", "R.COUNSEL", "R.N. Chakraborty"),
    ("184861797", "R.COUNSEL", "M. Ahmed"),
    ("50547742", "WIT", "Purushottam Bonde"),
    ("79816183", "WIT", "Pathan Basheerunnisa"),
]

bio_labels = BIO_LABELS
label_names = {i: label for i, label in enumerate(bio_labels)}

for doc_id, target_label, target_text in examples_to_check:
    text = val_texts[doc_id]
    entities = val_gold[doc_id]
    enc = tokenizer.encode(text, add_special_tokens=False)

    matching = [
        ent for ent in entities
        if str(ent[2]) == target_label
        and text[int(ent[0]):int(ent[1])] == target_text
    ]

    if not matching:
        print(f"\nNOT FOUND: {doc_id} | {target_label} | {target_text}")
        continue

    ent = matching[0]
    s, e, label = int(ent[0]), int(ent[1]), str(ent[2])

    tags = [label2id["O"]] * len(enc.ids)
    overlapping = [
        i for i, (a, b) in enumerate(enc.offsets)
        if b > a and a < e and b > s
    ]

    if overlapping:
        tags[overlapping[0]] = label2id[f"B-{label}"]
        for i in overlapping[1:]:
            tags[i] = label2id[f"I-{label}"]

    print(f"\n{target_label}: {target_text!r} | doc={doc_id}")
    for i in overlapping:
        a, b = enc.offsets[i]
        print(
            f"token={i:5d} offsets=({a},{b}) "
            f"text={text[a:b]!r} "
            f"assigned={label_names[tags[i]]}"
        )

In [ ]:
import torch
import torch.nn.functional as F
from collections import Counter, defaultdict

target_labels = ["A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"]
bio_to_id = {label: i for i, label in enumerate(BIO_LABELS)}

# Use the same 10 validation documents as before.
confusions = {
    label: Counter()
    for label in target_labels
}
gold_token_counts = Counter()

champion_model.eval()
device_C = next(champion_model.parameters()).device

with torch.inference_mode():
    for doc_id, text in sample_docs:
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids, offsets = enc.ids, enc.offsets
        if not ids:
            continue

        logit_sum = torch.zeros(
            len(ids), len(BIO_LABELS), device=device_C
        )
        counts = torch.zeros(len(ids), device=device_C)

        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            x = torch.tensor(
                [ids[start:end]], dtype=torch.long, device=device_C
            )
            logits = champion_model(x, torch.ones_like(x))[0]
            logit_sum[start:end] += logits
            counts[start:end] += 1
            if end == len(ids):
                break

        probs = F.softmax(
            logit_sum / counts.clamp_min(1).unsqueeze(-1), dim=-1
        )
        top_ids = probs.argmax(dim=-1).tolist()

        # Build gold token membership using character overlap.
        token_gold = defaultdict(set)
        for ent in val_gold[doc_id]:
            s, e, label = int(ent[0]), int(ent[1]), str(ent[2])
            if label not in target_labels:
                continue
            for i, (a, b) in enumerate(offsets):
                if b > a and a < e and b > s:
                    token_gold[i].add(label)

        for i, labels in token_gold.items():
            for label in labels:
                gold_token_counts[label] += 1
                predicted_tag = BIO_LABELS[top_ids[i]]
                predicted_class = (
                    predicted_tag.split("-", 1)[1]
                    if predicted_tag != "O"
                    else "O"
                )
                confusions[label][predicted_class] += 1

for label in target_labels:
    print(f"\nGold class: {label} | tokens: {gold_token_counts[label]}")
    for predicted_class, count in confusions[label].most_common(8):
        print(f"  predicted {predicted_class:12s}: {count}")

In [ ]:
import inspect

print(inspect.signature(bio_viterbi_decode))
print(inspect.signature(decode_spans))
print(inspect.signature(predict_doc_constrained_with_confidence))

In [ ]:
import inspect

print(inspect.getsource(bio_viterbi_decode))

In [ ]:
import inspect
print(inspect.getsource(predict_doc_constrained_with_confidence))

In [ ]:
import torch
import numpy as np
import time

# Keep the experiment isolated from the global champion model.
experiment_model = champion_model
experiment_model.eval()
experiment_device = next(experiment_model.parameters()).device

prec_id = BIO_LABELS.index("B-PREC")
prec_i_id = BIO_LABELS.index("I-PREC")
prec_ids = [prec_id, prec_i_id]

penalties = [0.0, 0.5, 1.0, 2.0]

# Use the known best champion thresholds.
thresholds = {
    "APP": 0.15,
    "RESP": 0.01,
    "A.COUNSEL": 0.01,
    "R.COUNSEL": 0.01,
    "JUDGE": 0.25,
    "WIT": 0.01,
    "AUTH": 0.35,
    "COURT": 0.45,
    "STAT": 0.30,
    "PREC": 0.35,
    "DATE": 0.55,
    "CASENO": 0.35,
}

# Cache averaged logits once; reuse for all penalty values.
cached_logits = {}
t0 = time.time()

with torch.inference_mode():
    for n, (doc_id, text) in enumerate(val_texts.items(), 1):
        enc = tokenizer.encode(text, add_special_tokens=False)
        ids = enc.ids
        if not ids:
            cached_logits[doc_id] = (text, enc.offsets, None)
            continue

        logit_sum = torch.zeros(
            (len(ids), len(BIO_LABELS)), dtype=torch.float32
        )
        token_count = torch.zeros(len(ids), dtype=torch.float32)

        for start in range(0, len(ids), WINDOW_STRIDE):
            end = min(start + MAX_LEN, len(ids))
            x = torch.tensor(
                [ids[start:end]],
                dtype=torch.long,
                device=experiment_device,
            )
            logits = experiment_model(x, torch.ones_like(x))[0]
            logits = logits[0].detach().float().cpu()

            logit_sum[start:end] += logits[:end - start]
            token_count[start:end] += 1

            if end == len(ids):
                break

        averaged = logit_sum / token_count.clamp_min(1).unsqueeze(-1)
        cached_logits[doc_id] = (text, enc.offsets, averaged)

        if n % 25 == 0:
            print(f"Cached {n}/{len(val_texts)} validation documents")

print(f"Cached logits in {time.time() - t0:.1f}s")

# Decode with each penalty.
results_by_penalty = {}

for penalty in penalties:
    raw = {}

    for doc_id, (text, offsets, averaged) in cached_logits.items():
        if averaged is None:
            raw[doc_id] = []
            continue

        adjusted = averaged.clone()
        adjusted[:, prec_ids] -= penalty

        pred_ids = bio_viterbi_decode(adjusted)
        spans = decode_spans(text, offsets, pred_ids)

        probs = torch.softmax(adjusted, dim=-1)
        pred_tensor = torch.tensor(pred_ids, dtype=torch.long)
        scored = []

        for s, e, label in spans:
            token_indices = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]
            conf = (
                float(probs[token_indices, pred_tensor[token_indices]].mean())
                if token_indices else 0.0
            )
            scored.append((s, e, label, conf))

        raw[doc_id] = scored

    filtered = filter_predictions_by_thresholds_for_texts(
        raw, thresholds, val_texts
    )
    metrics = calculate_exact_span_metrics(filtered, val_gold)

    results_by_penalty[penalty] = {
        "predictions": filtered,
        "metrics": metrics,
    }

    print(f"\nPREC penalty = {penalty}")
    print("Macro-F1:", metrics["macro_f1"])
    print("Predicted spans:", sum(len(v) for v in filtered.values()))

print("\nSummary")
for penalty in penalties:
    m = results_by_penalty[penalty]["metrics"]
    print(f"penalty={penalty:>3}: macro-F1={m['macro_f1']:.6f}")

In [ ]:
import inspect
print(inspect.signature(calculate_exact_span_metrics))

In [ ]:
# Inspect the metric structure from the first completed penalty.
print("Available metric keys:", list(metrics.keys()))
print("Metric output:", metrics)